# PathSight3D v2 rev7 — 3D Well Path Optimization

**Optimize well placement to maximize hydrocarbon coverage** from a single offshore platform, integrating seismic geobody data, fault zone geometry, and directional drilling constraints.

---

### What This Notebook Does

From raw seismic + fault data → optimized set of k wells with fault-priority control.

### Pipeline

| Cell | Stage | Description |
|------|-------|-------------|
| 1 | **Configuration** | All user-defined parameters (paths, well geometry, optimization settings) |
| 2 | Imports | Python packages |
| 3 | Data Containers | `RegularGrid3D` class — the core 3D grid structure |
| 4 | Utility Functions | Well trajectory (3D SLERP), coverage computation |
| 5 | Data Loading | SEG-Y or CSV → `RegularGrid3D` |
| 6 | Filtering | Apply spatial exclusions (boundary line, existing cones) |
| 6b | 3D Visualization | Interactive HTML of geobody grid |
| 6c | **Fault Processing** | Petrel picks → fault zones → cell classification + value assignment |
| 6d | Fault Visualization | 3D HTML with geobody + fault categories |
| 6e | Z-Slice Viewer | 2D plan-view slices at multiple depths |
| 7 | Platform Candidates | Generate 2D candidate grid |
| 8 | Reachability | Score each candidate by cone coverage → heatmap |
| 9 | Heatmap Plot | Visualize reachability scores |
| 10 | **Select Platform** | Pick platform → extract cone sub-grid |
| 11 | **Well Path Search** | Parallel brute-force: 9-param trajectories → HDF5 on disk |
| 11b | **Score Filtering** | Load from HDF5, filter by percentile, build sparse matrix |
| 12 | Optimization Functions | Greedy CELF + CP-SAT solvers (reusable) |
| 13 | **Two-Phase Optimization** | m fault wells + (k−m) geobody wells with swap refinement |
| 14 | **Creaming Curves** | Coverage vs. wells for each m value |
| 15 | 3D Scene Export | Interactive HTML per (k, m) with category-colored cells |
| 15b | Z-Slice Well Viewer | Well paths + coverage at 6 depth slices per (k, m) |
| 16 | Summary | Final statistics |

### Key Features (v2.7)

- **3D SLERP trajectory** — Segment D changes both inclination and azimuth via great-circle arc
- **Two-phase fault-priority picking** — m wells target fault cells, k−m target geobody
- **Parallel grid search** — chunk-based multiprocessing with HDF5 disk output
- **Memory-safe** — HDF5 intermediate storage, score-filtered sparse matrix loading
- **Corrected fault volume** — subtracts overlap geobody volume before distributing to fault-only cells
- **Deactivated fault filtering** — faults with None volume are excluded before cell classification


## Cell 1: Configuration

**⚠️ Edit this cell before running.** All user-defined parameters are here.

Key parameter groups:
- **`PathConfig`** — Input file paths, output directory, file format
- **`SEGYConfig`** — Trace header field names for SEG-Y reading
- **`GridSpacingConfig`** — Cell spacing (dx, dy, dz) and z-downsampling factor
- **`ConeConfig`** — Cone half-angle (θ), depth offset, max horizontal departure
- **`WellConfig`** — Well trajectory search ranges (v, α, p1, h, p2, β, t, DLS1, DLS2), coverage radius
- **`OptConfig`** — k_max, m_fault_values, CP-SAT settings
- **`VolumeConfig`** — Porosity, Sg, Bg, RF, fault volume assignments per (fault, side)
- **`FaultConfig`** — Fault file paths, buffer distance, contour cleanup parameters
- **`PlatformSelection`** — Fixed platform coordinates or use best from scoring
- **`ExclusionConfig`** — Boundary line, existing platform exclusion zones


In [ ]:
# ==============================================================================
# CONFIGURATION — Edit values below to customize the workflow
# ==============================================================================
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional, Tuple
import numpy as np

# ── FILE PATHS ────────────────────────────────────────────────────────────
@dataclass
class PathConfig:
    # Input data file (supports: .segy, .sgy, .csv, .txt, .dat)
    data_path: Path = Path(r"Offshore\Inputs\HC_voxels\HC_Probability_SI_50m")
    
    # File format: 'auto', 'segy', 'csv', 'txt'
    file_format: str = 'segy'
    
    # For CSV/TXT files: column configuration
    x_column: str = 'X'
    y_column: str = 'Y'
    z_column: str = 'Z'
    val_column: str = 'P'
    delimiter: str = ','
    skip_rows: int = 0
    has_header: bool = True
    
    # Z convention flag for geobody data
    # True  = input uses positive depth (e.g. 1200 m) -> negate to get z <= 0
    # False = input already uses negative depth (e.g. -1200 m) -> keep as-is
    z_positive_down: bool = True
    
    # Other paths
    shapefile_dir: Path = Path(r"Offshore\Inputs\Anomaly")
    output_dir: Path = Path(r"Offshore\Outputs07")
    
    # Grid cache
    grid_cache_path: Path = Path(output_dir / "grid_cache.npz")
    use_cached_grid: bool = False   # if True and cache exists, skip loading + faults
    
    def validate(self):
        if not self.use_cached_grid:
            if not self.data_path.exists():
                raise FileNotFoundError(f"Data file not found: {self.data_path}")
        if not self.shapefile_dir.exists():
            raise FileNotFoundError(f"Shapefile dir not found: {self.shapefile_dir}")
        self.output_dir.mkdir(parents=True, exist_ok=True)
        return self
    
    def get_format(self) -> str:
        if self.file_format != 'auto':
            return self.file_format.lower()
        ext = self.data_path.suffix.lower()
        if ext in ['.segy', '.sgy']:
            return 'segy'
        elif ext == '.csv':
            return 'csv'
        elif ext in ['.txt', '.dat', '.asc', '.xyz']:
            return 'txt'
        else:
            raise ValueError(f"Unknown file extension '{ext}'. Set file_format explicitly.")

# ── SEG-Y READER CONFIGURATION ────────────────────────────────────────────
@dataclass
class SEGYConfig:
    """User-configurable trace header field names for SEG-Y reading.
    Always opens with ignore_geometry=True and parses ALL trace headers.
    Adjust keys based on your SEG-Y file header contents."""
    il_key: str = 'INLINE_3D'             # Trace header key for inline number
    xl_key: str = 'CROSSLINE_3D'          # Trace header key for crossline number
    x_key: str = 'CDP_X'                  # Trace header key for X coordinate
    y_key: str = 'CDP_Y'                  # Trace header key for Y coordinate
    scalar_key: str = 'SourceGroupScalar' # Coordinate scalar header key

# ── GRID SPACING (for CSV/TXT files without regular grid) ─────────────────
@dataclass
class GridSpacingConfig:
    dx: Optional[float] = 25.3   # X spacing (m)
    dy: Optional[float] = 25.3   # Y spacing (m)
    dz: Optional[float] = 1.41   # Z spacing (m)
    z_factor: int = 3            # Z-axis downsampling: 1=native, 2=merge 2 layers, 3=merge 3

# ── CONE-CYLINDER GEOMETRY ────────────────────────────────────────────────
@dataclass
class ConeConfig:
    theta_deg: float = 60.0
    z_offset: float = 100.0
    hd: float = 1e16
    
    @property
    def theta_rad(self) -> float:
        return np.radians(self.theta_deg)
    
    @property
    def z0(self) -> float:
        return -self.z_offset
    
    @property
    def tan_theta(self) -> float:
        return np.tan(self.theta_rad)

# ── GRID SEARCH PARAMETERS (platform search grid, NOT the geobody grid) ───
@dataclass
class GridConfig:
    step: float = 500.0
    spacing: float = 500.0
    edge_buffer: float = 3000.0
    anomaly_buffer: float = 100.0

# ── WELL PATH PARAMETERS ──────────────────────────────────────────────────
@dataclass
class WellConfig:
    v_range: np.ndarray = field(default_factory=lambda: np.array([100.0]))
    alpha_range: np.ndarray = field(default_factory=lambda: np.arange(0.0, 359.0, 20.0))
    beta_range: np.ndarray = field(default_factory=lambda: np.arange(0.0, 359.0, 20.0))
    p1_range: np.ndarray = field(default_factory=lambda: np.arange(0.0, 60.0, 10.0))
    h_range: np.ndarray = field(default_factory=lambda: np.array([1100.0]))
    p2_range: np.ndarray = field(default_factory=lambda: np.arange(0.0, 60.0, 10.0))
    t_range: np.ndarray = field(default_factory=lambda: np.array([50.0]))
    DLS1_range: np.ndarray = field(default_factory=lambda: np.array([0.05, 0.15, 0.25]))
    DLS2_range: np.ndarray = field(default_factory=lambda: np.array([0.03, 0.09, 0.13]))
    
    # Azimuth change in drop segment D (3D SLERP)
    beta_mode: str = 'absolute'   # 'absolute' = compass bearing, 'relative' = delta from alpha
    # 'relative' example: np.array([-20, -10, 0, 10, 20])  → beta = alpha ± delta
    # 'absolute' example: np.arange(0, 360, 30)             → full azimuth sweep
    
    md_max: float = 1e16
    hd_max: float = 1e16
    coverage_radius: float = 500.0

# ── OPTIMIZATION PARAMETERS ───────────────────────────────────────────────
@dataclass
class OptConfig:
    k_max: int = 30
    time_limit: Optional[float] = 10800.0
    num_workers: int = 14
    use_greedy_hint: bool = True
    rel_gap: Optional[float] = None
    
    # Two-phase optimization: fault-first picking
    # For each m, pick m fault wells then (k-m) geobody wells
    # m=0 gives baseline (no fault priority)
    m_fault_values: List[int] = field(default_factory=lambda: [0,1,2,3,4,5,6,7,8,9,10])

# ── EXCLUSION COORDINATES (Optional) ──────────────────────────────────────
@dataclass
class ExclusionConfig:
    Thres: float = 0.5
    platform_x: np.ndarray = field(default_factory=lambda: np.array([]))
    platform_y: np.ndarray = field(default_factory=lambda: np.array([]))
    boundary_line: Optional[Tuple[Tuple[float, float], Tuple[float, float]]] = ((912182.64, 826616.60), (926763.03, 843532.50))

# ── VOLUME SCALING - Dual-mode: raw probabilities or hydrocarbon volume ───
@dataclass
class VolumeConfig:
    mode: str = 'volume'             # 'raw' = keep original values
                                   # 'volume' = convert to hydrocarbon volume
    
    # Geobody volume parameters (only used when mode='volume')
    # NOTE: bulk_volume (dx*dy*dz) is auto-extracted from grid - NOT user-specified
    poro: float = 0.2             # Porosity
    Sg: float = 0.7               # Gas saturation
    Bg: float = 0.01              # Gas formation volume factor
    RF: float = 0.6               # Recovery factor
    m3_to_ft3: float = 35.3147    # Unit conversion
    unit_factor: float = 1e-9     # Scale to BCF
    
    # Fault volume assignment: total volume per (fault_filename, side)
    # Keys are tuples: (fault_filename_stem, "dip"|"anti")
    # Values are total hydrocarbon volume for that fault zone
    fault_volumes: dict = field(default_factory=lambda: {
        ("Fault_2025_G2_TKN_01_W", "dip"):  None,
        ("Fault_2025_G2_TKN_01_W", "anti"): 52.68,
        ("Fault_2025_G2_TKN_02_W", "dip"):  None,
        ("Fault_2025_G2_TKN_02_W", "anti"):  17.71,
    })

# ── SELECTED PLATFORM FOR WELL SIMULATION ─────────────────────────────────
@dataclass
class PlatformSelection:
    x: float = 916430.10
    y: float = 838380.50
    use_best: bool = False

# ── FAULT ZONE CONFIGURATION ──────────────────────────────────────────────
@dataclass
class FaultConfig:
    enabled: bool = True               # Set True to enable fault integration
    
    # Fault files: list of paths (one file per fault, Petrel format)
    fault_dir: Path = Path(r"Offshore\Inputs\Faults")
    fault_pattern: str = "Fault_*"
    fault_files: List[Path] = field(default_factory=list)
    
    # Z convention flag for fault data
    # False = fault files already use negative depth (Petrel default)
    # True  = fault files use positive depth -> negate
    z_positive_down: bool = False
    
    # Fault zone parameters
    fault_value: float = 0.0             # Value assigned to fault zone cells (raw mode)
    offset_distance: float = 600.0       # Buffer distance (m) each side of fault
    trim_distance: float = 500.0         # Arc-length trim from each tip (m)
    grid_res: int = 400                  # Grid resolution for interpolation surface
    
    # Contour post-processing
    resample_spacing: float = 25.0
    smooth_window: int = 3
    max_turn_angle: float = 60.0
    
    # Merge behavior
    merge_into_grid: bool = True
    
    def resolve_files(self) -> List[Path]:
        if self.fault_files:
            return [Path(f) for f in self.fault_files]
        pattern = self.fault_dir / self.fault_pattern
        found = sorted(glob.glob(str(pattern)))
        return [Path(f) for f in found]

# ── CREATE INSTANCES ──────────────────────────────────────────────────────
PATHS = PathConfig()
SEGY_CFG = SEGYConfig()
GRID_SPACING = GridSpacingConfig()
CONE = ConeConfig()
GRID = GridConfig()
WELL = WellConfig()
OPT = OptConfig()
EXCLUSION = ExclusionConfig()
VOLUME = VolumeConfig()
PLATFORM = PlatformSelection()
FAULT = FaultConfig()

try:
    PATHS.validate()
    print("Configuration loaded successfully.")
    print(f"  Data file: {PATHS.data_path}")
    print(f"  Format: {PATHS.get_format()}")
    print(f"  Output: {PATHS.output_dir}")
    print(f"  Cone: theta={CONE.theta_deg} deg, hd={CONE.hd}m")
    print(f"  Z convention (geobody): z_positive_down={PATHS.z_positive_down}")
    print(f"  Z convention (faults):  z_positive_down={FAULT.z_positive_down}")
    print(f"  Z downsampling factor:  {GRID_SPACING.z_factor}")
    print(f"  Volume mode:            {VOLUME.mode}")
    print(f"  Grid cache:             {'ON' if PATHS.use_cached_grid else 'OFF'}")
except FileNotFoundError as e:
    print(f"Configuration error: {e}")
    print("  Please update the paths in PathConfig above.")


## Cell 2: Imports

All Python package imports. Requires: `numpy`, `scipy`, `pandas`, `geopandas`, `matplotlib`, `pyvista`, `segyio`, `shapely`, `h5py`, `ortools`, `tqdm`.


In [ ]:
# ==============================================================================
# IMPORTS — All required packages
# ==============================================================================
import os
import glob
import re
import csv
import math
import gc
import pickle
import h5py
from time import perf_counter
from heapq import heapify, heappush, heappop
from collections import defaultdict
from typing import Tuple, List, Dict, Optional, Any
from math import ceil

import numpy as np
import pandas as pd
import geopandas as gpd
import segyio

from scipy import sparse
from scipy.sparse import csr_matrix, csc_matrix, lil_matrix, coo_matrix
from scipy.spatial import cKDTree
from scipy.optimize import curve_fit

from shapely.ops import unary_union
from shapely.geometry import MultiPoint

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patheffects as pe

import pyvista as pv
from tqdm.auto import tqdm

from ortools.sat.python import cp_model

gpd.options.io_engine = "fiona"


# Fault Zone Pipeline imports
from scipy.interpolate import LinearNDInterpolator
from scipy.ndimage import uniform_filter1d
from shapely.geometry import LineString, Point
from matplotlib.path import Path as MplPath

print("All imports successful.")


## Cell 3: Data Containers

Defines `RegularGrid3D` — a structured 3D grid where every cell has a deterministic address `(i, j, k)` computed by O(1) arithmetic:

```
i = round((x − x_origin) / dx)
```

**Core arrays:** `values[nx, ny, nz]` (float32) and `active[nx, ny, nz]` (bool).

Additional arrays added during fault processing: `cell_type`, `fault_id`, `fault_side`, `viz_category`.


In [ ]:
# ==============================================================================
# DATA CONTAINERS — RegularGrid3D (core data structure)
# ==============================================================================

@dataclass
class RegularGrid3D:
    """
    Regularized 3D grid for geobody data.
    
    All geobody cells live on a structured grid with deterministic
    coordinate <-> index mapping via O(1) arithmetic.
    
    Attributes
    ----------
    x_origin, y_origin, z_origin : float
        World coordinates of the grid cell at index (0, 0, 0).
    dx, dy, dz : float
        Grid spacing in each dimension (meters).
    nx, ny, nz : int
        Number of cells in each dimension.
    values : np.ndarray, shape (nx, ny, nz)
        Property values (e.g., probability, porosity).
    active : np.ndarray, shape (nx, ny, nz), dtype=bool
        Which cells are part of the geobody.
    """
    x_origin: float
    y_origin: float
    z_origin: float
    dx: float
    dy: float
    dz: float
    nx: int
    ny: int
    nz: int
    values: np.ndarray    # (nx, ny, nz), dtype=float32
    active: np.ndarray    # (nx, ny, nz), dtype=bool
    
    # --- Coordinate <-> index conversion (O(1)) ---
    
    def xyz_to_ijk(self, x, y, z):
        """World coordinates -> grid indices (nearest cell). Vectorized."""
        i = np.round((np.asarray(x) - self.x_origin) / self.dx).astype(int)
        j = np.round((np.asarray(y) - self.y_origin) / self.dy).astype(int)
        k = np.round((np.asarray(z) - self.z_origin) / self.dz).astype(int)
        return i, j, k
    
    def ijk_to_xyz(self, i, j, k):
        """Grid indices -> world coordinates (cell centers). Vectorized."""
        x = self.x_origin + np.asarray(i, dtype=float) * self.dx
        y = self.y_origin + np.asarray(j, dtype=float) * self.dy
        z = self.z_origin + np.asarray(k, dtype=float) * self.dz
        return x, y, z
    
    def ijk_in_bounds(self, i, j, k):
        """Check if indices are within grid bounds. Vectorized."""
        return (
            (i >= 0) & (i < self.nx) &
            (j >= 0) & (j < self.ny) &
            (k >= 0) & (k < self.nz)
        )
    
    def flat_index(self, i, j, k):
        """Convert (i,j,k) to raveled flat index. Vectorized."""
        return np.ravel_multi_index((np.asarray(i), np.asarray(j), np.asarray(k)),
                                    (self.nx, self.ny, self.nz))
    
    def unravel(self, flat_idx):
        """Convert flat index back to (i, j, k)."""
        return np.unravel_index(flat_idx, (self.nx, self.ny, self.nz))
    
    # --- Grid axes ---
    
    @property
    def x_axis(self) -> np.ndarray:
        return self.x_origin + np.arange(self.nx) * self.dx
    
    @property
    def y_axis(self) -> np.ndarray:
        return self.y_origin + np.arange(self.ny) * self.dy
    
    @property
    def z_axis(self) -> np.ndarray:
        return self.z_origin + np.arange(self.nz) * self.dz
    
    # --- Active cell queries ---
    
    @property
    def n_active(self) -> int:
        return int(self.active.sum())
    
    @property
    def active_flat_indices(self) -> np.ndarray:
        """Raveled indices of all active cells. Used as sparse matrix column IDs."""
        return np.flatnonzero(self.active.ravel())
    
    @property
    def n_total(self) -> int:
        return self.nx * self.ny * self.nz
    
    @property
    def total_volume(self) -> float:
        return self.n_active * self.dx * self.dy * self.dz
    
    # --- Extract scattered (x, y, z, val) for active cells ---
    
    def active_xyz(self) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Return (x, y, z) arrays for all active cells."""
        I, J, K = np.where(self.active)
        return self.ijk_to_xyz(I, J, K)
    
    def active_values(self) -> np.ndarray:
        """Return property values for all active cells."""
        return self.values[self.active]
    
    # --- Build lookup: flat_index -> position in active list ---
    
    def build_active_lookup(self) -> np.ndarray:
        """
        Build array mapping flat grid index -> position in active list.
        
        Returns array of length n_total where:
          lookup[flat_idx] = position in active_flat_indices (0..n_active-1)
          lookup[flat_idx] = -1 if cell is not active
        
        This is essential for building sparse coverage matrices where
        columns correspond to active cells only (not all grid cells).
        """
        lookup = np.full(self.n_total, -1, dtype=np.int32)
        active_flat = self.active_flat_indices
        lookup[active_flat] = np.arange(len(active_flat), dtype=np.int32)
        return lookup
    
    # --- Filtering (modifies active mask in-place or returns copy) ---
    
    def apply_mask(self, mask_3d: np.ndarray):
        """Set active=False where mask_3d is True (exclusion). In-place."""
        self.active[mask_3d] = False
    
    def copy(self) -> 'RegularGrid3D':
        """Return a deep copy."""
        return RegularGrid3D(
            x_origin=self.x_origin, y_origin=self.y_origin, z_origin=self.z_origin,
            dx=self.dx, dy=self.dy, dz=self.dz,
            nx=self.nx, ny=self.ny, nz=self.nz,
            values=self.values.copy(), active=self.active.copy()
        )
    
    # --- Summary ---
    
    def summary(self) -> str:
        x_min, x_max = self.x_origin, self.x_origin + (self.nx - 1) * self.dx
        y_min, y_max = self.y_origin, self.y_origin + (self.ny - 1) * self.dy
        z_min, z_max = self.z_origin, self.z_origin + (self.nz - 1) * self.dz
        vals = self.values[self.active]
        if vals.size > 0:
            val_str = f"[{vals.min():.4g}, {vals.max():.4g}]"
        else:
            val_str = "(no active cells)"
        return (
            f"RegularGrid3D: {self.nx} × {self.ny} × {self.nz} = {self.n_total:,} cells\n"
            f"  Active: {self.n_active:,} ({100*self.n_active/self.n_total:.1f}%)\n"
            f"  X: [{x_min:.1f}, {x_max:.1f}], dx={self.dx:.2f}\n"
            f"  Y: [{y_min:.1f}, {y_max:.1f}], dy={self.dy:.2f}\n"
            f"  Z: [{z_min:.1f}, {z_max:.1f}], dz={self.dz:.2f}\n"
            f"  Val: {val_str} (active only)\n"
            f"  Volume: {self.total_volume:,.0f} m³"
        )
    
    def __len__(self):
        return self.n_active


# ==============================================================================
# SUPPORTING CONTAINERS (unchanged from v1)
# ==============================================================================

@dataclass
class PlatformCandidates:
    x: np.ndarray
    y: np.ndarray
    x_grid: np.ndarray
    y_grid: np.ndarray
    flat_idx: np.ndarray
    
    def __len__(self):
        return len(self.x)

@dataclass
class PlatformResult:
    candidates: PlatformCandidates
    heatmap: np.ndarray
    best_idx: int
    best_x: float
    best_y: float
    best_score: float

@dataclass
class WellPathResult:
    """Results from well path simulation.
    
    coverage: sparse matrix (N_valid, N_active_cells) where columns
              correspond to positions in grid.active_flat_indices.
    """
    params: np.ndarray       # (N_valid, 8) well parameters
    coverage: csr_matrix     # (N_valid, N_active) sparse
    scores: np.ndarray       # (N_valid,) coverage scores
    platform_x: float
    platform_y: float

print("✓ Data containers defined (RegularGrid3D).")


## Cell 4: Utility Functions

Two critical functions defined here:

**`three_segment_path_xy`** — Computes a 5-segment well trajectory (Vertical → Build → Hold → Drop → Tangent) with 9 parameters. Segment D uses **SLERP** (Spherical Linear Interpolation) to smoothly change both inclination (p1→p2) and azimuth (α→β) along a great-circle arc at constant dogleg severity.

**`coverage_on_grid`** — Two-gate coverage algorithm. For each depth along the open-hole segment: Gate 1 (Hit) checks if the well passes through an active cell; Gate 2 (Expand) marks all active cells within the coverage radius at that depth. Returns flat grid indices (~16 KB per call).


In [ ]:
# ==============================================================================
# UTILITY FUNCTIONS — Well trajectory + coverage computation
# ==============================================================================

# ── SEG-Y Utilities ──────────────────────────────────────────────────────────

def apply_scalco(arr: np.ndarray, scal: np.ndarray) -> np.ndarray:
    scal = np.asarray(scal, dtype=float)
    scale = np.where(scal > 0, scal, np.where(scal < 0, 1.0 / np.abs(scal), 1.0))
    return arr * scale

def read_text_header(f) -> List[str]:
    raw = f.text[0]
    if isinstance(raw, (bytes, bytearray)):
        try:
            txt = raw.decode("ascii", errors="replace")
        except Exception:
            txt = raw.decode("cp500", errors="replace")
    else:
        txt = str(raw)
    return [txt[i:i+80] for i in range(0, min(len(txt), 3200), 80)]

def parse_depth_from_text_header(lines: List[str]) -> Tuple[Optional[float], Optional[float], Optional[int]]:
    for ln in lines:
        if "Depth min" in ln:
            m = re.search(
                r"Depth\s+min:\s*([+-]?\d+\.?\d*)\s+max:\s*([+-]?\d+\.?\d*)\s+delta:\s*([+-]?\d+\.?\d*)",
                ln
            )
            if m:
                return float(m.group(1)), float(m.group(2)), int(float(m.group(3)))
    return None, None, None

def median_step(arr: np.ndarray) -> float:
    d = np.diff(np.sort(np.unique(arr)))
    d = d[np.isfinite(d) & (d != 0)]
    return float(np.median(np.abs(d))) if d.size else np.nan

# -----------------------------------------------------------------------------
# Geometry Utilities
# -----------------------------------------------------------------------------

def cone_radius(z: np.ndarray, z0: float, tan_theta: float, hd: float) -> np.ndarray:
    return np.minimum((z0 - z) * tan_theta, hd)

def mask_below_line(
    x: np.ndarray, y: np.ndarray,
    x1: float, y1: float, x2: float, y2: float,
    inclusive: bool = True
) -> np.ndarray:
    vx, vy = x2 - x1, y2 - y1
    denom = vx * vx + vy * vy
    if denom < 1e-12:
        return np.zeros(len(x), dtype=bool)
    t = ((x - x1) * vx + (y - y1) * vy) / denom
    y_on = y1 + t * vy
    return np.less_equal(y, y_on) if inclusive else np.less(y, y_on)


def cone_mask_on_grid(
    grid: RegularGrid3D,
    x0: float, y0: float, z0: float,
    tan_theta: float, hd: float
) -> np.ndarray:
    """
    Compute boolean 3D mask of grid cells inside cone-cylinder from (x0, y0, z0).
    
    Instead of per-point distance checks, this works per z-layer:
    for each k, compute the cone radius R(z_k), then mask all (i,j) cells
    whose distance from (x0,y0) exceeds R(z_k).
    
    Returns: bool array of shape (nx, ny, nz)
    """
    mask = np.zeros((grid.nx, grid.ny, grid.nz), dtype=bool)
    
    # Precompute horizontal distances from (x0, y0) for all (i, j)
    x_ax = grid.x_axis  # (nx,)
    y_ax = grid.y_axis  # (ny,)
    DX = x_ax - x0       # (nx,)
    DY = y_ax - y0       # (ny,)
    D2 = DX[:, None]**2 + DY[None, :]**2  # (nx, ny) - squared horizontal distance
    
    for k in range(grid.nz):
        z_k = grid.z_origin + k * grid.dz
        if z_k >= z0:
            continue  # Must be below apex
        Rz = min((z0 - z_k) * tan_theta, hd)
        mask[:, :, k] = D2 <= Rz * Rz
    
    return mask


# -----------------------------------------------------------------------------
# Well Path Function (unchanged geometry)
# -----------------------------------------------------------------------------

def three_segment_path_xy(
    x0: float, y0: float, z: np.ndarray,
    alpha_deg: float, v: float, p1_deg: float, h: float,
    p2_deg: float, beta_deg: float, t: float,
    DLS1_deg_per_m: float = 0.5, DLS2_deg_per_m: float = 0.5,
    md_step: float = 1.0,
) -> Tuple[np.ndarray, np.ndarray, float, float]:
    """3-segment wellpath with 3D SLERP dogleg for segment D.
    
    Segments A-C: 2D (constant azimuth = alpha).
    Segment D: 3D great-circle arc from (p1, alpha) to (p2, beta) at constant DLS.
    Segment E: straight at (p2, beta).
    """
    z = np.asarray(z, dtype=float)
    z_clamped = np.minimum(z, 0.0)
    tvd_targets = -z_clamped
    tvd_max = float(np.max(tvd_targets))

    alpha = np.deg2rad(alpha_deg)
    beta = np.deg2rad(beta_deg)
    p1 = np.deg2rad(p1_deg)
    p2 = np.deg2rad(p2_deg)

    # --- Segment A: Vertical (θ=0) ---
    MD_A = float(v)

    # --- Segment B: Build (0 → p1) at DLS1 ---
    MD_B = p1_deg / DLS1_deg_per_m if p1_deg > 0.0 else 0.0
    d1 = MD_B * 0.5 * (np.cos(0.0) + np.cos(p1)) if MD_B > 0.0 else 0.0

    # --- Segment D: 3D SLERP from (p1, alpha) to (p2, beta) ---
    # Direction unit vectors
    d_start = np.array([np.sin(p1)*np.cos(alpha), np.sin(p1)*np.sin(alpha), np.cos(p1)])
    d_end   = np.array([np.sin(p2)*np.cos(beta),  np.sin(p2)*np.sin(beta),  np.cos(p2)])
    
    cos_delta = np.clip(np.dot(d_start, d_end), -1.0, 1.0)
    delta_rad = np.arccos(cos_delta)  # great-circle angle
    delta_deg = np.degrees(delta_rad)
    
    MD_D = delta_deg / DLS2_deg_per_m if delta_deg > 1e-6 else 0.0
    
    # Fast T1 pre-screen: use trapezoidal approximation for d2
    # This avoids the expensive SLERP integration for clearly infeasible paths
    d2_approx = MD_D * 0.5 * (d_start[2] + d_end[2]) if MD_D > 0 else 0.0  # cos(θ) endpoints
    dz_cased_approx = t * np.cos(p2)
    T1_approx = h - d1 - d2_approx - dz_cased_approx
    if T1_approx <= -MD_D * 0.1:  # generous margin (10% of MD_D) for approx error
        raise ValueError(f"T1 pre-screen failed (T1_approx={T1_approx:.1f}, MD_D={MD_D:.1f})")
    
    # Pre-compute TVD consumed by segment D (numerical integration)
    n_d_steps = max(2, int(np.ceil(MD_D / md_step)) + 1) if MD_D > 0 else 0
    d2 = 0.0
    if n_d_steps > 0 and delta_rad > 1e-8:
        sin_delta = np.sin(delta_rad)
        for i_d in range(n_d_steps):
            f = (i_d + 0.5) / n_d_steps
            # SLERP
            w0 = np.sin((1 - f) * delta_rad) / sin_delta
            w1 = np.sin(f * delta_rad) / sin_delta
            d_f = w0 * d_start + w1 * d_end
            d2 += d_f[2] * (MD_D / n_d_steps)  # cos(θ) component
    elif n_d_steps > 0:
        # delta ≈ 0: direction unchanged
        d2 = MD_D * d_start[2]
    
    dz_cased_tangent = t * np.cos(p2)
    T1 = h - d1 - d2 - dz_cased_tangent

    if T1 <= 0:
        raise ValueError(f"Computed T1 <= 0 (T1={T1:.3f} m).")

    # --- Segment C: Hold tangent at (p1, alpha) ---
    MD_C = T1 / max(np.cos(p1), 1e-6)
    
    # --- Segment E: final tangent at (p2, beta) ---
    MD_E_cased = float(t)
    tvd_casing_end = v + h
    tvd_remaining = max(0.0, tvd_max - tvd_casing_end)
    MD_E_open = tvd_remaining / max(np.cos(p2), 1e-6)
    MD_total = MD_A + MD_B + MD_C + MD_D + MD_E_cased + MD_E_open

    # === Trace full path in 3D ===
    dMD = float(md_step)
    n_steps = max(2, int(np.ceil(MD_total / dMD)) + 1)
    MD = np.linspace(0.0, MD_total, n_steps)
    
    # Accumulate (x, y, tvd) directly
    x_arr = np.full(n_steps, x0)
    y_arr = np.full(n_steps, y0)
    tvd = np.zeros(n_steps)

    mA_end = MD_A
    mB_end = MD_A + MD_B
    mC_end = mB_end + MD_C
    mD_end = mC_end + MD_D

    # Precompute SLERP helpers
    sin_delta = np.sin(delta_rad) if delta_rad > 1e-8 else 0.0
    ca, sa = np.cos(alpha), np.sin(alpha)
    cb, sb = np.cos(beta), np.sin(beta)

    for i in range(1, n_steps):
        s_mid = 0.5 * (MD[i-1] + MD[i])
        dmd = MD[i] - MD[i-1]
        
        if s_mid <= mA_end:
            # Segment A: vertical
            tvd[i] = tvd[i-1] + dmd
            x_arr[i] = x_arr[i-1]
            y_arr[i] = y_arr[i-1]
        
        elif s_mid <= mB_end and MD_B > 0.0:
            # Segment B: build (0→p1), azimuth = alpha
            th = ((s_mid - mA_end) / MD_B) * p1
            tvd[i] = tvd[i-1] + np.cos(th) * dmd
            x_arr[i] = x_arr[i-1] + np.sin(th) * ca * dmd
            y_arr[i] = y_arr[i-1] + np.sin(th) * sa * dmd
        
        elif s_mid <= mC_end:
            # Segment C: hold at (p1, alpha)
            tvd[i] = tvd[i-1] + np.cos(p1) * dmd
            x_arr[i] = x_arr[i-1] + np.sin(p1) * ca * dmd
            y_arr[i] = y_arr[i-1] + np.sin(p1) * sa * dmd
        
        elif s_mid <= mD_end and MD_D > 0.0:
            # Segment D: 3D SLERP
            f = (s_mid - mC_end) / MD_D
            f = np.clip(f, 0.0, 1.0)
            
            if sin_delta > 1e-8:
                w0 = np.sin((1 - f) * delta_rad) / sin_delta
                w1 = np.sin(f * delta_rad) / sin_delta
                d_f = w0 * d_start + w1 * d_end
            else:
                d_f = d_start  # no turn
            
            tvd[i] = tvd[i-1] + d_f[2] * dmd
            x_arr[i] = x_arr[i-1] + d_f[0] * dmd
            y_arr[i] = y_arr[i-1] + d_f[1] * dmd
        
        else:
            # Segment E: straight at (p2, beta)
            tvd[i] = tvd[i-1] + np.cos(p2) * dmd
            x_arr[i] = x_arr[i-1] + np.sin(p2) * cb * dmd
            y_arr[i] = y_arr[i-1] + np.sin(p2) * sb * dmd

    # Interpolate: for each target TVD, find (x, y)
    tvd_monotonic = np.maximum.accumulate(tvd)
    x_out = np.interp(tvd_targets, tvd_monotonic, x_arr)
    y_out = np.interp(tvd_targets, tvd_monotonic, y_arr)
    
    HD = float(np.sqrt((x_arr[-1] - x0)**2 + (y_arr[-1] - y0)**2))
    return x_out, y_out, MD_total, HD


# ── Grid-Based Coverage ──────────────────────────────────────────────────────

def coverage_on_grid(
    x_line: np.ndarray,
    y_line: np.ndarray,
    z_line: np.ndarray,
    grid: RegularGrid3D,
    radius: float,
) -> np.ndarray:
    """
    Find active grid cells covered by a well path.
    
    Same two-gate logic (Hit + Expand) but returns FLAT INDICES instead
    of allocating a full (nx, ny, nz) bool array.
    
    Gate 1 ("Hit"): Nearest cell must be active at this depth.
    Gate 2 ("Expand"): All active cells within radius at same k-layer.
    
    Returns: sorted 1D array of flat grid indices of covered cells.
             Memory: O(n_covered) instead of O(nx * ny * nz).
             For 401x388x1076 grid: ~16 KB vs 167 MB per call.
    """
    x_line = np.asarray(x_line, dtype=float).ravel()
    y_line = np.asarray(y_line, dtype=float).ravel()
    z_line = np.asarray(z_line, dtype=float).ravel()
    
    # Neighborhood radius in grid units
    ri = int(np.ceil(radius / grid.dx))
    rj = int(np.ceil(radius / grid.dy))
    r2 = radius * radius
    
    di_range = np.arange(-ri, ri + 1)
    dj_range = np.arange(-rj, rj + 1)
    
    nx, ny, nz = grid.nx, grid.ny, grid.nz
    
    # Collect flat indices into a set (auto-deduplicates)
    covered_set = set()
    
    for idx in range(len(x_line)):
        xw, yw, zw = x_line[idx], y_line[idx], z_line[idx]
        
        # O(1) index conversion
        ic = int(round((xw - grid.x_origin) / grid.dx))
        jc = int(round((yw - grid.y_origin) / grid.dy))
        kc = int(round((zw - grid.z_origin) / grid.dz))
        
        # Bounds check
        if kc < 0 or kc >= nz:
            continue
        if ic < 0 or ic >= nx or jc < 0 or jc >= ny:
            continue
        
        # ---- Gate 1: "Hit" test ----
        if not grid.active[ic, jc, kc]:
            continue
        
        # ---- Gate 2: "Expand" to radius ----
        for di in di_range:
            ii = ic + di
            if ii < 0 or ii >= nx:
                continue
            dx_world = grid.x_origin + ii * grid.dx - xw
            dx2 = dx_world * dx_world
            if dx2 > r2:
                continue
            
            for dj in dj_range:
                jj = jc + dj
                if jj < 0 or jj >= ny:
                    continue
                
                if not grid.active[ii, jj, kc]:
                    continue
                
                dy_world = grid.y_origin + jj * grid.dy - yw
                if dx2 + dy_world * dy_world <= r2:
                    covered_set.add(ii * ny * nz + jj * nz + kc)
    
    if covered_set:
        return np.array(sorted(covered_set), dtype=np.int64)
    else:
        return np.empty(0, dtype=np.int64)


print("✓ Utility functions defined.")


## Cell 5: Data Loading

Reads geobody data into `RegularGrid3D`:

- **SEG-Y**: Reads trace headers via segyio, detects grid geometry from inline/crossline/depth coordinates, maps traces to (i,j,k) indices.
- **CSV/TXT**: Loads scattered (X,Y,Z,Value) points, estimates spacing, snaps to regular grid.

**User parameters:** `PathConfig.data_path`, `PathConfig.file_format`, `PathConfig.z_positive_down`, `SEGYConfig` header keys, `GridSpacingConfig` for spacing overrides.


In [ ]:
# ==============================================================================
# DATA LOADING — SEG-Y or CSV → RegularGrid3D
# ==============================================================================
t_start = perf_counter()


def parse_trace_headers(segyfile, n_traces):
    """Parse ALL trace headers into a pandas DataFrame (one row per trace).
    Based on segyreadtu.ipynb approach - ignores geometry entirely."""
    headers = segyio.tracefield.keys
    df = pd.DataFrame(index=range(n_traces), columns=headers.keys())
    for k, v in headers.items():
        df[k] = segyfile.attributes(v)[:]
    return df


def load_segy_to_grid(path, segy_cfg=None, z_positive_down=True):
    """
    Load SEG-Y file into RegularGrid3D using trace-header-only approach.
    
    Always opens with ignore_geometry=True - never relies on segyio auto-geometry.
    User-configurable header keys via SEGYConfig.
    
    Returns grid with z <= 0 convention (z_origin = deepest, dz > 0 toward surface).
    """
    if segy_cfg is None:
        segy_cfg = SEGYConfig()
    
    print(f"  Opening SEG-Y with ignore_geometry=True...")
    with segyio.open(str(path), "r", ignore_geometry=True) as f:
        f.mmap()
        n_traces = f.tracecount
        n_samples = f.samples.size
        
        # Step 1: Parse ALL trace headers into DataFrame
        print(f"  Parsing {n_traces:,} trace headers...")
        trace_headers = parse_trace_headers(f, n_traces)
        
        # Step 2: Read all trace data
        print(f"  Reading trace data ({n_traces:,} x {n_samples})...")
        data = f.trace.raw[:].astype(np.float32)  # (n_traces, n_samples)
        
        # Step 3: Extract inline/crossline using configured keys
        il_vals = trace_headers[segy_cfg.il_key].values.astype(int)
        xl_vals = trace_headers[segy_cfg.xl_key].values.astype(int)
        
        inlines = np.sort(np.unique(il_vals))
        xlines = np.sort(np.unique(xl_vals))
        n_il = len(inlines)
        n_xl = len(xlines)
        
        print(f"  Inlines:    {n_il} ({segy_cfg.il_key}: {inlines[0]}..{inlines[-1]})")
        print(f"  Crosslines: {n_xl} ({segy_cfg.xl_key}: {xlines[0]}..{xlines[-1]})")
        
        # Step 4: Extract X/Y coordinates with scalar correction
        scalar = trace_headers[segy_cfg.scalar_key].iloc[0]
        # segyio convention: negative = divisor, positive = multiplier
        scale = abs(scalar) if scalar < 0 else (1.0 / scalar if scalar > 0 else 1.0)
        
        x_coords = trace_headers[segy_cfg.x_key].values.astype(float) / scale
        y_coords = trace_headers[segy_cfg.y_key].values.astype(float) / scale
        
        # Validate coordinates
        valid_x = np.isfinite(x_coords) & (x_coords != 0)
        valid_y = np.isfinite(y_coords) & (y_coords != 0)
        if not (valid_x.any() and valid_y.any()):
            raise ValueError(
                f"Coordinates from {segy_cfg.x_key}/{segy_cfg.y_key} are all zero or invalid. "
                f"Check SEGYConfig keys."
            )
        
        print(f"  Coord scalar: {scalar} (divide by {scale})")
        print(f"  X range: [{x_coords[valid_x].min():.1f}, {x_coords[valid_x].max():.1f}]")
        print(f"  Y range: [{y_coords[valid_y].min():.1f}, {y_coords[valid_y].max():.1f}]")
        
        # Step 5: Extract Z axis
        z_axis = f.samples.astype(float)
    
    # Step 6: Apply z convention
    if z_positive_down:
        z_axis = -np.abs(z_axis)
        print(f"  Z negated (z_positive_down=True): [{z_axis.min():.1f}, {z_axis.max():.1f}]")
    else:
        if z_axis.max() > 0:
            print(f"  WARNING: z_positive_down=False but z has positive values [{z_axis.min():.1f}, {z_axis.max():.1f}]")
        print(f"  Z kept as-is: [{z_axis.min():.1f}, {z_axis.max():.1f}]")
    
    # Ensure z_axis is sorted ascending (deepest first, surface last)
    if z_axis[0] > z_axis[-1]:
        z_axis = z_axis[::-1]
        data = data[:, ::-1]  # flip sample axis to match
    
    # Step 7: Compute dx, dy from coordinates
    order = np.lexsort((xl_vals, il_vals))
    x_sorted = x_coords[order].reshape(n_il, n_xl)
    y_sorted = y_coords[order].reshape(n_il, n_xl)
    
    x_axis = np.median(x_sorted, axis=0)  # n_xl values (crossline direction = X)
    y_axis = np.median(y_sorted, axis=1)  # n_il values (inline direction = Y)
    
    dx = median_step(x_axis)
    dy = median_step(y_axis)
    dz = abs(median_step(z_axis))
    print(f"  Spacing: dx={dx:.2f}, dy={dy:.2f}, dz={dz:.2f}")
    
    # Step 8: Reshape to 3D cube
    if n_il * n_xl == n_traces:
        cube = data[order].reshape(n_il, n_xl, n_samples).astype(np.float32)
    else:
        print(f"  Missing traces: {n_il * n_xl - n_traces:,} - using sparse populate")
        cube = np.zeros((n_il, n_xl, n_samples), dtype=np.float32)
        il_map = {v: i for i, v in enumerate(inlines)}
        xl_map = {v: j for j, v in enumerate(xlines)}
        for t in range(n_traces):
            ii = il_map.get(il_vals[t])
            jj = xl_map.get(xl_vals[t])
            if ii is not None and jj is not None:
                cube[ii, jj, :] = data[t]
    
    del data, trace_headers  # free raw data
    
    # Step 9: Transpose to (nx=crossline, ny=inline, nz=depth)
    cube = cube.transpose(1, 0, 2).copy()
    
    # Build active mask
    Thres = EXCLUSION.Thres
    vals = cube.astype(np.float32)
    active = vals != 0.0
    if Thres is not None:
        active &= vals >= Thres
    
    n_active = active.sum()
    print(f"  Active cells: {n_active:,} / {cube.size:,} ({100*n_active/cube.size:.1f}%)")
    
    return RegularGrid3D(
        x_origin=float(x_axis[0]), y_origin=float(y_axis[0]), z_origin=float(z_axis[0]),
        dx=dx, dy=dy, dz=dz,
        nx=n_xl, ny=n_il, nz=n_samples,
        values=vals,
        active=active
    )


def load_csv_to_grid(
    path, x_col, y_col, z_col, val_col,
    delimiter=',', skip_rows=0, has_header=True,
    grid_spacing=None, z_positive_down=True,
):
    """
    Load CSV/TXT scattered points and snap onto a RegularGrid3D.
    Applies z convention: if z_positive_down=True, negates z values.
    """
    
    if has_header:
        df = pd.read_csv(path, delimiter=delimiter, skiprows=skip_rows, engine='python')
        print(f"  Columns found: {list(df.columns)}")
    else:
        df = pd.read_csv(path, delimiter=delimiter, skiprows=skip_rows, header=None, engine='python')
        df.columns = [str(i) for i in range(len(df.columns))]
        print(f"  No header, using column indices: {list(df.columns)}")
    
    def get_col(name):
        if name in df.columns:
            return df[name].values.astype(float)
        try:
            return df.iloc[:, int(name)].values.astype(float)
        except (ValueError, IndexError):
            raise KeyError(f"Column '{name}' not found. Available: {list(df.columns)}")
    
    x = get_col(x_col)
    y = get_col(y_col)
    z = get_col(z_col)
    val = get_col(val_col)
    print(f"  Loaded {len(x):,} points")
    
    # Apply z convention
    if z_positive_down:
        z = -np.abs(z)
        print(f"  Z negated (z_positive_down=True): [{z.min():.1f}, {z.max():.1f}]")
    else:
        print(f"  Z kept as-is: [{z.min():.1f}, {z.max():.1f}]")
    
    # Remove exact zeros
    nonzero = val != 0.0
    if nonzero.sum() < len(val):
        print(f"  Filtered to {nonzero.sum():,} non-zero points")
        x, y, z, val = x[nonzero], y[nonzero], z[nonzero], val[nonzero]
    
    # Determine spacing
    dx = grid_spacing.dx if (grid_spacing and grid_spacing.dx) else median_step(x)
    dy = grid_spacing.dy if (grid_spacing and grid_spacing.dy) else median_step(y)
    dz = grid_spacing.dz if (grid_spacing and grid_spacing.dz) else abs(median_step(z))
    print(f"  Grid spacing: dx={dx:.2f}, dy={dy:.2f}, dz={dz:.2f}")
    
    # Compute grid axes from data extent
    x_min, x_max = x.min(), x.max()
    y_min, y_max = y.min(), y.max()
    z_min, z_max = z.min(), z.max()
    
    nx = int(round((x_max - x_min) / dx)) + 1
    ny = int(round((y_max - y_min) / dy)) + 1
    nz = int(round((z_max - z_min) / dz)) + 1
    print(f"  Grid dimensions: {nx} x {ny} x {nz} = {nx*ny*nz:,} cells")
    
    # Allocate grid (float32 for memory efficiency)
    values = np.zeros((nx, ny, nz), dtype=np.float32)
    counts = np.zeros((nx, ny, nz), dtype=np.int32)
    
    # Snap points to grid cells
    i_idx = np.round((x - x_min) / dx).astype(int)
    j_idx = np.round((y - y_min) / dy).astype(int)
    k_idx = np.round((z - z_min) / dz).astype(int)
    
    # Clamp to bounds
    i_idx = np.clip(i_idx, 0, nx - 1)
    j_idx = np.clip(j_idx, 0, ny - 1)
    k_idx = np.clip(k_idx, 0, nz - 1)
    
    # Accumulate
    np.add.at(values, (i_idx, j_idx, k_idx), val.astype(np.float32))
    np.add.at(counts, (i_idx, j_idx, k_idx), 1)
    
    occupied = counts > 0
    values[occupied] /= counts[occupied]
    
    # Active mask
    Thres = EXCLUSION.Thres
    active = occupied.copy()
    if Thres is not None:
        active &= values >= Thres
    
    n_active = active.sum()
    print(f"  Active cells: {n_active:,} / {nx*ny*nz:,} ({100*n_active/(nx*ny*nz):.1f}%)")
    
    # z_origin = z_min (deepest, most negative), dz > 0 (toward surface)
    return RegularGrid3D(
        x_origin=x_min, y_origin=y_min, z_origin=z_min,
        dx=dx, dy=dy, dz=dz,
        nx=nx, ny=ny, nz=nz,
        values=values, active=active
    )


def downsample_z(grid, z_factor):
    """
    Merge every z_factor consecutive z-layers into one.
    Active rule: any-active (new cell active if ANY source cell active).
    Value rule: max (new cell value = max of active source values).
    """
    if z_factor <= 1:
        return grid
    
    old_nz = grid.nz
    new_nz = ceil(old_nz / z_factor)
    new_dz = grid.dz * z_factor
    
    print(f"  Z-axis downsampling: factor={z_factor}, {old_nz} -> {new_nz} layers, dz={grid.dz:.2f} -> {new_dz:.2f}")
    
    new_values = np.zeros((grid.nx, grid.ny, new_nz), dtype=np.float32)
    new_active = np.zeros((grid.nx, grid.ny, new_nz), dtype=bool)
    
    for k_new in range(new_nz):
        k_start = k_new * z_factor
        k_end = min(k_start + z_factor, old_nz)
        
        slab_active = grid.active[:, :, k_start:k_end]
        slab_values = grid.values[:, :, k_start:k_end]
        
        new_active[:, :, k_new] = slab_active.any(axis=2)
        
        # Max of active values only
        masked = np.where(slab_active, slab_values, -np.inf)
        max_vals = masked.max(axis=2)
        max_vals[~new_active[:, :, k_new]] = 0.0
        new_values[:, :, k_new] = max_vals
    
    result = RegularGrid3D(
        x_origin=grid.x_origin, y_origin=grid.y_origin, z_origin=grid.z_origin,
        dx=grid.dx, dy=grid.dy, dz=new_dz,
        nx=grid.nx, ny=grid.ny, nz=new_nz,
        values=new_values, active=new_active,
    )
    print(f"  Active cells after downsample: {result.n_active:,}")
    return result


def prescan_fault_z(fault_cfg):
    """Pre-scan all fault files to find the minimum (deepest) z value.
    Applies z_positive_down flag. Returns None if no faults."""
    if not fault_cfg.enabled:
        return None
    
    fault_files = fault_cfg.resolve_files()
    if not fault_files:
        return None
    
    z_fault_min = np.inf
    for fp in fault_files:
        data_z = []
        in_data = False
        with open(fp) as f:
            for line in f:
                line = line.strip()
                if line == 'END HEADER':
                    in_data = True
                    continue
                if not in_data or line == '' or line.startswith('#'):
                    continue
                parts = line.split()
                if len(parts) >= 3:
                    try:
                        data_z.append(float(parts[2]))
                    except ValueError:
                        continue
        if data_z:
            z_arr = np.array(data_z)
            if fault_cfg.z_positive_down:
                z_arr = -np.abs(z_arr)
            z_fault_min = min(z_fault_min, float(z_arr.min()))
    
    if np.isinf(z_fault_min):
        return None
    
    print(f"  Fault z pre-scan: z_fault_min = {z_fault_min:.1f}")
    return z_fault_min


def extend_grid_to_full_range(grid, z_min_target, z_max_target=0.0):
    """
    Extend grid so it covers [z_min_target, z_max_target] (typically deepest to surface).
    
    Convention: z_origin = z_min_target (deepest, most negative), dz > 0 (toward surface).
    k=0 is deepest, k=nz-1 is shallowest (closest to surface).
    
    Places existing geobody data at the correct k-offset within the extended grid.
    """
    dz_abs = abs(grid.dz)
    
    # Current geobody z range
    old_z_min = grid.z_origin                                    # deepest
    old_z_max = grid.z_origin + (grid.nz - 1) * dz_abs          # shallowest
    
    # Target range
    new_z_min = min(z_min_target, old_z_min)
    new_z_max = z_max_target  # 0.0 = surface
    
    new_nz = int(round((new_z_max - new_z_min) / dz_abs)) + 1
    
    print(f"  Extending grid z-range:")
    print(f"    Geobody:  [{old_z_min:.1f}, {old_z_max:.1f}] ({grid.nz} layers)")
    print(f"    Target:   [{new_z_min:.1f}, {new_z_max:.1f}] ({new_nz} layers)")
    
    if new_nz == grid.nz and abs(new_z_min - old_z_min) < 1e-6:
        print(f"    No extension needed.")
        return grid
    
    # Allocate extended arrays
    new_values = np.zeros((grid.nx, grid.ny, new_nz), dtype=np.float32)
    new_active = np.zeros((grid.nx, grid.ny, new_nz), dtype=bool)
    
    # Place old data at correct k-offset
    k_offset = int(round((old_z_min - new_z_min) / dz_abs))
    k_end = k_offset + grid.nz
    
    if k_offset >= 0 and k_end <= new_nz:
        new_values[:, :, k_offset:k_end] = grid.values
        new_active[:, :, k_offset:k_end] = grid.active
    else:
        # Fallback: place cell by cell
        for old_k in range(grid.nz):
            old_z = old_z_min + old_k * dz_abs
            new_k = int(round((old_z - new_z_min) / dz_abs))
            if 0 <= new_k < new_nz:
                new_values[:, :, new_k] = grid.values[:, :, old_k]
                new_active[:, :, new_k] = grid.active[:, :, old_k]
    
    result = RegularGrid3D(
        x_origin=grid.x_origin, y_origin=grid.y_origin, z_origin=new_z_min,
        dx=grid.dx, dy=grid.dy, dz=dz_abs,
        nx=grid.nx, ny=grid.ny, nz=new_nz,
        values=new_values, active=new_active,
    )
    
    new_z_axis = result.z_axis
    print(f"    New z range: [{new_z_axis[0]:.1f}, {new_z_axis[-1]:.1f}]")
    print(f"    Active cells: {result.n_active:,}")
    
    return result


# ==============================================================================
# Grid Cache Save / Load
# ==============================================================================

def save_grid_cache(grid, cell_type, fault_id, fault_side, fault_name_to_id,
                    all_fault_contours, filepath):
    """Save grid + fault classification to compressed sparse cache."""
    filepath = Path(filepath)
    
    active_ijk = np.argwhere(grid.active).astype(np.int32)   # (N, 3)
    
    np.savez_compressed(
        filepath,
        meta=np.array([grid.x_origin, grid.y_origin, grid.z_origin,
                        grid.dx, grid.dy, grid.dz,
                        grid.nx, grid.ny, grid.nz]),
        active_ijk=active_ijk,
        active_vals=grid.values[grid.active].astype(np.float32),
        active_cell_type=cell_type[grid.active].astype(np.int8),
        active_fault_id=fault_id[grid.active].astype(np.int8),
        active_fault_side=fault_side[grid.active].astype(np.int8),
    )
    
    # Contours + fault mapping (Shapely objects not numpy-serializable)
    meta_path = filepath.with_suffix('.meta.pkl')
    with open(meta_path, 'wb') as pf:
        pickle.dump({
            'contours': all_fault_contours,
            'fault_name_to_id': fault_name_to_id,
        }, pf)
    
    size_npz = filepath.stat().st_size / (1024 * 1024)
    size_pkl = meta_path.stat().st_size / (1024 * 1024)
    print(f"  Grid cache saved: {filepath.name} ({size_npz:.1f} MB) + .meta.pkl ({size_pkl:.1f} MB)")
    print(f"  Active cells stored: {len(active_ijk):,}")


def load_grid_cache(filepath):
    """Load grid + fault classification from sparse cache."""
    filepath = Path(filepath)
    
    data = np.load(filepath)
    meta = data['meta']
    x_origin, y_origin, z_origin, dx, dy, dz = meta[:6]
    nx, ny, nz = int(meta[6]), int(meta[7]), int(meta[8])
    
    active_ijk = data['active_ijk']   # (N, 3)
    active_vals = data['active_vals']
    active_ct = data['active_cell_type']
    active_fid = data['active_fault_id']
    active_fs = data['active_fault_side']
    
    # Reconstruct full arrays
    values = np.zeros((nx, ny, nz), dtype=np.float32)
    active = np.zeros((nx, ny, nz), dtype=bool)
    cell_type = np.zeros((nx, ny, nz), dtype=np.int8)
    fault_id = np.zeros((nx, ny, nz), dtype=np.int8)
    fault_side = np.zeros((nx, ny, nz), dtype=np.int8)
    
    ii, jj, kk = active_ijk[:, 0], active_ijk[:, 1], active_ijk[:, 2]
    values[ii, jj, kk] = active_vals
    active[ii, jj, kk] = True
    cell_type[ii, jj, kk] = active_ct
    fault_id[ii, jj, kk] = active_fid
    fault_side[ii, jj, kk] = active_fs
    
    grid = RegularGrid3D(
        x_origin=float(x_origin), y_origin=float(y_origin), z_origin=float(z_origin),
        dx=float(dx), dy=float(dy), dz=float(dz),
        nx=nx, ny=ny, nz=nz,
        values=values, active=active,
    )
    
    # Load contours + fault_name_to_id
    meta_path = filepath.with_suffix('.meta.pkl')
    if meta_path.exists():
        with open(meta_path, 'rb') as pf:
            meta_dict = pickle.load(pf)
        all_fault_contours = meta_dict.get('contours', {})
        fault_name_to_id = meta_dict.get('fault_name_to_id', {})
    else:
        all_fault_contours = {}
        fault_name_to_id = {}
    
    print(f"  Grid cache loaded: {grid.nx}x{grid.ny}x{grid.nz}, {grid.n_active:,} active cells")
    return grid, cell_type, fault_id, fault_side, fault_name_to_id, all_fault_contours


def load_data_to_grid(paths, grid_spacing=None):
    fmt = paths.get_format()
    print(f"Loading {fmt.upper()} file: {paths.data_path.name}")
    if fmt == 'segy':
        return load_segy_to_grid(paths.data_path, segy_cfg=SEGY_CFG, z_positive_down=paths.z_positive_down)
    elif fmt in ['csv', 'txt']:
        return load_csv_to_grid(
            path=paths.data_path,
            x_col=paths.x_column, y_col=paths.y_column,
            z_col=paths.z_column, val_col=paths.val_column,
            delimiter=paths.delimiter, skip_rows=paths.skip_rows,
            has_header=paths.has_header, grid_spacing=grid_spacing,
            z_positive_down=paths.z_positive_down,
        )
    else:
        raise ValueError(f"Unsupported format: {fmt}")


# ==============================================================================
# MAIN LOADING SEQUENCE
# ==============================================================================

if PATHS.use_cached_grid and PATHS.grid_cache_path.exists():
    # --- Cache path: skip loading + fault processing ---
    print("=" * 70)
    print("LOADING FROM GRID CACHE")
    print("=" * 70)
    filtered_grid, cell_type, fault_id, fault_side, fault_name_to_id, all_fault_contours = \
        load_grid_cache(PATHS.grid_cache_path)
    
    # Derive viz_category for visualization compatibility
    viz_category = np.zeros_like(cell_type)
    viz_category[cell_type == 1] = 1                                     # gray (geobody)
    viz_category[(cell_type == 2) & (fault_side == 1)] = 2               # yellow (dip)
    viz_category[(cell_type == 2) & (fault_side == 2)] = 3               # blue (anti-dip)
    viz_category[cell_type == 3] = 4                                     # red (overlap)
    category = viz_category
    
    t_elapsed = perf_counter() - t_start
    print(f"\nCache loaded in {t_elapsed:.2f}s")
    print(filtered_grid.summary())

else:
    # --- Normal path: full loading pipeline ---
    
    # Step 1: Load geobody data
    raw_grid = load_data_to_grid(PATHS, GRID_SPACING)
    
    # Step 2: Z-axis downsampling (if configured)
    raw_grid = downsample_z(raw_grid, GRID_SPACING.z_factor)
    
    # Step 3: Pre-scan fault z range
    z_fault_min = prescan_fault_z(FAULT)
    
    # Step 4: Extend grid from deepest depth to surface (0)
    z_geo_min = raw_grid.z_origin  # deepest geobody z
    z_deep = min(z_geo_min, z_fault_min) if z_fault_min is not None else z_geo_min
    raw_grid = extend_grid_to_full_range(raw_grid, z_min_target=z_deep, z_max_target=0.0)
    
    t_elapsed = perf_counter() - t_start
    print(f"\nData loaded in {t_elapsed:.2f}s")
    print(raw_grid.summary())


## Cell 6: Filtering

Deactivates grid cells based on spatial exclusions:
- Existing platform cone masks (from `ExclusionConfig.platform_x/y`)
- Boundary line exclusion (from `ExclusionConfig.boundary_line`)

No data is copied — cells are deactivated by setting `active[i,j,k] = False`.


In [ ]:
# ==============================================================================
# FILTERING — Deactivate excluded cells (no data copying)
# ==============================================================================

def filter_grid(
    grid: RegularGrid3D,
    cone: ConeConfig,
    exclusion: ExclusionConfig
) -> RegularGrid3D:
    """Apply exclusion masks by deactivating grid cells. Returns a copy."""
    
    filtered = grid.copy()
    n_before = filtered.n_active
    
    # Exclude points in existing platform cones
    n_excluded_cones = 0
    for xe, ye in zip(exclusion.platform_x, exclusion.platform_y):
        cmask = cone_mask_on_grid(filtered, xe, ye, cone.z0, cone.tan_theta, cone.hd)
        n_exc = (filtered.active & cmask).sum()
        n_excluded_cones += n_exc
        filtered.active[cmask] = False
    
    if len(exclusion.platform_x) > 0:
        print(f"  Excluded {n_excluded_cones:,} cells in {len(exclusion.platform_x)} cone(s)")
    
    # Exclude cells below boundary line
    if exclusion.boundary_line is not None:
        (x1, y1), (x2, y2) = exclusion.boundary_line
        # Compute mask for all (i, j) positions
        x_ax = filtered.x_axis
        y_ax = filtered.y_axis
        below_2d = mask_below_line(
            np.repeat(x_ax, len(y_ax)),
            np.tile(y_ax, len(x_ax)),
            x1, y1, x2, y2
        ).reshape(filtered.nx, filtered.ny)
        
        # Expand to 3D (all k layers)
        below_3d = below_2d[:, :, np.newaxis].repeat(filtered.nz, axis=2)
        n_excluded_boundary = (filtered.active & below_3d).sum()
        filtered.active[below_3d] = False
        print(f"  Excluded {n_excluded_boundary:,} cells below boundary line")
    
    return filtered


print("Applying exclusion filters...")
filtered_grid = filter_grid(raw_grid, CONE, EXCLUSION)

print(f"\n✓ Filtering complete")
print(f"  Original active: {raw_grid.n_active:,}")
print(f"  Filtered active: {filtered_grid.n_active:,} ({100*filtered_grid.n_active/raw_grid.n_active:.1f}%)")


## Cell 6b: 3D Grid Visualization (HTML)

Exports the full geobody grid as an interactive 3D HTML file using PyVista. Builds hex mesh from active cells only (memory-efficient). Useful for verifying grid geometry before proceeding.

**Output:** `output_dir/3d_scenes/geobody_filtered.html`


In [ ]:
# ==============================================================================
# 3D GRID VISUALIZATION — Interactive HTML export (PyVista)
# ==============================================================================

def build_active_voxels(grid: RegularGrid3D, scalars: np.ndarray = None,
                        scalar_name: str = "value", z_scale: float = 1.0):
    """
    Build PyVista mesh from ONLY active cells (no full-grid allocation).
    
    Instead of creating a 1.2B-cell RectilinearGrid and thresholding,
    this directly constructs hexahedral cells for the ~576K active cells.
    
    For very large active counts (>500K), falls back to PolyData point
    cloud with glyph-free rendering for speed.
    
    Parameters
    ----------
    grid : RegularGrid3D
    scalars : 1D array of values for active cells (default: grid.active_values())
    scalar_name : name for the scalar array
    z_scale : vertical exaggeration
    
    Returns
    -------
    pv.UnstructuredGrid or pv.PolyData
    """
    I, J, K = np.where(grid.active)
    n = len(I)
    
    if n == 0:
        print("  WARNING: No active cells!")
        return pv.PolyData()
    
    if scalars is None:
        scalars = grid.values[grid.active]
    
    hdx, hdy, hdz = grid.dx / 2, grid.dy / 2, (grid.dz / 2) * z_scale
    
    # For moderate cell counts, build proper hexahedral voxels
    if n <= 2_000_000:
        # 8 corners per hexahedron
        cx = grid.x_origin + I.astype(float) * grid.dx
        cy = grid.y_origin + J.astype(float) * grid.dy
        cz = (grid.z_origin + K.astype(float) * grid.dz) * z_scale
        
        # Build corner points for all hexahedra at once
        # VTK hexahedron vertex order: 
        #   bottom face: 0,1,2,3 (counter-clockwise from below)
        #   top face:    4,5,6,7 (counter-clockwise from below)
        offsets_x = np.array([-1, 1, 1, -1, -1, 1, 1, -1], dtype=float) * hdx
        offsets_y = np.array([-1, -1, 1, 1, -1, -1, 1, 1], dtype=float) * hdy
        offsets_z = np.array([-1, -1, -1, -1, 1, 1, 1, 1], dtype=float) * hdz
        
        # Shape: (n, 8, 3)
        points = np.empty((n * 8, 3), dtype=np.float32)
        for v in range(8):
            points[v::8, 0] = cx + offsets_x[v]
            points[v::8, 1] = cy + offsets_y[v]
            points[v::8, 2] = cz + offsets_z[v]
        
        # Cell connectivity: each cell references 8 points
        cell_type = np.full(n, 12, dtype=np.uint8)  # VTK_HEXAHEDRON = 12
        base = np.arange(n, dtype=np.int64) * 8
        cells = np.empty((n, 9), dtype=np.int64)
        cells[:, 0] = 8  # number of points per cell
        for v in range(8):
            cells[:, v + 1] = base + v
        
        ugrid = pv.UnstructuredGrid(cells.ravel(), cell_type, points)
        ugrid.cell_data[scalar_name] = scalars.astype(np.float32)
        print(f"  Built {n:,} hexahedral voxels ({points.nbytes / 1e6:.0f} MB points)")
        return ugrid
    
    else:
        # For very large counts, use PolyData (point cloud) — much lighter
        cx = grid.x_origin + I.astype(float) * grid.dx
        cy = grid.y_origin + J.astype(float) * grid.dy
        cz = (grid.z_origin + K.astype(float) * grid.dz) * z_scale
        
        cloud = pv.PolyData(np.column_stack([cx, cy, cz]).astype(np.float32))
        cloud.point_data[scalar_name] = scalars.astype(np.float32)
        print(f"  Built point cloud with {n:,} points ({cloud.points.nbytes / 1e6:.0f} MB)")
        return cloud


def export_grid_html(
    grid: RegularGrid3D,
    outfile: Path,
    scalar_name: str = "Probability",
    cmap: str = "viridis",
    clim: tuple = None,
    opacity: float = 0.85,
    show_outline: bool = True,
    show_axes: bool = True,
    show_bounds: bool = True,
    z_scale: float = 1.0,
):
    """
    Export RegularGrid3D as interactive 3D HTML — memory-efficient version.
    
    Builds mesh from ONLY active cells (no full-grid RectilinearGrid).
    Adds an outline box from z=0 to z_min to show full depth extent.
    """
    scalars = grid.active_values()
    mesh = build_active_voxels(grid, scalars, scalar_name, z_scale)
    
    if mesh.n_cells == 0 and mesh.n_points == 0:
        print("  WARNING: No active cells to display!")
        return
    
    if clim is None:
        vals = mesh.cell_data.get(scalar_name, mesh.point_data.get(scalar_name, None))
        if vals is not None and len(vals) > 0:
            clim = (float(vals.min()), float(vals.max()))
    
    pv.global_theme.background = 'white'
    pv.global_theme.window_size = (1400, 1000)
    pl = pv.Plotter(off_screen=True)
    
    is_cloud = isinstance(mesh, pv.PolyData) and mesh.n_cells == 0
    
    if is_cloud:
        pl.add_mesh(
            mesh,
            scalars=scalar_name,
            cmap=cmap,
            clim=clim,
            point_size=3,
            render_points_as_spheres=False,
            opacity=opacity,
            scalar_bar_args={
                "title": scalar_name, "vertical": True,
                "position_x": 0.87, "position_y": 0.1,
                "width": 0.06, "height": 0.7, "fmt": "%.2f",
            },
        )
    else:
        pl.add_mesh(
            mesh,
            scalars=scalar_name,
            cmap=cmap,
            clim=clim,
            opacity=opacity,
            show_edges=False,
            scalar_bar_args={
                "title": scalar_name, "vertical": True,
                "position_x": 0.87, "position_y": 0.1,
                "width": 0.06, "height": 0.7, "fmt": "%.2f",
            },
        )
    
    # Outline box: from z=0 (surface) to deepest grid extent
    if show_outline:
        x_min = grid.x_origin - grid.dx / 2
        x_max = grid.x_origin + (grid.nx - 0.5) * grid.dx
        y_min = grid.y_origin - grid.dy / 2
        y_max = grid.y_origin + (grid.ny - 0.5) * grid.dy
        z_min_box = (grid.z_origin + (grid.nz - 0.5) * grid.dz) * z_scale
        z_max_box = 0.0 * z_scale  # surface = 0
        
        outline = pv.Box(bounds=[x_min, x_max, y_min, y_max, z_min_box, z_max_box])
        pl.add_mesh(outline.outline(), color='#555555', line_width=1.5)
    
    if show_axes:
        pl.add_axes(line_width=2)
    if show_bounds:
        pl.show_bounds(grid=True, location='outer', all_edges=True,
                       font_size=8, color='gray')
    
    pl.camera_position = 'iso'
    pl.camera.zoom(1.3)
    pl.export_html(str(outfile))
    pl.close()
    
    size_mb = os.path.getsize(str(outfile)) / 1024 / 1024
    print(f"  ✓ Exported: {outfile} ({size_mb:.1f} MB, {mesh.n_cells or mesh.n_points:,} cells/points)")


# Export filtered grid
viz_dir = PATHS.output_dir / "3d_scenes"
viz_dir.mkdir(parents=True, exist_ok=True)

print("Exporting interactive 3D grid visualization...")
export_grid_html(
    filtered_grid,
    outfile=viz_dir / "geobody_filtered.html",
    scalar_name="Probability",
    cmap="viridis",
    clim=(EXCLUSION.Thres, 1.0),
    z_scale=1.0,
)


## Cell 6c: Fault Zone Processing

**Core fault integration pipeline.** For each fault file:

1. Load Petrel scattered picks (X, Y, Z)
2. Interpolate fault surface via `LinearNDInterpolator`
3. Extract contour at each grid depth layer
4. Cleanup: Resample → Smooth → **U-fold trim (PCA)** → Curvature → Zigzag → Settling → Monotonic
5. Buffer each contour into dip/anti-dip polygons (Shapely single-sided buffer)
6. Rasterize onto grid → stamp `fault_id`, `fault_side`, `cell_type`

**Deactivated fault handling:** Faults with `fault_volumes[name, side] = None` are excluded before rasterization — their cells never enter the grid, preventing false overlap labels.

**Fault value assignment (volume mode):**
```
V_fault_cell = (Total_fault_vol − Σ overlap_geobody_vol) / N_fault_only_cells
```
Overlap cells keep their geobody-derived volume. If overlap volume ≥ total defined, fault-only cells are deactivated.

**User parameters:** `FaultConfig` (files, buffer distance, trim distance, contour cleanup), `VolumeConfig.fault_volumes` (total BCF per fault side, or None to deactivate).


In [ ]:
# ==============================================================================
# FAULT PROCESSING — Contour extraction, cleanup, buffer, rasterization
# ==============================================================================

def load_petrel_fault(filepath, z_positive_down=False):
    """Parse Petrel fault file -> (N, 3) array of [X, Y, Z].
    Applies z convention if z_positive_down=True."""
    data, in_data = [], False
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if line == 'END HEADER':
                in_data = True
                continue
            if not in_data or line == '' or line.startswith('#'):
                continue
            parts = line.split()
            if len(parts) >= 3:
                try:
                    data.append([float(parts[0]), float(parts[1]), float(parts[2])])
                except ValueError:
                    continue
    pts = np.array(data)
    if len(pts) > 0 and z_positive_down:
        pts[:, 2] = -np.abs(pts[:, 2])
    return pts


def resample_curve(xy, spacing):
    """Resample 2D curve to uniform arc-length spacing."""
    if spacing <= 0:
        return xy
    d = np.sqrt(np.sum(np.diff(xy, axis=0)**2, axis=1))
    s = np.concatenate([[0], np.cumsum(d)])
    total = s[-1]
    if total < spacing * 3:
        return xy
    n_pts = max(10, int(total / spacing))
    s_new = np.linspace(0, total, n_pts)
    x_new = np.interp(s_new, s, xy[:, 0])
    y_new = np.interp(s_new, s, xy[:, 1])
    return np.column_stack([x_new, y_new])

def smooth_curve(xy, window):
    if window <= 1:
        return xy
    xs = uniform_filter1d(xy[:, 0], window, mode='nearest')
    ys = uniform_filter1d(xy[:, 1], window, mode='nearest')
    return np.column_stack([xs, ys])

def curvature_trim_tips(xy, max_angle_deg=60.0, scan_fraction=0.15):
    if len(xy) < 20:
        return xy
    d = np.diff(xy, axis=0)
    angles = np.arctan2(d[:, 1], d[:, 0])
    turn = np.abs(np.diff(angles))
    turn = np.minimum(turn, 2 * np.pi - turn)
    turn_deg = np.degrees(turn)
    n = len(xy)
    scan_n = max(5, int(n * scan_fraction))
    start = 0
    for k in range(min(scan_n, len(turn_deg))):
        if turn_deg[k] > max_angle_deg:
            start = k + 2
        else:
            break
    end = n
    for k in range(len(turn_deg) - 1, max(len(turn_deg) - scan_n - 1, -1), -1):
        if k < len(turn_deg) and turn_deg[k] > max_angle_deg:
            end = k
        else:
            break
    start = min(start, n // 4)
    end = max(end, 3 * n // 4)
    if end <= start + 10:
        return xy
    return xy[start:end]

def zigzag_trim_tips(xy, scan_fraction=0.20, short_factor=0.5, angle_thresh=35.0):
    if len(xy) < 20:
        return xy
    seg_len = np.sqrt(np.sum(np.diff(xy, axis=0)**2, axis=1))
    d = np.diff(xy, axis=0)
    angles = np.arctan2(d[:, 1], d[:, 0])
    turn = np.abs(np.diff(angles))
    turn = np.minimum(turn, 2 * np.pi - turn)
    turn_deg = np.degrees(turn)
    n = len(xy)
    scan_n = max(8, int(n * scan_fraction))
    median_seg = np.median(seg_len)
    short_thresh = median_seg * short_factor
    start = 0
    for k in range(min(scan_n, len(turn_deg))):
        is_short = seg_len[k] < short_thresh if k < len(seg_len) else False
        is_sharp = turn_deg[k] > angle_thresh if k < len(turn_deg) else False
        if (is_short and is_sharp) or (is_sharp and k < 5):
            start = k + 2
    end = n
    for k in range(len(turn_deg) - 1, max(len(turn_deg) - scan_n - 1, -1), -1):
        is_short = seg_len[min(k+1, len(seg_len)-1)] < short_thresh
        is_sharp = turn_deg[k] > angle_thresh
        if (is_short and is_sharp) or (is_sharp and k >= len(turn_deg) - 5):
            end = min(end, k)
    start = min(start, n // 4)
    end = max(end, 3 * n // 4)
    if end <= start + 10:
        return xy
    return xy[start:end]

def settling_trim_tips(xy, scan_fraction=0.10, deviate_thresh=50.0):
    if len(xy) < 30:
        return xy
    n = len(xy)
    scan_n = max(5, int(n * scan_fraction))
    seg = np.diff(xy, axis=0)
    seg_angles = np.arctan2(seg[:, 1], seg[:, 0])
    q_start = max(scan_n, n // 10)
    q_end_ref = min(n - scan_n, n // 10 + scan_n)
    if q_end_ref <= q_start:
        return xy
    start_ref_angle = np.mean(seg_angles[q_start:q_end_ref])
    end_ref_start = max(0, n - 1 - q_end_ref)
    end_ref_end = n - 1 - q_start
    if end_ref_end <= end_ref_start or end_ref_end >= len(seg_angles):
        return xy
    end_ref_angle = np.mean(seg_angles[end_ref_start:end_ref_end])
    start = 0
    for k in range(min(scan_n, len(seg_angles))):
        dev = np.abs(seg_angles[k] - start_ref_angle)
        dev = min(dev, 2 * np.pi - dev)
        if np.degrees(dev) > deviate_thresh:
            start = k + 1
    end = n
    for k in range(len(seg_angles) - 1, max(len(seg_angles) - scan_n - 1, -1), -1):
        dev = np.abs(seg_angles[k] - end_ref_angle)
        dev = min(dev, 2 * np.pi - dev)
        if np.degrees(dev) > deviate_thresh:
            end = k + 1
    start = min(start, n // 4)
    end = max(end, 3 * n // 4)
    if end <= start + 10:
        return xy
    return xy[start:end]

def monotonic_trim_tips(xy, scan_fraction=0.10, reversal_thresh=120.0):
    if len(xy) < 20:
        return xy
    n = len(xy)
    scan_n = max(3, int(n * scan_fraction))
    seg = np.diff(xy, axis=0)
    angles = np.arctan2(seg[:, 1], seg[:, 0])
    turn = np.abs(np.diff(angles))
    turn = np.minimum(turn, 2 * np.pi - turn)
    turn_deg = np.degrees(turn)
    start = 0
    for k in range(min(scan_n, len(turn_deg))):
        if turn_deg[k] > reversal_thresh:
            start = k + 2
        else:
            break
    end = n
    for k in range(len(turn_deg) - 1, max(len(turn_deg) - scan_n - 1, -1), -1):
        if turn_deg[k] > reversal_thresh:
            end = k
        else:
            break
    start = min(start, n // 4)
    end = max(end, 3 * n // 4)
    if end <= start + 10:
        return xy
    return xy[start:end]

def arc_length_trim_xy(xy, trim_dist):
    d = np.sqrt(np.sum(np.diff(xy, axis=0)**2, axis=1))
    s = np.concatenate([[0], np.cumsum(d)])
    mask = (s >= trim_dist) & (s <= s[-1] - trim_dist)
    if mask.sum() < 5:
        return None
    return xy[mask]


def label_dip_side(xy, eval_gradient_fn):
    mid = len(xy) // 2
    xc, yc = xy[mid]
    dx = xy[min(mid+1, len(xy)-1), 0] - xy[max(mid-1, 0), 0]
    dy = xy[min(mid+1, len(xy)-1), 1] - xy[max(mid-1, 0), 1]
    tm = max(np.hypot(dx, dy), 1e-12)
    dx, dy = dx / tm, dy / tm
    grad = eval_gradient_fn(xc, yc)
    if grad is None or len(grad) < 2:
        return 'left'
    gx, gy = -float(grad[0]), -float(grad[1])
    dot_left = gx * (-dy) + gy * dx
    dot_right = gx * dy + gy * (-dx)
    return 'left' if dot_left > dot_right else 'right'


def extract_polygon_coords(geom):
    if geom is None or geom.is_empty:
        return None
    if geom.geom_type == 'MultiPolygon':
        geom = max(geom.geoms, key=lambda g: g.area)
    if geom.geom_type != 'Polygon':
        return None
    return np.array(geom.exterior.coords)


def rasterize_polygon_to_grid_layer(poly_coords, grid_points_xy):
    if poly_coords is None or len(poly_coords) < 3:
        return None
    path = MplPath(poly_coords[:, :2])
    return path.contains_points(grid_points_xy)


def extract_all_contours_batch(Xg, Yg, Zg, z_levels):
    fig_tmp, ax_tmp = plt.subplots()
    cs = ax_tmp.contour(Xg, Yg, Zg, levels=z_levels)
    plt.close(fig_tmp)
    contours = {}
    for i, z_lev in enumerate(z_levels):
        if i >= len(cs.allsegs):
            continue
        segs = cs.allsegs[i]
        if not segs:
            continue
        longest = max(segs, key=lambda s: len(s))
        xy = np.array(longest)
        if len(xy) >= 10:
            contours[z_lev] = xy
    return contours


# ==============================================================================
# PART B: Per-Fault Processing Pipeline (with fault_id stamping)
# ==============================================================================

def process_single_fault(
    fault_file, grid, offset_distance, trim_distance, grid_res,
    resample_spacing, smooth_window, max_turn_angle, fault_value,
    this_fault_id, z_positive_down=False,
    active_sides=('dip', 'anti'),
):
    """
    Full pipeline for one fault file -> dip/anti-dip masks + fault_id on the grid.
    
    Returns: (fault_dip, fault_anti, fault_id_arr, fault_contours, fault_name, k_min, k_max)
             or (None, None, None, {}, fault_name, None, None) on failure
    """
    fault_name = os.path.basename(str(fault_file))
    print(f"\n  -- Processing fault: {fault_name} (id={this_fault_id}) --")
    
    # Step 1: Load fault picks (with z convention)
    fault_pts = load_petrel_fault(str(fault_file), z_positive_down=z_positive_down)
    if len(fault_pts) < 10:
        print(f"    WARNING: Only {len(fault_pts)} points, skipping.")
        return None, None, None, {}, fault_name, None, None
    
    x_raw, y_raw, z_raw = fault_pts.T
    print(f"    Loaded {len(fault_pts)} picks")
    print(f"    X: [{x_raw.min():.1f}, {x_raw.max():.1f}]")
    print(f"    Y: [{y_raw.min():.1f}, {y_raw.max():.1f}]")
    print(f"    Z: [{z_raw.min():.1f}, {z_raw.max():.1f}]")
    
    # Step 2: Interpolate fault surface
    interp = LinearNDInterpolator(fault_pts[:, :2], z_raw)
    
    def eval_gradient(x, y, dx_step=10.0):
        x, y = np.atleast_1d(x).ravel(), np.atleast_1d(y).ravel()
        zc = interp(x, y)
        zx = interp(x + dx_step, y)
        zy = interp(x, y + dx_step)
        gx = np.where(np.isfinite(zx) & np.isfinite(zc), (zx - zc) / dx_step, 0.0)
        gy = np.where(np.isfinite(zy) & np.isfinite(zc), (zy - zc) / dx_step, 0.0)
        return np.column_stack([gx, gy]) if len(gx) > 1 else np.array([gx[0], gy[0]])
    
    # Build evaluation grid for contour extraction
    xg = np.linspace(x_raw.min(), x_raw.max(), grid_res)
    yg = np.linspace(y_raw.min(), y_raw.max(), grid_res)
    Xg, Yg = np.meshgrid(xg, yg)
    Zg = interp(Xg, Yg)
    
    z_surface_vals = Zg[~np.isnan(Zg)]
    if len(z_surface_vals) == 0:
        print(f"    WARNING: Interpolation produced all NaN, skipping.")
        return None, None, None, {}, fault_name, None, None
    
    z_fault_min = float(z_surface_vals.min())
    z_fault_max = float(z_surface_vals.max())
    print(f"    Interpolated Z range: [{z_fault_min:.0f}, {z_fault_max:.0f}]")
    
    # Step 3: Find relevant z-layers ONLY
    z_axis = grid.z_axis
    x_axis = grid.x_axis
    y_axis = grid.y_axis
    
    relevant_mask = (z_axis >= z_fault_min) & (z_axis <= z_fault_max)
    relevant_k = np.where(relevant_mask)[0]
    
    if len(relevant_k) == 0:
        print(f"    WARNING: No grid layers overlap with fault z-range, skipping.")
        return None, None, None, {}, fault_name, None, None
    
    k_min, k_max = relevant_k[0], relevant_k[-1]
    relevant_z_levels = z_axis[relevant_mask]
    print(f"    Relevant grid layers: {len(relevant_k)} (k={k_min}..{k_max}), "
          f"skipping {grid.nz - len(relevant_k)} layers")
    
    # Step 4: Batch extract ALL contours
    print(f"    Extracting contours (batch)...")
    raw_contours = extract_all_contours_batch(Xg, Yg, Zg, relevant_z_levels)
    print(f"    Got {len(raw_contours)} valid contours from {len(relevant_z_levels)} levels")
    
    del Xg, Yg, Zg, z_surface_vals, fault_pts
    
    if len(raw_contours) == 0:
        print(f"    WARNING: No valid contours extracted, skipping.")
        return None, None, None, {}, fault_name, None, None
    
    # Step 5: Pre-compute grid meshgrid ONCE
    XX, YY = np.meshgrid(x_axis, y_axis, indexing='ij')
    grid_points_xy = np.column_stack([XX.ravel(), YY.ravel()])
    del XX, YY
    nx, ny = len(x_axis), len(y_axis)
    
    # Step 6: Allocate partial k-range arrays for memory efficiency
    k_range_size = k_max - k_min + 1
    fault_dip = np.zeros((grid.nx, grid.ny, k_range_size), dtype=bool)
    fault_anti = np.zeros((grid.nx, grid.ny, k_range_size), dtype=bool)
    fault_id_arr = np.zeros((grid.nx, grid.ny, k_range_size), dtype=np.int8)
    fault_contours = {}
    
    n_filled = 0
    z_to_k = {z_axis[k]: k for k in relevant_k}
    contour_items = [(z_lev, raw_contours[z_lev]) for z_lev in sorted(raw_contours.keys())]
    
    for z_lev, xy in tqdm(contour_items, desc=f"    Processing {fault_name}", leave=False):
        k = z_to_k.get(z_lev)
        if k is None:
            continue
        
        k_local = k - k_min  # index into partial arrays
        
        # Clean contour
        xy = resample_curve(xy, resample_spacing)
        xy = smooth_curve(xy, smooth_window)
        xy = curvature_trim_tips(xy, max_turn_angle)
        xy = zigzag_trim_tips(xy)
        xy = settling_trim_tips(xy)
        xy = monotonic_trim_tips(xy)
        
        if len(xy) < 10:
            continue
        
        if trim_distance > 0:
            xy_t = arc_length_trim_xy(xy, trim_distance)
            if xy_t is None or len(xy_t) < 10:
                continue
            xy = xy_t
        
        # Build Shapely buffers
        line = LineString(xy)
        buf_left = line.buffer(offset_distance, single_sided=True,
                               cap_style='flat', join_style='round')
        buf_right = line.buffer(-offset_distance, single_sided=True,
                                cap_style='flat', join_style='round')
        
        dip_side = label_dip_side(xy, eval_gradient)
        if dip_side == 'left':
            buf_dip, buf_anti = buf_left, buf_right
        else:
            buf_dip, buf_anti = buf_right, buf_left
        
        # Rasterize only active sides
        if 'dip' in active_sides:
            dip_coords = extract_polygon_coords(buf_dip)
            dip_mask = rasterize_polygon_to_grid_layer(dip_coords, grid_points_xy)
            if dip_mask is not None:
                dip_2d = dip_mask.reshape(nx, ny)
                fault_dip[:, :, k_local] |= dip_2d
                fault_id_arr[dip_2d, k_local] = this_fault_id
        
        if 'anti' in active_sides:
            anti_coords = extract_polygon_coords(buf_anti)
            anti_mask = rasterize_polygon_to_grid_layer(anti_coords, grid_points_xy)
            if anti_mask is not None:
                anti_2d = anti_mask.reshape(nx, ny)
                fault_anti[:, :, k_local] |= anti_2d
                fault_id_arr[anti_2d, k_local] = this_fault_id
        
        fault_contours[k] = {
            'z_level': z_lev,
            'xy': xy,
            'line': line,
            'dip_side': dip_side,
            'buffer_dip': buf_dip,
            'buffer_anti': buf_anti,
        }
        
        n_filled += 1
    
    del grid_points_xy, raw_contours
    
    print(f"    Layers with valid fault zone: {n_filled}")
    print(f"    Fault dip cells:  {fault_dip.sum():,}")
    print(f"    Fault anti cells: {fault_anti.sum():,}")
    
    return fault_dip, fault_anti, fault_id_arr, fault_contours, fault_name, k_min, k_max



# ── Multi-Fault Aggregation + Value Assignment ─────────────────────────────

def integrate_all_faults(grid, fault_cfg, volume_cfg):
    """
    Process all fault files and merge results onto the grid.
    
    Returns
    -------
    cell_type : int8 (nx, ny, nz) -- 0=empty, 1=geobody, 2=fault only, 3=overlap
    fault_id : int8 (nx, ny, nz) -- 0=no fault, 1..N=fault file index
    fault_side : int8 (nx, ny, nz) -- 0=no fault, 1=dip, 2=anti-dip
    fault_name_to_id : dict {name: int}
    all_fault_contours : dict {fault_name: {k: slice_info}}
    viz_category : int8 (nx, ny, nz) -- 0-4 for visualization
    """
    fault_files = fault_cfg.resolve_files()
    
    if not fault_files:
        print("  No fault files found!")
        return None, None, None, {}, {}, None
    
    print(f"  Found {len(fault_files)} fault file(s):")
    for fp in fault_files:
        print(f"    - {fp}")
    
    # Accumulate across all faults
    all_dip = np.zeros((grid.nx, grid.ny, grid.nz), dtype=bool)
    all_anti = np.zeros((grid.nx, grid.ny, grid.nz), dtype=bool)
    all_fault_id = np.zeros((grid.nx, grid.ny, grid.nz), dtype=np.int8)
    all_contours = {}
    fault_name_to_id = {}
    
    for fault_idx, fault_file in enumerate(tqdm(fault_files, desc="Processing faults"), start=1):
        fault_name_stem = os.path.basename(str(fault_file))
        if '.' in fault_name_stem:
            fault_name_stem = os.path.splitext(fault_name_stem)[0]
        fault_name_to_id[fault_name_stem] = fault_idx
        
        # Check if this fault has any active sides before processing
        if volume_cfg.mode == 'volume':
            dip_vol = volume_cfg.fault_volumes.get((fault_name_stem, "dip"), "MISSING")
            anti_vol = volume_cfg.fault_volumes.get((fault_name_stem, "anti"), "MISSING")
            dip_active = dip_vol not in (None,)
            anti_active = anti_vol not in (None,)
            
            if not dip_active and not anti_active:
                print(f"\n  -- Skipping fault: {fault_name_stem} (both sides deactivated) --")
                continue
        else:
            dip_active = True
            anti_active = True
        
        # Determine which sides are active
        active_sides = []
        if dip_active:
            active_sides.append('dip')
        if anti_active:
            active_sides.append('anti')
        
        result = process_single_fault(
            fault_file=fault_file,
            grid=grid,
            offset_distance=fault_cfg.offset_distance,
            trim_distance=fault_cfg.trim_distance,
            grid_res=fault_cfg.grid_res,
            resample_spacing=fault_cfg.resample_spacing,
            smooth_window=fault_cfg.smooth_window,
            max_turn_angle=fault_cfg.max_turn_angle,
            fault_value=fault_cfg.fault_value,
            this_fault_id=fault_idx,
            z_positive_down=fault_cfg.z_positive_down,
            active_sides=tuple(active_sides),
        )
        
        dip, anti, fid_arr, contours, name, k_min, k_max = result
        
        if dip is not None:
            k_range_size = k_max - k_min + 1
            
            # Merge — deactivated sides were already skipped in process_single_fault
            all_dip[:, :, k_min:k_min+k_range_size] |= dip
            all_anti[:, :, k_min:k_min+k_range_size] |= anti
            
            fid_mask = fid_arr > 0
            all_fault_id[:, :, k_min:k_min+k_range_size][fid_mask] = fid_arr[fid_mask]
            
            all_contours[name] = contours
            del dip, anti, fid_arr
            gc.collect()
    
    print(f"\n  Fault name -> ID mapping: {fault_name_to_id}")
    
    # ── Classify cells: cell_type, fault_id, fault_side ───────────────────────
    cell_type = np.zeros((grid.nx, grid.ny, grid.nz), dtype=np.int8)
    fault_side = np.zeros((grid.nx, grid.ny, grid.nz), dtype=np.int8)
    
    # Cell type
    cell_type[grid.active] = 1                                     # geobody only
    cell_type[all_dip & ~grid.active] = 2                          # fault only
    cell_type[all_anti & ~grid.active] = 2                         # fault only
    cell_type[(all_dip | all_anti) & grid.active] = 3              # overlap
    
    # Fault side
    fault_side[all_dip] = 1    # dip
    fault_side[all_anti] = 2   # anti-dip (overwrites dip where both exist)
    
    # fault_id already accumulated
    fault_id = all_fault_id
    
    # Free accumulation arrays
    del all_dip, all_anti, all_fault_id
    gc.collect()
    
    n_geo = int((cell_type == 1).sum())
    n_fault = int((cell_type == 2).sum())
    n_overlap = int((cell_type == 3).sum())
    
    print(f"\n  Cell classification:")
    print(f"    Geobody only (type=1): {n_geo:,}")
    print(f"    Fault only   (type=2): {n_fault:,}")
    print(f"    Overlap      (type=3): {n_overlap:,}")
    
    # ── Merge fault cells into grid ───────────────────────────────────────────
    if fault_cfg.merge_into_grid:
        new_fault = (cell_type == 2)
        overlap = (cell_type == 3)
        
        grid.active[new_fault] = True
        grid.values[new_fault] = fault_cfg.fault_value
        grid.values[overlap] = np.maximum(grid.values[overlap], fault_cfg.fault_value)
        
        del new_fault, overlap
        gc.collect()
        
        print(f"\n  Merged fault into grid: {grid.n_active:,} total active cells")
    
    # ── Value assignment (volume mode) ────────────────────────────────────────
    if volume_cfg.mode == 'volume':
        print(f"\n  Applying volume conversion (mode='volume')...")
        bulk_vol = grid.dx * grid.dy * grid.dz
        hcV = (bulk_vol * volume_cfg.poro * volume_cfg.Sg * volume_cfg.RF * volume_cfg.m3_to_ft3 * volume_cfg.unit_factor / volume_cfg.Bg)
        print(f"    Bulk volume per cell: {bulk_vol:.2f} m3")
        print(f"    HcV multiplier: {hcV:.6e}")
        
        # Step 1: Geobody cells -- probability * hcV
        geo_mask = (cell_type == 1) | (cell_type == 3)
        grid.values[geo_mask] *= hcV
        print(f"    Geobody+overlap cells scaled: {geo_mask.sum():,}")
        
        # Step 2: Fault-only cells -- total_vol / n_cells per (fault, side)
        # If total_vol is None, deactivate those cells (no hydrocarbon value)
        for (fname, side_str), total_vol in volume_cfg.fault_volumes.items():
            fid = fault_name_to_id.get(fname)
            if fid is None:
                print(f"    WARNING: fault '{fname}' not found in fault_name_to_id, skipping")
                continue
            side_val = 1 if side_str == 'dip' else 2
            
            mask_fault_only = (fault_id == fid) & (fault_side == side_val) & (cell_type == 2)
            mask_overlap    = (fault_id == fid) & (fault_side == side_val) & (cell_type == 3)
            n_fault_only = int(mask_fault_only.sum())
            n_overlap = int(mask_overlap.sum())
            
            if n_fault_only == 0 and n_overlap == 0:
                continue
            
            if total_vol is None:
                # Already pre-filtered before classification — skip
                # (fault mask was cleared, so mask_fault_only should be empty)
                if n_fault_only > 0:
                    # Safety: deactivate any remaining cells
                    grid.active[mask_fault_only] = False
                    grid.values[mask_fault_only] = 0.0
                    cell_type[mask_fault_only] = 0
                    print(f"    ({fname}, {side_str}): None -- deactivated {n_fault_only:,} residual cells")
                continue
            
            # Overlap cells already have geobody volume from Step 1
            overlap_vol_sum = float(grid.values[mask_overlap].sum())
            remaining_vol = total_vol - overlap_vol_sum
            
            if n_fault_only > 0 and remaining_vol > 0:
                per_cell = remaining_vol / n_fault_only
                grid.values[mask_fault_only] = per_cell
                print(f"    ({fname}, {side_str}): ({total_vol:.2f} - {overlap_vol_sum:.2f} overlap) / {n_fault_only:,} = {per_cell:.6e} per cell")
            elif n_fault_only > 0:
                # Overlap geobody already exceeds defined fault volume
                grid.active[mask_fault_only] = False
                grid.values[mask_fault_only] = 0.0
                cell_type[mask_fault_only] = 0
                print(f"    ({fname}, {side_str}): overlap vol ({overlap_vol_sum:.2f}) >= total ({total_vol:.2f}), deactivated {n_fault_only:,} fault-only cells")
            else:
                print(f"    ({fname}, {side_str}): no fault-only cells, {n_overlap:,} overlap cells retain geobody values")
        
        print(f"    Value range after volume conversion: "
              f"[{grid.values[grid.active].min():.6e}, {grid.values[grid.active].max():.6e}]")
    
    # --- Derive viz_category for visualization compatibility ---
    viz_category = np.zeros_like(cell_type)
    viz_category[cell_type == 1] = 1                                     # gray (geobody)
    viz_category[(cell_type == 2) & (fault_side == 1)] = 2               # yellow (dip)
    viz_category[(cell_type == 2) & (fault_side == 2)] = 3               # blue (anti-dip)
    viz_category[cell_type == 3] = 4                                     # red (overlap)
    
    return cell_type, fault_id, fault_side, fault_name_to_id, all_contours, viz_category



# ── Execute ─────────────────────────────────────────────────────────────────

if FAULT.enabled and not (PATHS.use_cached_grid and PATHS.grid_cache_path.exists()):
    print("=" * 70)
    print("FAULT ZONE INTEGRATION")
    print("=" * 70)
    
    result = integrate_all_faults(
        grid=filtered_grid,
        fault_cfg=FAULT,
        volume_cfg=VOLUME,
    )
    
    if result[0] is not None:
        cell_type, fault_id, fault_side, fault_name_to_id, all_fault_contours, category = result
        
        # Auto-save grid cache
        print(f"\n  Saving grid cache...")
        save_grid_cache(
            filtered_grid, cell_type, fault_id, fault_side,
            fault_name_to_id, all_fault_contours,
            PATHS.grid_cache_path,
        )
    else:
        print("  No faults processed, creating default arrays...")
        cell_type = np.zeros((filtered_grid.nx, filtered_grid.ny, filtered_grid.nz), dtype=np.int8)
        cell_type[filtered_grid.active] = 1
        fault_id = np.zeros_like(cell_type)
        fault_side = np.zeros_like(cell_type)
        fault_name_to_id = {}
        all_fault_contours = {}
        category = cell_type.copy()

elif not FAULT.enabled:
    print("Fault integration disabled (set FAULT.enabled = True in FaultConfig)")
    # Create default classification arrays
    cell_type = np.zeros((filtered_grid.nx, filtered_grid.ny, filtered_grid.nz), dtype=np.int8)
    cell_type[filtered_grid.active] = 1
    fault_id = np.zeros_like(cell_type)
    fault_side = np.zeros_like(cell_type)
    fault_name_to_id = {}
    all_fault_contours = {}
    category = cell_type.copy()


## Cell 6d: 3D Fault + Geobody Visualization (HTML)

Interactive 3D scene with 4 color categories: Gray (geobody), Yellow (fault dip), Blue (fault anti-dip), Red (overlap).

**Output:** `output_dir/3d_scenes/geobody_faults.html`


In [ ]:
# ==============================================================================
# 3D FAULT + GEOBODY VISUALIZATION — Category-colored HTML export
# ==============================================================================

def build_category_voxels(category, grid, cat_val, z_scale=1.0):
    """Build PyVista mesh for cells matching a specific category value."""
    mask = (category == cat_val)
    I, J, K = np.where(mask)
    n = len(I)
    if n == 0:
        return None, 0
    
    hdx, hdy, hdz = grid.dx / 2, grid.dy / 2, (grid.dz / 2) * z_scale
    
    if n <= 2_000_000:
        cx = grid.x_origin + I.astype(float) * grid.dx
        cy = grid.y_origin + J.astype(float) * grid.dy
        cz = (grid.z_origin + K.astype(float) * grid.dz) * z_scale
        
        offsets_x = np.array([-1, 1, 1, -1, -1, 1, 1, -1], dtype=float) * hdx
        offsets_y = np.array([-1, -1, 1, 1, -1, -1, 1, 1], dtype=float) * hdy
        offsets_z = np.array([-1, -1, -1, -1, 1, 1, 1, 1], dtype=float) * hdz
        
        points = np.empty((n * 8, 3), dtype=np.float32)
        for v in range(8):
            points[v::8, 0] = cx + offsets_x[v]
            points[v::8, 1] = cy + offsets_y[v]
            points[v::8, 2] = cz + offsets_z[v]
        
        cell_type = np.full(n, 12, dtype=np.uint8)
        base = np.arange(n, dtype=np.int64) * 8
        cells = np.empty((n, 9), dtype=np.int64)
        cells[:, 0] = 8
        for v in range(8):
            cells[:, v + 1] = base + v
        
        mesh = pv.UnstructuredGrid(cells.ravel(), cell_type, points)
        return mesh, n
    else:
        cx = grid.x_origin + I.astype(float) * grid.dx
        cy = grid.y_origin + J.astype(float) * grid.dy
        cz = (grid.z_origin + K.astype(float) * grid.dz) * z_scale
        cloud = pv.PolyData(np.column_stack([cx, cy, cz]).astype(np.float32))
        return cloud, n


def export_fault_geobody_html(
    category, grid, outfile,
    z_scale=1.0, opacity_geo=0.3, opacity_fault=0.7, opacity_overlap=1.0,
):
    """
    Export combined geobody + fault zone 3D HTML — memory-efficient.
    Builds meshes from ONLY classified cells (no full-grid allocation).
    Gray=geobody, Yellow=dip, Blue=anti-dip, Red=overlap.
    """
    pv.global_theme.background = 'white'
    pv.global_theme.window_size = (1600, 1100)
    pl = pv.Plotter(off_screen=True)
    
    cat_config = [
        (1, "Geobody (SEG-Y/CSV)",  '#999999', opacity_geo),
        (2, "Fault Dip",            '#FFD700', opacity_fault),
        (3, "Fault Anti-Dip",       '#4169E1', opacity_fault),
        (4, "Overlap (Fault+Geo)",  '#FF2222', opacity_overlap),
    ]
    
    legend_entries = []
    for cat_val, label, color, opacity in cat_config:
        mesh, n_cells = build_category_voxels(category, grid, cat_val, z_scale)
        if mesh is None or n_cells == 0:
            continue
        
        is_cloud = isinstance(mesh, pv.PolyData) and mesh.n_cells == 0
        if is_cloud:
            pl.add_mesh(mesh, color=color, opacity=opacity,
                        point_size=3, render_points_as_spheres=False,
                        label=f"{label} ({n_cells:,})")
        else:
            pl.add_mesh(mesh, color=color, opacity=opacity,
                        show_edges=False, label=f"{label} ({n_cells:,})")
        legend_entries.append((f"{label} ({n_cells:,})", color))
        print(f"    {label}: {n_cells:,} cells")
    
    if legend_entries:
        pl.add_legend(labels=legend_entries, bcolor='white',
                      face='rectangle', size=(0.25, 0.2))
    
    # Outline box: z=0 (surface) to deepest grid extent
    x_min = grid.x_origin - grid.dx / 2
    x_max = grid.x_origin + (grid.nx - 0.5) * grid.dx
    y_min = grid.y_origin - grid.dy / 2
    y_max = grid.y_origin + (grid.ny - 0.5) * grid.dy
    z_min_box = (grid.z_origin + (grid.nz - 0.5) * grid.dz) * z_scale
    z_max_box = 0.0 * z_scale
    
    outline = pv.Box(bounds=[x_min, x_max, y_min, y_max, z_min_box, z_max_box])
    pl.add_mesh(outline.outline(), color='#555555', line_width=1.5)
    
    pl.add_axes(line_width=2)
    pl.show_bounds(grid=True, location='outer', all_edges=True,
                   font_size=8, color='gray')
    pl.camera_position = 'iso'
    pl.camera.zoom(1.2)
    pl.export_html(str(outfile))
    pl.close()
    
    size_mb = os.path.getsize(str(outfile)) / 1024 / 1024
    print(f"  ✓ Exported 3D HTML: {outfile} ({size_mb:.1f} MB)")



# ==============================================================================
# Execute 3D Visualization
# ==============================================================================

if FAULT.enabled and category is not None:
    viz_dir = PATHS.output_dir / "3d_scenes"
    viz_dir.mkdir(parents=True, exist_ok=True)
    
    print("Exporting 3D fault + geobody visualization...")
    export_fault_geobody_html(
        category, filtered_grid,
        outfile=viz_dir / "fault_geobody_combined.html",
        z_scale=1.0,
    )
else:
    print("Skipped — run fault processing first (Cell 6c)")


## Cell 6e: Z-Slice Viewer

Plan-view slices at user-defined depths showing cell categories + fault contour traces + buffer boundaries.

**User parameters:** `Z_SLICE_PARAMS` (xmin, xmax, ymin, ymax, z, figsize, dpi).


In [ ]:
# ==============================================================================
# Z-SLICE VISUALIZATION — Plan-view category plots at user-defined depths
# ==============================================================================

def plot_z_slice(
    category,
    grid,
    all_fault_contours,
    xmin=None, xmax=None,
    ymin=None, ymax=None,
    z=None,
    figsize=(14, 12),
    dpi=150,
    outfile=None,
    show_grid_lines=True,
    grid_line_color=0.82,
):
    """
    Plot a plan-view z-slice of the classified grid.
    
    Memory optimizations vs previous version:
      - dpi reduced 150 -> 100 (saves ~55% raster memory per figure)
      - figsize reduced 16x14 -> 14x12
      - Grid lines burned into RGBA pixel array (0 matplotlib Line2D objects vs ~1000)
      - Explicit del + gc.collect() after savefig
    
    Parameters
    ----------
    category : int8 (nx, ny, nz) - 0=empty, 1=geobody, 2=dip, 3=anti, 4=overlap
    grid : RegularGrid3D
    all_fault_contours : dict {fault_name: {k_index: slice_info}}
    xmin, xmax, ymin, ymax : bounding box (None = full grid extent)
    z : z-level to plot (None = first layer with active geobody cells)
    figsize : figure size
    dpi : output resolution
    outfile : save path (None = display only)
    show_grid_lines : bool (default True - burned into pixel array, zero memory overhead)
    grid_line_color : float 0-1 (0=black, 1=white, default 0.82 = light gray)
    """
    x_axis = grid.x_axis
    y_axis = grid.y_axis
    z_axis = grid.z_axis
    
    # Default z: first layer with geobody cells
    if z is None:
        for k in range(grid.nz):
            if grid.active[:, :, k].any():
                z = z_axis[k]
                break
        if z is None:
            print("  WARNING: No active cells found in grid!")
            return
    
    # Find nearest k-index
    k_idx = int(np.argmin(np.abs(z_axis - z)))
    z_actual = z_axis[k_idx]
    
    # Default bounding box = full grid
    if xmin is None: xmin = x_axis[0] - grid.dx / 2
    if xmax is None: xmax = x_axis[-1] + grid.dx / 2
    if ymin is None: ymin = y_axis[0] - grid.dy / 2
    if ymax is None: ymax = y_axis[-1] + grid.dy / 2
    
    # Crop indices to bounding box (avoid rendering full grid)
    i_lo = max(0, int(np.floor((xmin - x_axis[0]) / grid.dx)) - 1)
    i_hi = min(grid.nx, int(np.ceil((xmax - x_axis[0]) / grid.dx)) + 2)
    j_lo = max(0, int(np.floor((ymin - y_axis[0]) / grid.dy)) - 1)
    j_hi = min(grid.ny, int(np.ceil((ymax - y_axis[0]) / grid.dy)) + 2)
    
    # Extract only the visible portion
    cat_slice = category[i_lo:i_hi, j_lo:j_hi, k_idx]
    
    # Build color image (RGBA) - cropped size only
    color_map = {
        0: [1.0, 1.0, 1.0, 0.0],       # empty = transparent
        1: [0.6, 0.6, 0.6, 0.8],        # gray = geobody
        2: [1.0, 0.84, 0.0, 0.8],       # yellow = dip
        3: [0.255, 0.412, 0.882, 0.8],   # blue = anti-dip
        4: [1.0, 0.133, 0.133, 1.0],     # red = overlap
    }
    
    rgba = np.zeros((i_hi - i_lo, j_hi - j_lo, 4), dtype=np.float32)
    for cat_val, color in color_map.items():
        mask = cat_slice == cat_val
        rgba[mask] = color
    
    # ------------------------------------------------------------------
    # Burn grid lines into RGBA pixel array BEFORE passing to imshow.
    # Upscales each cell to (cell_px x cell_px) pixels via np.repeat,
    # then overwrites border pixels with light gray.
    # Result: visible cell borders with ZERO matplotlib Line2D objects.
    # Old method: ~1000 axvline/axhline calls = ~1000 Line2D objects/plot.
    # New method: pure numpy array ops, 0 extra matplotlib objects.
    # Memory: 395x672 -> 1185x2016 x 4 bytes = ~9.5 MB (temporary, freed)
    # ------------------------------------------------------------------
    if show_grid_lines:
        cell_px = 3  # pixels per grid cell (1px border + 2px interior)
        gc_val = grid_line_color
        
        # Upscale: each cell becomes cell_px x cell_px pixels
        rgba = np.repeat(np.repeat(rgba, cell_px, axis=0), cell_px, axis=1)
        
        # Draw vertical cell borders (every cell_px columns along axis 0)
        rgba[::cell_px, :, :3] = gc_val
        rgba[::cell_px, :, 3] = np.maximum(rgba[::cell_px, :, 3], 0.4)
        
        # Draw horizontal cell borders (every cell_px rows along axis 1)
        rgba[:, ::cell_px, :3] = gc_val
        rgba[:, ::cell_px, 3] = np.maximum(rgba[:, ::cell_px, 3], 0.4)
    
    # Cropped edges for imshow extent
    x_edge_lo = grid.x_origin + i_lo * grid.dx - grid.dx / 2
    x_edge_hi = grid.x_origin + i_hi * grid.dx - grid.dx / 2
    y_edge_lo = grid.y_origin + j_lo * grid.dy - grid.dy / 2
    y_edge_hi = grid.y_origin + j_hi * grid.dy - grid.dy / 2
    
    # imshow expects (rows, cols) = (ny, nx), so transpose
    rgba_plot = np.transpose(rgba, (1, 0, 2))
    del rgba  # free immediately
    
    # Plot
    fig, ax = plt.subplots(figsize=figsize)
    
    ax.imshow(
        rgba_plot,
        origin='lower',
        extent=[x_edge_lo, x_edge_hi, y_edge_lo, y_edge_hi],
        aspect='equal',
        interpolation='nearest',
    )
    del rgba_plot  # free after imshow consumes it
    
    # Overlay fault contours
    fault_colors = plt.cm.tab10(np.linspace(0, 1, max(len(all_fault_contours), 1)))
    
    for fi, (fault_name, contours_by_k) in enumerate(all_fault_contours.items()):
        fc = fault_colors[fi % len(fault_colors)]
        
        if k_idx not in contours_by_k:
            available_k = sorted(contours_by_k.keys())
            if not available_k:
                continue
            nearest_k = min(available_k, key=lambda k: abs(k - k_idx))
            if abs(nearest_k - k_idx) > 5:
                continue
            sl = contours_by_k[nearest_k]
        else:
            sl = contours_by_k[k_idx]
        
        # Fault trace
        xy = sl['xy']
        ax.plot(xy[:, 0], xy[:, 1], '-', color=fc, linewidth=2.5, zorder=5,
                label=f"Fault: {fault_name}")
        
        # Dip polygon boundary
        dip_coords = extract_polygon_coords(sl['buffer_dip'])
        if dip_coords is not None:
            ax.plot(dip_coords[:, 0], dip_coords[:, 1], '--',
                    color=fc, linewidth=1.2, alpha=0.7, zorder=4)
        
        # Anti-dip polygon boundary
        anti_coords = extract_polygon_coords(sl['buffer_anti'])
        if anti_coords is not None:
            ax.plot(anti_coords[:, 0], anti_coords[:, 1], ':',
                    color=fc, linewidth=1.2, alpha=0.7, zorder=4)
    
    # Bounding box
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_xlabel('X (m)', fontsize=12)
    ax.set_ylabel('Y (m)', fontsize=12)
    ax.set_title(f'Z-Slice at z = {z_actual:.1f} m  (k = {k_idx})', fontsize=14)
    ax.set_aspect('equal')
    
    # Legend
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    legend_elements = [
        Patch(facecolor='#999999', edgecolor='black', alpha=0.8, label='Geobody (SEG-Y/CSV)'),
        Patch(facecolor='#FFD700', edgecolor='black', alpha=0.8, label='Fault Dip'),
        Patch(facecolor='#4169E1', edgecolor='black', alpha=0.8, label='Fault Anti-Dip'),
        Patch(facecolor='#FF2222', edgecolor='black', alpha=1.0, label='Overlap (Fault+Geo)'),
    ]
    for fi, fault_name in enumerate(all_fault_contours.keys()):
        fc = fault_colors[fi % len(fault_colors)]
        legend_elements.append(
            Line2D([0], [0], color=fc, linewidth=2.5, label=f"Fault: {fault_name}"))
    legend_elements.append(Line2D([0], [0], color='gray', linewidth=1.2,
                                  linestyle='--', label='Dip boundary'))
    legend_elements.append(Line2D([0], [0], color='gray', linewidth=1.2,
                                  linestyle=':', label='Anti-dip boundary'))
    
    ax.legend(handles=legend_elements, loc='upper right', fontsize=9,
              framealpha=0.9, edgecolor='gray')
    
    plt.tight_layout()
    
    if outfile:
        fig.savefig(str(outfile), dpi=dpi, bbox_inches='tight')
    
    # Aggressive cleanup to prevent memory accumulation in batch loops
    plt.close(fig)
    del fig, ax
    gc.collect()


# ==============================================================================
# Execute Z-Slice Plot
# ==============================================================================
# Edit these parameters and re-run this cell to explore different slices:

Z_SLICE_PARAMS = dict(
    xmin=None,     # None = full grid extent (or set e.g. 910000)
    xmax=None,     # None = full grid extent (or set e.g. 925000)
    ymin=None,     # None = full grid extent (or set e.g. 830000)
    ymax=None,     # None = full grid extent (or set e.g. 845000)
    z=-1187.4        # None = first layer with geobody data (or set e.g. -2500.0)
)

if FAULT.enabled and category is not None:
    viz_dir = PATHS.output_dir / "3d_scenes"
    viz_dir.mkdir(parents=True, exist_ok=True)
    
    print("Plotting 2D z-slice...")
    plot_z_slice(
        category=category,
        grid=filtered_grid,
        all_fault_contours=all_fault_contours,
        outfile=viz_dir / "z_slice.png",
        **Z_SLICE_PARAMS,
    )
else:
    print("Skipped - run fault processing first (Cell 6c)")


In [ ]:
# ==============================================================================
# Z-SLICE BATCH EXPORT — Reuses figure objects for speed
# ==============================================================================

def batch_z_slices(
    category, grid, all_fault_contours,
    k_list, out_dir,
    xmin=None, xmax=None, ymin=None, ymax=None,
    figsize=(14, 12), dpi=100,
    show_grid_lines=True, grid_line_color=0.82,
):
    """
    Export z-slice PNGs for all k-indices using a single reusable figure.
    
    Creates fig/axes/legend/labels ONCE. Each iteration only updates:
      - Image pixel data (im.set_data)
      - Fault contour line coordinates (line.set_data)
      - Title text
    
    ~10x fewer matplotlib object creates vs calling plot_z_slice() in a loop.
    """
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    import matplotlib
    matplotlib.use('Agg')  # non-interactive backend
    
    x_axis = grid.x_axis
    y_axis = grid.y_axis
    z_axis = grid.z_axis
    
    # Bounding box defaults
    if xmin is None: xmin = x_axis[0] - grid.dx / 2
    if xmax is None: xmax = x_axis[-1] + grid.dx / 2
    if ymin is None: ymin = y_axis[0] - grid.dy / 2
    if ymax is None: ymax = y_axis[-1] + grid.dy / 2
    
    # Crop indices (constant for all slices since bounding box is fixed)
    i_lo = max(0, int(np.floor((xmin - x_axis[0]) / grid.dx)) - 1)
    i_hi = min(grid.nx, int(np.ceil((xmax - x_axis[0]) / grid.dx)) + 2)
    j_lo = max(0, int(np.floor((ymin - y_axis[0]) / grid.dy)) - 1)
    j_hi = min(grid.ny, int(np.ceil((ymax - y_axis[0]) / grid.dy)) + 2)
    
    x_edge_lo = grid.x_origin + i_lo * grid.dx - grid.dx / 2
    x_edge_hi = grid.x_origin + i_hi * grid.dx - grid.dx / 2
    y_edge_lo = grid.y_origin + j_lo * grid.dy - grid.dy / 2
    y_edge_hi = grid.y_origin + j_hi * grid.dy - grid.dy / 2
    
    # Color map for categories
    color_map = {
        0: [1.0, 1.0, 1.0, 0.0],       # empty = transparent
        1: [0.6, 0.6, 0.6, 0.8],        # gray = geobody
        2: [1.0, 0.84, 0.0, 0.8],       # yellow = dip
        3: [0.255, 0.412, 0.882, 0.8],   # blue = anti-dip
        4: [1.0, 0.133, 0.133, 1.0],     # red = overlap
    }
    
    cell_px = 3 if show_grid_lines else 1
    
    # Helper: build RGBA for a given k-index
    def build_rgba(k_idx):
        cat_slice = category[i_lo:i_hi, j_lo:j_hi, k_idx]
        rgba = np.zeros((i_hi - i_lo, j_hi - j_lo, 4), dtype=np.float32)
        for cat_val, color in color_map.items():
            m = cat_slice == cat_val
            rgba[m] = color
        
        if show_grid_lines:
            rgba = np.repeat(np.repeat(rgba, cell_px, axis=0), cell_px, axis=1)
            gc_val = grid_line_color
            rgba[::cell_px, :, :3] = gc_val
            rgba[::cell_px, :, 3] = np.maximum(rgba[::cell_px, :, 3], 0.4)
            rgba[:, ::cell_px, :3] = gc_val
            rgba[:, ::cell_px, 3] = np.maximum(rgba[:, ::cell_px, 3], 0.4)
        
        return np.transpose(rgba, (1, 0, 2))  # (ny_px, nx_px, 4) for imshow
    
    # ==================================================================
    # CREATE FIGURE ONCE - frame, labels, legend, fault line placeholders
    # ==================================================================
    fig, ax = plt.subplots(figsize=figsize)
    
    # Initial image (first slice) - establishes the AxesImage object
    first_k = k_list[0]
    initial_rgba = build_rgba(first_k)
    im = ax.imshow(
        initial_rgba,
        origin='lower',
        extent=[x_edge_lo, x_edge_hi, y_edge_lo, y_edge_hi],
        aspect='equal',
        interpolation='nearest',
    )
    del initial_rgba
    
    # Fixed labels
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_xlabel('X (m)', fontsize=12)
    ax.set_ylabel('Y (m)', fontsize=12)
    ax.set_aspect('equal')
    title_obj = ax.set_title('', fontsize=14)  # updated per slice
    
    # Pre-create reusable Line2D objects for each fault (trace + dip + anti)
    fault_names = list(all_fault_contours.keys())
    n_faults = len(fault_names)
    fault_colors = plt.cm.tab10(np.linspace(0, 1, max(n_faults, 1)))
    
    # 3 lines per fault: trace (solid), dip boundary (dashed), anti boundary (dotted)
    fault_lines = {}
    for fi, fname in enumerate(fault_names):
        fc = fault_colors[fi % len(fault_colors)]
        line_trace, = ax.plot([], [], '-', color=fc, linewidth=2.5, zorder=5)
        line_dip, = ax.plot([], [], '--', color=fc, linewidth=1.2, alpha=0.7, zorder=4)
        line_anti, = ax.plot([], [], ':', color=fc, linewidth=1.2, alpha=0.7, zorder=4)
        fault_lines[fname] = (line_trace, line_dip, line_anti)
    
    # Fixed legend (created once)
    legend_elements = [
        Patch(facecolor='#999999', edgecolor='black', alpha=0.8, label='Geobody (SEG-Y/CSV)'),
        Patch(facecolor='#FFD700', edgecolor='black', alpha=0.8, label='Fault Dip'),
        Patch(facecolor='#4169E1', edgecolor='black', alpha=0.8, label='Fault Anti-Dip'),
        Patch(facecolor='#FF2222', edgecolor='black', alpha=1.0, label='Overlap (Fault+Geo)'),
    ]
    for fi, fname in enumerate(fault_names):
        fc = fault_colors[fi % len(fault_colors)]
        legend_elements.append(Line2D([0], [0], color=fc, linewidth=2.5, label=f"Fault: {fname}"))
    legend_elements.append(Line2D([0], [0], color='gray', linewidth=1.2,
                                  linestyle='--', label='Dip boundary'))
    legend_elements.append(Line2D([0], [0], color='gray', linewidth=1.2,
                                  linestyle=':', label='Anti-dip boundary'))
    ax.legend(handles=legend_elements, loc='upper right', fontsize=9,
              framealpha=0.9, edgecolor='gray')
    
    plt.tight_layout()
    
    # ==================================================================
    # LOOP: update image + fault lines + title, then savefig
    # ==================================================================
    for k_idx in tqdm(k_list, desc="Z-slices"):
        z_actual = z_axis[k_idx]
        
        # 1. Update image data
        new_rgba = build_rgba(k_idx)
        im.set_data(new_rgba)
        del new_rgba
        
        # 2. Update title
        title_obj.set_text(f'Z-Slice at z = {z_actual:.1f} m  (k = {k_idx})')
        
        # 3. Update fault contour lines
        for fname in fault_names:
            line_trace, line_dip, line_anti = fault_lines[fname]
            contours_by_k = all_fault_contours[fname]
            
            # Find contour data for this k
            if k_idx in contours_by_k:
                sl = contours_by_k[k_idx]
            else:
                # Try nearest within 5 layers
                available_k = sorted(contours_by_k.keys())
                nearest_k = min(available_k, key=lambda kk: abs(kk - k_idx)) if available_k else None
                if nearest_k is not None and abs(nearest_k - k_idx) <= 5:
                    sl = contours_by_k[nearest_k]
                else:
                    sl = None
            
            if sl is not None:
                xy = sl['xy']
                line_trace.set_data(xy[:, 0], xy[:, 1])
                
                dip_coords = extract_polygon_coords(sl['buffer_dip'])
                if dip_coords is not None:
                    line_dip.set_data(dip_coords[:, 0], dip_coords[:, 1])
                else:
                    line_dip.set_data([], [])
                
                anti_coords = extract_polygon_coords(sl['buffer_anti'])
                if anti_coords is not None:
                    line_anti.set_data(anti_coords[:, 0], anti_coords[:, 1])
                else:
                    line_anti.set_data([], [])
            else:
                # No contour at this depth - hide lines
                line_trace.set_data([], [])
                line_dip.set_data([], [])
                line_anti.set_data([], [])
        
        # 4. Save
        outfile = out_dir / f"z_slice_k{k_idx:04d}_z{z_actual:.1f}.png"
        fig.savefig(str(outfile), dpi=dpi, bbox_inches='tight')
    
    # Cleanup the single figure
    plt.close(fig)
    del fig, ax, im, fault_lines
    gc.collect()


# ==============================================================================
# Execute Batch Export
# ==============================================================================

Z_SLICE_PARAMS = dict(
    xmin=909000,
    xmax=919000,
    ymin=827000,
    ymax=844000,
)

if FAULT.enabled and category is not None:
    viz_dir = PATHS.output_dir / "3d_scenes" / "z_slicesZ"
    viz_dir.mkdir(parents=True, exist_ok=True)
    
    # Collect all k-indices that have fault contour data
    all_k = set()
    for fault_name, contours_by_k in all_fault_contours.items():
        all_k.update(contours_by_k.keys())
    all_k = sorted(all_k)
    
    print(f"Exporting {len(all_k)} z-slices (reusable figure)...")
    
    batch_z_slices(
        category=category,
        grid=filtered_grid,
        all_fault_contours=all_fault_contours,
        k_list=all_k,
        out_dir=viz_dir,
        dpi=100,
        figsize=(14, 12),
        show_grid_lines=True,
        **Z_SLICE_PARAMS,
    )
    
    print(f"Saved {len(all_k)} slices to {viz_dir}")
else:
    print("Skipped - run fault processing first (Cell 6c)")


## Cell 7: Platform Candidate Generation

Generates a 2D grid of candidate platform locations over the geobody footprint, applies polygon exclusions from shapefiles.

**User parameters:** `GridConfig.step`, `GridConfig.spacing`, `GridConfig.edge_buffer`, `ExclusionConfig`.


In [ ]:
# ==============================================================================
# PLATFORM CANDIDATES — Generate 2D surface grid of candidate locations
# ==============================================================================

def generate_platform_candidates(
    grid: RegularGrid3D,
    grid_cfg: GridConfig,
    shapefile_dir: Path
) -> Tuple[PlatformCandidates, Optional[Any], Optional[Any]]:
    
    # Use active cell extents for bounds
    ax, ay, _ = grid.active_xyz()
    
    lx = (np.floor(ax.min() / grid_cfg.step) - 1) * grid_cfg.step
    ux = (np.ceil(ax.max() / grid_cfg.step) + 1) * grid_cfg.step
    ly = (np.floor(ay.min() / grid_cfg.step) - 1) * grid_cfg.step
    uy = (np.ceil(ay.max() / grid_cfg.step) + 1) * grid_cfg.step
    
    x_grid = np.arange(lx, ux, grid_cfg.spacing)
    y_grid = np.arange(ly, uy, grid_cfg.spacing)
    
    X0, Y0 = np.meshgrid(x_grid, y_grid, indexing="ij")
    cx_all, cy_all = X0.ravel(), Y0.ravel()
    n_total = len(cx_all)
    print(f"  Grid: {len(x_grid)} x {len(y_grid)} = {n_total:,} points")
    
    # Load anomaly polygons
    shp_files = list(shapefile_dir.glob("**/*.shp"))
    poly_gdfs = []
    for shp in shp_files:
        try:
            gdf = gpd.read_file(shp)
            if not gdf.empty:
                poly_gdfs.append(gdf)
                print(f"  Loaded {len(gdf)} polygon(s) from {shp.name}")
        except Exception as e:
            print(f"  Skipping {shp.name}: {e}")
    
    if poly_gdfs:
        poly_all = gpd.GeoDataFrame(pd.concat(poly_gdfs, ignore_index=True), crs=poly_gdfs[0].crs)
        excl_geom = unary_union(poly_all.geometry).buffer(grid_cfg.anomaly_buffer)
    else:
        poly_all = None
        excl_geom = None
        print("  No anomaly polygons found")
    
    centers_gdf = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(cx_all, cy_all),
        crs=poly_gdfs[0].crs if poly_gdfs else None
    )
    
    excl_poly = centers_gdf.within(excl_geom).to_numpy() if excl_geom is not None else np.zeros(n_total, dtype=bool)
    
    # Use grid active footprint for data extent
    footprint = MultiPoint(np.c_[ax, ay]).convex_hull
    footprint_buffer = footprint.buffer(grid_cfg.edge_buffer)
    inside_data = centers_gdf.within(footprint_buffer).to_numpy()
    
    keep = ~excl_poly & inside_data
    print(f"  Excluded by anomaly buffer: {excl_poly.sum():,}")
    print(f"  Excluded by data edge: {(~inside_data).sum():,}")
    
    candidates = PlatformCandidates(
        x=cx_all[keep], y=cy_all[keep],
        x_grid=x_grid, y_grid=y_grid,
        flat_idx=np.flatnonzero(keep)
    )
    return candidates, poly_all, excl_geom


print("Generating platform candidates...")
platform_candidates, poly_all, excl_geom = generate_platform_candidates(
    filtered_grid, GRID, PATHS.shapefile_dir
)
print(f"\n✓ Valid candidates: {len(platform_candidates):,}")


## Cell 8: Reachability Scoring

For each platform candidate, counts active cells inside a cone extending from surface to depth. The cone radius at depth z is `min(|z−z₀|×tan(θ), hd_max)`.

**User parameters:** `ConeConfig.theta_deg`, `ConeConfig.z_offset`, `ConeConfig.hd`.


In [ ]:
# ==============================================================================
# REACHABILITY SCORING — Count active cells inside cone per candidate
# ==============================================================================

def compute_platform_reachability(
    grid: RegularGrid3D,
    candidates: PlatformCandidates,
    cone: ConeConfig,
    volume: VolumeConfig,
    show_progress: bool = True
) -> Tuple[PlatformResult, float]:
    """Compute reachability score for each platform candidate using grid operations."""
    
    t_start = perf_counter()
    
    n_centers = len(candidates.x)
    Nx, Ny = len(candidates.x_grid), len(candidates.y_grid)
    
    # Precompute per-layer cone radii squared (shared for all candidates)
    z_ax = grid.z_axis
    Rz2 = np.zeros(grid.nz)
    valid_k = np.zeros(grid.nz, dtype=bool)
    for k in range(grid.nz):
        if z_ax[k] < cone.z0:
            Rz = min((cone.z0 - z_ax[k]) * cone.tan_theta, cone.hd)
            Rz2[k] = Rz * Rz
            valid_k[k] = True
    
    weights = grid.values
    
    # Storage
    heatmap = np.full(Nx * Ny, -1.0)
    
    x_ax = grid.x_axis
    y_ax = grid.y_axis
    
    for ci in tqdm(range(n_centers), disable=not show_progress, desc="Computing reachability"):
        x0, y0 = candidates.x[ci], candidates.y[ci]
        flat_idx = candidates.flat_idx[ci]
        
        # Horizontal distance squared for all (i, j) from this platform
        DX2 = (x_ax - x0) ** 2  # (nx,)
        DY2 = (y_ax - y0) ** 2  # (ny,)
        D2_ij = DX2[:, None] + DY2[None, :]  # (nx, ny)
        
        score = 0.0
        for k in range(grid.nz):
            if not valid_k[k]:
                continue
            # Cells within cone at this depth layer
            in_cone_ij = D2_ij <= Rz2[k]
            # Intersect with active mask at this layer
            hit = in_cone_ij & grid.active[:, :, k]
            if np.any(hit):
                score += weights[:, :, k][hit].sum()
        
        heatmap[flat_idx] = score
    
    # Find best
    valid_mask = heatmap >= 0
    valid_scores = np.where(valid_mask, heatmap, -np.inf)
    best_flat = np.argmax(valid_scores)
    
    best_cand_idx = np.searchsorted(candidates.flat_idx, best_flat)
    if best_cand_idx >= n_centers or candidates.flat_idx[best_cand_idx] != best_flat:
        cand_scores = heatmap[candidates.flat_idx]
        best_cand_idx = np.argmax(cand_scores)
    
    t_elapsed = perf_counter() - t_start
    
    result = PlatformResult(
        candidates=candidates,
        heatmap=heatmap.reshape(Nx, Ny),
        best_idx=best_cand_idx,
        best_x=candidates.x[best_cand_idx],
        best_y=candidates.y[best_cand_idx],
        best_score=heatmap[candidates.flat_idx[best_cand_idx]]
    )
    return result, t_elapsed


print("Computing platform reachability...")
platform_result, reachability_time = compute_platform_reachability(
    filtered_grid, platform_candidates, CONE, VOLUME
)

print(f"\n✓ Reachability computed in {reachability_time:.2f}s")
print(f"  Best platform: ({platform_result.best_x:.1f}, {platform_result.best_y:.1f})")
print(f"  Best score: {platform_result.best_score:.0f}")


## Cell 9: Platform Heatmap

Plots the reachability score for each candidate as a 2D heatmap. Identifies the best platform location.


In [ ]:
# ==============================================================================
# PLATFORM HEATMAP — Reachability score visualization
# ==============================================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 10), layout='constrained')

# Extract active cell footprint for plotting
ax_pts, ay_pts, _ = filtered_grid.active_xyz()

ax1.scatter(ax_pts[::10], ay_pts[::10], s=1, c='#FF7070', alpha=0.5, label='Voxels (1:10)')

if poly_all is not None:
    poly_all.plot(ax=ax1, color='k', linewidth=1.5, zorder=4, alpha=0.4, label='Anomalies')
if excl_geom is not None:
    gpd.GeoSeries([excl_geom]).plot(ax=ax1, color='k', linewidth=1.5, zorder=4, alpha=0.2)

ax1.scatter(platform_candidates.x, platform_candidates.y, s=2, c='#003569', alpha=0.8, 
            marker='o', label='Candidates')
ax1.set_xlim(platform_candidates.x_grid.min(), platform_candidates.x_grid.max())
ax1.set_ylim(platform_candidates.y_grid.min(), platform_candidates.y_grid.max())
ax1.set_aspect('equal')
ax1.set_xlabel('X (m)')
ax1.set_ylabel('Y (m)')
ax1.legend(loc='upper right')
ax1.set_title('Platform Candidates and Data Points')

# Heatmap
masked = np.ma.masked_where(platform_result.heatmap < 0, platform_result.heatmap)
cmap = plt.colormaps['viridis'].copy()
cmap.set_bad('black')

extent = (platform_candidates.x_grid.min(), platform_candidates.x_grid.max(),
          platform_candidates.y_grid.min(), platform_candidates.y_grid.max())

display_data = masked

im = ax2.imshow(display_data.T, origin='lower', extent=extent, aspect='equal', cmap=cmap)
ax2.scatter([platform_result.best_x], [platform_result.best_y], 
            s=200, c='crimson', marker='*', edgecolor='white', linewidth=1.5, zorder=10)
ax2.set_xlim(platform_candidates.x_grid.min(), platform_candidates.x_grid.max())
ax2.set_ylim(platform_candidates.y_grid.min(), platform_candidates.y_grid.max())
ax2.set_xlabel('X (m)')
ax2.set_ylabel('Y (m)')
ax2.set_title('Reachability Heatmap')

cbar = fig.colorbar(im, ax=[ax1, ax2], location='bottom', pad=0.02, shrink=0.5)
cbar.set_label('Reachable Coverage')
fig.savefig(PathConfig().output_dir / "platform_heatmap.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"\nBest Platform: ({platform_result.best_x:.2f}, {platform_result.best_y:.2f})")
print(f"  Score: {platform_result.best_score:.2f}")


## Cell 10: Select Platform & Cone Sub-Grid

Selects the platform (either the best from scoring or a user-specified location) and extracts the cone sub-grid: only active cells inside the reachable cone are kept.

**User parameters:** `PlatformSelection.x`, `PlatformSelection.y`, `PlatformSelection.use_best`.


In [ ]:
# ==============================================================================
# SELECT PLATFORM — Extract cone sub-grid for optimization
# ==============================================================================

if PLATFORM.use_best:
    xb, yb = platform_result.best_x, platform_result.best_y
    print(f"Using best platform from analysis: ({xb:.1f}, {yb:.1f})")
else:
    xb, yb = PLATFORM.x, PLATFORM.y
    print(f"Using configured platform: ({xb:.1f}, {yb:.1f})")

# Create cone grid: copy filtered grid, then restrict to cone
cone_grid = filtered_grid.copy()
cone_mask = cone_mask_on_grid(cone_grid, xb, yb, CONE.z0, CONE.tan_theta, CONE.hd)
cone_grid.active &= cone_mask  # Only keep cells that are both active AND in cone

print(f"\n✓ Voxels in platform cone: {cone_grid.n_active:,}")
print(cone_grid.summary())


**!!!!Skip this cell if `output_dir/well_search.h5` already presented!!!!**

## Cell 11: Well Path Grid Search (Parallel, Disk-Backed)

Brute-force search over all 9-parameter well trajectory combinations. Each combination: compute trajectory → check feasibility (T1>0, MD, HD) → compute open-hole coverage → store to HDF5.

**Parallelized** via chunk-based multiprocessing (N workers, each processing CHUNK_SIZE combos).

**User parameters:**
- **`N_WORKERS`** — Number of parallel workers (default 4, ~1.5 GB RAM each)
- **`CHUNK_SIZE`** — Combos per chunk (controls progress granularity)
- **`WellConfig`** — All trajectory search ranges: `v_range`, `alpha_range`, `p1_range`, `h_range`, `p2_range`, `beta_range` (+ `beta_mode`), `t_range`, `DLS1_range`, `DLS2_range`, `md_max`, `hd_max`, `coverage_radius`

**Output:** `output_dir/well_search.h5`


In [ ]:
# ==============================================================================
# WELL PATH GRID SEARCH — Parallel, chunk-based, HDF5 output
# ==============================================================================
#
# User parameters:
#   N_WORKERS  — Number of parallel processes (adjust to CPU cores)
#   CHUNK_SIZE — Combos per work unit (controls progress bar granularity)
#   Well trajectory ranges are defined in WellConfig (Cell 1)
#
import h5py
import multiprocessing as mp
import inspect
import time as _time

N_WORKERS = 4          # Adjust to your CPU cores (~1.5 GB RAM each)
CHUNK_SIZE = 200      # Combos per chunk

x0, y0 = (
    (best_platform_x, best_platform_y) if PLATFORM.use_best 
    else (PLATFORM.x, PLATFORM.y)
)
print(f"Running parallel well path grid search from platform ({x0:.1f}, {y0:.1f})...")
print(f"  Workers: {N_WORKERS}, Chunk size: {CHUNK_SIZE}")

# z-samples: depths with active cells + surface
z_all = cone_grid.z_axis
z_active_mask = np.any(cone_grid.active, axis=(0, 1))
z_samples = np.sort(np.unique(np.r_[0.0, z_all[z_active_mask]]))

# flat_to_col: maps flat grid index → sparse matrix column (-1 if inactive)
flat_to_col = np.full(cone_grid.nx * cone_grid.ny * cone_grid.nz, -1, dtype=np.int32)
active_flat = cone_grid.active_flat_indices
for col, flat_idx in enumerate(active_flat):
    flat_to_col[flat_idx] = col

well = WELL

# ── Build parameter combinations ─────────────────────────────────────────────
print("  Building parameter combinations...")

all_combos = []
for v in well.v_range:
    for h in well.h_range:
        z_casing = -(v + h)
        has_voxels = bool(np.any(z_samples <= z_casing))
        
        for alpha in well.alpha_range:
            if well.beta_mode == 'relative':
                beta_values = (alpha + well.beta_range) % 360.0
            else:
                beta_values = well.beta_range
            
            for p1 in well.p1_range:
                valid_p2 = well.p2_range[well.p2_range <= p1]
                for p2 in valid_p2:
                    for beta in beta_values:
                        for t in well.t_range:
                            for DLS1 in well.DLS1_range:
                                for DLS2 in well.DLS2_range:
                                    all_combos.append((v, h, alpha, p1, p2, beta, t, DLS1, DLS2, has_voxels))

n_combos = len(all_combos)

# Split into chunks
chunks = [all_combos[i:i+CHUNK_SIZE] for i in range(0, n_combos, CHUNK_SIZE)]
n_chunks = len(chunks)
print(f"  Total combinations: {n_combos:,}")
print(f"  Chunks: {n_chunks} × {CHUNK_SIZE}")
del all_combos
gc.collect()

# Paths
h5_path = PATHS.output_dir / "well_search.h5"
tmp_dir = PATHS.output_dir / "_well_search_tmp"
tmp_dir.mkdir(parents=True, exist_ok=True)

# ── Worker function (runs in child process) ─────────────────────────────────

def _process_chunk(args):
    """Process a chunk of combos and write results to a temp HDF5 file."""
    (chunk_id, combo_list, x0, y0, z_samples,
     grid_dict, flat_to_col, well_dict, tmp_dir_str) = args
    
    import numpy as np
    import h5py
    import sys
    from pathlib import Path
    
    tmp_dir_p = Path(tmp_dir_str)
    if str(tmp_dir_p) not in sys.path:
        sys.path.insert(0, str(tmp_dir_p))
    from _well_search_funcs import three_segment_path_xy, coverage_on_grid
    
    class _G:
        pass
    grid = _G()
    for k_attr, v_attr in grid_dict.items():
        setattr(grid, k_attr, v_attr)
    
    z_masks = {}
    params_out = []
    scores_out = []
    indices_out = []
    indptr_out = [0]
    
    stats = {'n_geometry': 0, 'n_md': 0, 'n_hd': 0, 'n_zero': 0, 'n_no_voxels': 0}
    
    for combo in combo_list:
        v, h, alpha, p1, p2, beta, t, DLS1, DLS2, has_voxels = combo
        
        if not has_voxels:
            stats['n_no_voxels'] += 1
            continue
        
        z_casing = -(v + h)
        if (v, h) not in z_masks:
            z_masks[(v, h)] = z_samples <= z_casing
        z_mask = z_masks[(v, h)]
        z_sub = z_samples[z_mask]
        
        try:
            x_full, y_full, MD_total, HD = three_segment_path_xy(
                x0=x0, y0=y0, z=z_samples,
                alpha_deg=alpha, v=v, p1_deg=p1, h=h,
                p2_deg=p2, beta_deg=beta, t=t,
                DLS1_deg_per_m=DLS1, DLS2_deg_per_m=DLS2
            )
        except ValueError:
            stats['n_geometry'] += 1
            continue
        
        if MD_total > well_dict['md_max']:
            stats['n_md'] += 1
            continue
        if HD > well_dict['hd_max']:
            stats['n_hd'] += 1
            continue
        
        covered_flat = coverage_on_grid(
            x_full[z_mask], y_full[z_mask], z_sub,
            grid, well_dict['coverage_radius'])
        if len(covered_flat) == 0:
            stats['n_zero'] += 1
            continue
        
        cols = flat_to_col[covered_flat]
        cols = cols[cols >= 0]
        if len(cols) == 0:
            stats['n_zero'] += 1
            continue
        
        params_out.append([v, alpha, p1, h, p2, beta, t, DLS1, DLS2])
        scores_out.append(len(cols))
        indices_out.extend(cols.tolist())
        indptr_out.append(indptr_out[-1] + len(cols))
    
    # Write to temp HDF5
    out_path = Path(tmp_dir_str) / f"chunk_{chunk_id:06d}.h5"
    n_accepted = len(params_out)
    
    with h5py.File(str(out_path), 'w') as hf:
        if n_accepted > 0:
            hf.create_dataset('params',  data=np.array(params_out, dtype=np.float32))
            hf.create_dataset('scores',  data=np.array(scores_out, dtype=np.int32))
            hf.create_dataset('indptr',  data=np.array(indptr_out, dtype=np.int64))
            hf.create_dataset('indices', data=np.array(indices_out, dtype=np.int32))
        hf.attrs['n_accepted'] = n_accepted
        hf.attrs['n_evaluated'] = len(combo_list)
        hf.attrs['chunk_id'] = chunk_id
        for k, v in stats.items():
            hf.attrs[k] = v
    
    return chunk_id


# ── Export functions for Windows multiprocessing ─────────────────────────────
_worker_module_path = tmp_dir / "_well_search_funcs.py"
with open(str(_worker_module_path), 'w') as _mf:
    _mf.write("from __future__ import annotations\nimport numpy as np\nfrom typing import Tuple\n\n")
    _mf.write(inspect.getsource(three_segment_path_xy))
    _mf.write("\n\n")
    _mf.write(inspect.getsource(coverage_on_grid))
    _mf.write("\n\n")
    _mf.write("import h5py\n\n")
    _mf.write(inspect.getsource(_process_chunk))
    _mf.write("\n\n")

import sys
if str(tmp_dir) not in sys.path:
    sys.path.insert(0, str(tmp_dir))

# Pack grid and well config as plain dicts (must be picklable for multiprocessing)
grid_dict = {
    'x_origin': cone_grid.x_origin, 'y_origin': cone_grid.y_origin, 'z_origin': cone_grid.z_origin,
    'dx': cone_grid.dx, 'dy': cone_grid.dy, 'dz': cone_grid.dz,
    'nx': cone_grid.nx, 'ny': cone_grid.ny, 'nz': cone_grid.nz,
    'active': cone_grid.active,
}
well_dict = {
    'md_max': well.md_max, 'hd_max': well.hd_max,
    'coverage_radius': well.coverage_radius,
}

chunk_args = [
    (ci, chunk, x0, y0, z_samples, grid_dict, flat_to_col, well_dict, str(tmp_dir))
    for ci, chunk in enumerate(chunks)
]
del chunks
gc.collect()

# ── Launch parallel workers ─────────────────────────────────────────────────
# Import worker function from temp module so Windows spawn can find it
from _well_search_funcs import _process_chunk as _process_chunk_importable

print(f"\nLaunching {N_WORKERS} workers for {n_chunks} chunks...\n")
t_start = perf_counter()

# Use apply_async + polling for reliable progress on Windows
pool = mp.Pool(processes=N_WORKERS)
async_results = [pool.apply_async(_process_chunk_importable, (ca,)) for ca in chunk_args]
pool.close()  # no more tasks

pbar = tqdm(total=n_chunks, desc="Grid search", unit="chunk")
done_set = set()

while len(done_set) < n_chunks:
    for i, ar in enumerate(async_results):
        if i not in done_set and ar.ready():
            done_set.add(i)
            pbar.update(1)
    _time.sleep(0.3)

pbar.close()
pool.join()

# Collect results (just chunk_ids, used for error checking)
for ar in async_results:
    ar.get()  # raises if worker had an exception

t_parallel = perf_counter() - t_start
print(f"  All workers finished in {t_parallel:.1f}s")

# ── Merge chunk files into final HDF5 ───────────────────────────────────────
print(f"\nMerging {n_chunks} chunk files...")

chunk_files = sorted(tmp_dir.glob("chunk_*.h5"), key=lambda p: p.name)

# First pass: gather stats and sizes
total_stats = {'n_geometry': 0, 'n_md': 0, 'n_hd': 0, 'n_zero': 0, 'n_no_voxels': 0}
total_accepted = 0
total_indices_len = 0

for cf in chunk_files:
    with h5py.File(str(cf), 'r') as hf:
        n_a = int(hf.attrs['n_accepted'])
        total_accepted += n_a
        if n_a > 0:
            total_indices_len += hf['indices'].shape[0]
        for key in total_stats:
            total_stats[key] += int(hf.attrs.get(key, 0))

k_well = total_accepted
print(f"  Total accepted: {k_well:,}, total indices: {total_indices_len:,}")

# Second pass: write merged HDF5
with h5py.File(str(h5_path), 'w') as hf_out:
    ds_params  = hf_out.create_dataset('params',  shape=(total_accepted, 9), dtype='f4')
    ds_scores  = hf_out.create_dataset('scores',  shape=(total_accepted,),   dtype='i4')
    ds_indptr  = hf_out.create_dataset('indptr',  shape=(total_accepted + 1,), dtype='i8')
    ds_indices = hf_out.create_dataset('indices', shape=(total_indices_len,),  dtype='i4')
    
    hf_out.create_dataset('cone_cell_type_active', data=category[cone_grid.active].astype(np.int8))
    
    hf_out.attrs['platform_x'] = x0
    hf_out.attrs['platform_y'] = y0
    hf_out.attrs['n_active'] = len(active_flat)
    hf_out.attrs['n_wells'] = k_well
    hf_out.attrs['n_workers'] = N_WORKERS
    hf_out.attrs['parallel_time_s'] = t_parallel
    
    param_offset = 0
    idx_offset = 0
    ds_indptr[0] = 0
    
    for cf in tqdm(chunk_files, desc="Merging"):
        with h5py.File(str(cf), 'r') as hf_in:
            n_w = int(hf_in.attrs['n_accepted'])
            if n_w == 0:
                continue
            
            ds_params[param_offset:param_offset + n_w] = hf_in['params'][:]
            ds_scores[param_offset:param_offset + n_w] = hf_in['scores'][:]
            
            n_idx = hf_in['indices'].shape[0]
            if n_idx > 0:
                ds_indices[idx_offset:idx_offset + n_idx] = hf_in['indices'][:]
            
            chunk_indptr = hf_in['indptr'][:]
            ds_indptr[param_offset + 1:param_offset + 1 + n_w] = chunk_indptr[1:] + idx_offset
            
            param_offset += n_w
            idx_offset += n_idx

# Cleanup
if str(tmp_dir) in sys.path:
    sys.path.remove(str(tmp_dir))
import shutil
shutil.rmtree(str(tmp_dir), ignore_errors=True)

del flat_to_col, chunk_args
gc.collect()

# ── Summary ─────────────────────────────────────────────────────────────────
n_accepted = total_stats.get('n_accepted', k_well)
n_rejected = n_combos - k_well

print(f"\n  Rejection Summary:")
print(f"  {'Criterion':<40s} {'Rejected':>10s} {'%':>7s}")
print(f"  {'-'*60}")
for name, key in [
    ("No voxels below casing shoe", 'n_no_voxels'),
    ("Geometry infeasible (T1 <= 0)", 'n_geometry'),
    ("MD > md_max", 'n_md'),
    ("HD > hd_max", 'n_hd'),
    ("Zero voxel coverage", 'n_zero'),
]:
    count = total_stats[key]
    print(f"  {name:<40s} {count:>10,d} {100*count/max(n_combos,1):>6.1f}%")
print(f"  {'-'*60}")
print(f"  {'Total rejected':<40s} {n_rejected:>10,d} {100*n_rejected/max(n_combos,1):>6.1f}%")
print(f"  {'Accepted':<40s} {k_well:>10,d} {100*k_well/max(n_combos,1):>6.1f}%")

print(f"\n✓ Grid search completed in {t_parallel:.1f}s ({N_WORKERS} workers, {n_chunks} chunks)")
print(f"  Valid well paths: {k_well:,}")
print(f"  HDF5 file size: {h5_path.stat().st_size / 1e6:.1f} MB")


## Cell 11b: Score Analysis & Filtering

Loads well scores from `well_search.h5`, visualizes the coverage score distribution (histogram, CDF, ranked scores), then filters by percentile threshold and builds the in-memory sparse coverage matrix.

**⚠️ User parameter:** **`SCORE_PERCENTILE`** — Wells scoring below this percentile are excluded. Set to `0` to keep all wells. Increase if memory is insufficient.

**Can be re-run** without re-running the grid search — just change the threshold and execute.


In [ ]:
# ==============================================================================
# SCORE ANALYSIS & FILTERING — Load from HDF5, filter, build sparse matrix
# ==============================================================================
#
# User parameter:
#   SCORE_PERCENTILE — Wells below this percentile are excluded (0 = keep all)
#
import h5py

h5_path = PATHS.output_dir / "well_search.h5"

with h5py.File(str(h5_path), 'r') as hf:
    scores = hf['scores'][:]
    n_total = len(scores)
    platform_x = float(hf.attrs['platform_x'])
    platform_y = float(hf.attrs['platform_y'])
    n_active = int(hf.attrs['n_active'])
    print(f"Loaded {n_total:,} well scores from {h5_path.name}")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

ax = axes[0, 0]
ax.hist(scores, bins=100, color='#F3B421', edgecolor='black', linewidth=0.3, alpha=0.85)
ax.set_xlabel('Coverage Score (active cells covered)')
ax.set_ylabel('Number of Wells')
ax.set_title(f'(a) Full Distribution — {n_total:,} wells')
ax.axvline(np.median(scores), color='red', linestyle='--', linewidth=1.5, label=f'Median: {np.median(scores):.0f}')
ax.axvline(np.mean(scores), color='blue', linestyle='--', linewidth=1.5, label=f'Mean: {np.mean(scores):.0f}')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

ax = axes[0, 1]
ax.hist(scores, bins=100, color='#1E4DE9', edgecolor='black', linewidth=0.3, alpha=0.7)
ax.set_xlabel('Coverage Score')
ax.set_ylabel('Number of Wells (log)')
ax.set_yscale('log')
ax.set_title('(b) Log Scale')
ax.grid(True, alpha=0.2)

ax = axes[1, 0]
sorted_scores = np.sort(scores)
cdf = np.arange(1, n_total + 1) / n_total * 100
ax.plot(sorted_scores, cdf, '-', color='#0B7A75', linewidth=2)
ax.set_xlabel('Coverage Score')
ax.set_ylabel('Cumulative % of Wells')
ax.set_title('(c) CDF')
ax.grid(True, alpha=0.3)
for pct, color, ls in [(10, '#999999', ':'), (25, '#666666', '--'), (50, 'red', '--')]:
    thresh = np.percentile(scores, pct)
    ax.axvline(thresh, color=color, linestyle=ls, linewidth=1)
    ax.text(thresh, pct + 2, f'P{pct}: {thresh:.0f}', fontsize=8, color=color)

ax = axes[1, 1]
ax.plot(np.arange(1, n_total + 1), sorted_scores[::-1], '-', color='#E63946', linewidth=1.5)
ax.set_xlabel('Well Rank (best to worst)')
ax.set_ylabel('Coverage Score')
ax.set_title('(d) Ranked Scores')
ax.grid(True, alpha=0.2)
for n_keep, alpha_v, label in [(10000, 0.3, '10K'), (20000, 0.15, '20K'), (50000, 0.08, '50K')]:
    if n_keep <= n_total:
        ax.axvspan(0, n_keep, alpha=alpha_v, color='#1E4DE9', label=f'Top {label}')
ax.legend(fontsize=9, loc='upper right')

fig.suptitle('Well Coverage Score Distribution', fontsize=15, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(PATHS.output_dir / "well_score_distribution.png", dpi=300)

print(f"\nScore Summary ({n_total:,} wells):")
for lbl, val in [("Min", scores.min()), ("P10", np.percentile(scores,10)),
                  ("P25", np.percentile(scores,25)), ("Median", np.median(scores)),
                  ("Mean", np.mean(scores)), ("P75", np.percentile(scores,75)),
                  ("P90", np.percentile(scores,90)), ("Max", scores.max())]:
    print(f"  {lbl:<8s}: {val:>10,.0f}")

print(f"\nSurviving wells at various thresholds:")
for pct in [0, 5, 10, 15, 20, 25, 30, 50]:
    thresh = np.percentile(scores, pct)
    n_survive = int(np.sum(scores >= thresh))
    print(f"  P{pct:<3d}: {n_survive:>8,d} wells ({100*n_survive/n_total:>5.1f}%), score >= {thresh:,.0f}")

# ── Filter & Build Sparse Matrix ────────────────────────────────────────────
SCORE_PERCENTILE = 95   # >>> ADJUST THIS <<<

score_threshold = np.percentile(scores, SCORE_PERCENTILE)
keep_mask = scores >= score_threshold
keep_indices = np.where(keep_mask)[0]
n_keep = len(keep_indices)

print(f"\n{'='*60}")
print(f"FILTERING: P{SCORE_PERCENTILE} threshold = {score_threshold:.0f}")
print(f"  Keeping {n_keep:,} of {n_total:,} wells ({100*n_keep/n_total:.1f}%)")
print(f"  Loading filtered wells from HDF5...")

with h5py.File(str(h5_path), 'r') as hf:
    all_params = hf['params'][:]
    all_indptr = hf['indptr'][:]

# Build CSR matrix directly from HDF5 — no Python list accumulation
# Uses pre-allocated numpy arrays instead of growing Python lists (28 bytes/int → 4 bytes/int)
# Step 1: Compute new indptr — count non-zeros per filtered well
print("  Building CSR indptr...")
new_indptr = np.zeros(n_keep + 1, dtype=np.int64)
for new_idx, old_idx in enumerate(keep_indices):
    nnz_this = int(all_indptr[old_idx + 1] - all_indptr[old_idx])
    new_indptr[new_idx + 1] = new_indptr[new_idx] + nnz_this

total_nnz = int(new_indptr[-1])
est_gb = total_nnz * 5 / 1e9  # 4 bytes index + 1 byte data per entry
print(f"  Total nnz: {total_nnz:,} (~{est_gb:.2f} GB for CSR)")

if est_gb > 14.0:
    print(f"  WARNING: Estimated {est_gb:.1f} GB exceeds safe limit!")
    print(f"  Increase SCORE_PERCENTILE to reduce wells. Current: P{SCORE_PERCENTILE}")
    print(f"  Try P25 or P50 to reduce matrix size.")
    raise MemoryError(f"CSR matrix too large: {est_gb:.1f} GB estimated")

# Step 2: Fill indices from HDF5 in chunks (avoids loading all indices at once)
print(f"  Allocating indices array ({total_nnz:,} entries)...")
new_indices = np.empty(total_nnz, dtype=np.int32)

# Read HDF5 indices in chunks to avoid loading all at once
CHUNK_READ = 50000  # wells per read chunk
with h5py.File(str(h5_path), 'r') as hf:
    ds_indices = hf['indices']
    
    for chunk_start in tqdm(range(0, n_keep, CHUNK_READ), desc="Loading CSR"):
        chunk_end = min(chunk_start + CHUNK_READ, n_keep)
        
        for new_idx in range(chunk_start, chunk_end):
            old_idx = keep_indices[new_idx]
            src_start = int(all_indptr[old_idx])
            src_end = int(all_indptr[old_idx + 1])
            dst_start = int(new_indptr[new_idx])
            dst_end = int(new_indptr[new_idx + 1])
            
            if src_end > src_start:
                new_indices[dst_start:dst_end] = ds_indices[src_start:src_end]

# Step 3: Assemble CSR matrix from (data, indices, indptr)
new_data = np.ones(total_nnz, dtype=np.int8)
well_coverage = csr_matrix(
    (new_data, new_indices, new_indptr),
    shape=(n_keep, n_active)
)

del new_data, new_indices, all_indptr
gc.collect()

# Extract filtered params
filtered_params = all_params[keep_indices]
filtered_scores = scores[keep_indices]
del all_params
gc.collect()

@dataclass
class WellPathResult:
    params: np.ndarray
    coverage: csr_matrix
    scores: np.ndarray
    platform_x: float
    platform_y: float

well_result = WellPathResult(
    params=filtered_params.astype(np.float32),
    coverage=well_coverage,
    scores=filtered_scores.astype(np.int32),
    platform_x=platform_x,
    platform_y=platform_y,
)

print(f"  Coverage matrix: {well_coverage.shape}, nnz={well_coverage.nnz:,}")
print(f"  Score range: [{well_result.scores.min()}, {well_result.scores.max()}]")
print(f"  Estimated RAM: {well_coverage.nnz * 12 / 1e9:.2f} GB")
print(f"\u2713 well_result ready for optimization")


## Cell 12: Optimization Functions

Defines the optimization solvers (unchanged across versions):
- **`celf_max_coverage`** — Greedy CELF algorithm (Cost-Effective Lazy Forward). Picks k wells one at a time, maximizing marginal weighted coverage. Submodularity guarantee: ≥ 63% of optimal.
- **`max_coverage_ortools`** — CP-SAT exact solver via OR-Tools. ILP formulation with optional greedy warm-start.
- **`to_sparse_csr_csc`** — Format converter for the coverage matrix.


In [ ]:
# ==============================================================================
# OPTIMIZATION FUNCTIONS
# ==============================================================================

def to_sparse_csr_csc(coverage) -> Tuple[csr_matrix, csc_matrix]:
    if sparse.issparse(coverage):
        C = coverage
    else:
        C = csr_matrix(coverage, dtype=np.int8)
    return C.tocsr(copy=False), C.tocsc(copy=False)


def celf_max_coverage(
    C_csr: csr_matrix, k: int, weights: Optional[np.ndarray] = None
) -> Tuple[List[int], np.ndarray]:
    P, W = C_csr.shape
    if weights is None:
        weights = np.ones(W, dtype=float)
    else:
        weights = np.asarray(weights, dtype=float)
    
    indptr, indices = C_csr.indptr, C_csr.indices
    covered = np.zeros(W, dtype=bool)
    
    gains = np.zeros(P, dtype=float)
    for p in range(P):
        w_idx = indices[indptr[p]:indptr[p+1]]
        if w_idx.size:
            gains[p] = weights[w_idx].sum()
    
    heap = [(-gains[p], p, -1) for p in range(P)]
    heapify(heap)
    
    chosen = []
    iter_id = 0
    while len(chosen) < k and heap:
        gneg, p, last_eval = heappop(heap)
        if last_eval == iter_id:
            chosen.append(p)
            w_idx = indices[indptr[p]:indptr[p+1]]
            covered[w_idx] = True
            iter_id += 1
        else:
            w_idx = indices[indptr[p]:indptr[p+1]]
            new_gain = weights[w_idx[~covered[w_idx]]].sum() if w_idx.size else 0.0
            heappush(heap, (-new_gain, p, iter_id))
    return chosen, covered


def max_coverage_ortools(
    C_csc: csc_matrix, k: int,
    weights: Optional[np.ndarray] = None,
    hint: Optional[List[int]] = None,
    objective_floor: Optional[int] = None,
    time_limit: Optional[float] = None,
    num_workers: int = 0,
    rel_gap: Optional[float] = None
) -> Dict:
    P, W = C_csc.shape
    if weights is None:
        weights = np.ones(W, dtype=int)
    else:
        weights = np.asarray(weights, dtype=int)
    
    indptr, indices = C_csc.indptr, C_csc.indices
    cov = [indices[indptr[w]:indptr[w+1]].tolist() for w in range(W)]
    
    model = cp_model.CpModel()
    y = [model.NewBoolVar(f"y_{p}") for p in range(P)]
    x = [model.NewBoolVar(f"x_{w}") for w in range(W)]
    
    model.Add(sum(y) == int(k))
    for w in range(W):
        if cov[w]:
            model.Add(sum(y[p] for p in cov[w]) >= x[w])
        else:
            model.Add(x[w] == 0)
    
    obj_expr = sum(int(weights[w]) * x[w] for w in range(W))
    model.Maximize(obj_expr)
    
    if hint is not None:
        hint_set = set(hint)
        for p in range(P):
            model.AddHint(y[p], 1 if p in hint_set else 0)
        for w in range(W):
            model.AddHint(x[w], 1 if any(p in hint_set for p in cov[w]) else 0)
    
    if objective_floor is not None:
        model.Add(obj_expr >= int(objective_floor))
    
    solver = cp_model.CpSolver()
    if time_limit is not None:
        solver.parameters.max_time_in_seconds = float(time_limit)
    if num_workers:
        solver.parameters.num_search_workers = int(num_workers)
    if rel_gap is not None:
        solver.parameters.relative_gap_limit = float(rel_gap)
    
    status = solver.Solve(model)
    y_val = np.array([solver.Value(v) for v in y], dtype=int)
    x_val = np.array([solver.Value(v) for v in x], dtype=int)
    
    return {
        "status": status,
        "objective": float(solver.ObjectiveValue()),
        "selected": np.where(y_val == 1)[0].tolist(),
        "covered": np.where(x_val == 1)[0].tolist(),
        "best_bound": solver.BestObjectiveBound(),
        "runtime_s": solver.WallTime(),
    }


print("✓ Optimization functions defined.")


## Cell 13: Two-Phase Optimization

**Core optimization cell.** For each (k, m) pair:

- **Phase 1 (m fault wells):** CELF with fault+overlap cell weights (geobody-only zeroed), followed by swap refinement to improve the greedy solution.
- **Phase 2 (k−m geobody wells):** CELF with geobody-only weights (fault+overlap+Phase 1 coverage zeroed).
- **Combined objective:** Total BCF across both phases using full original weights.

When m=0: standard single-phase CELF (baseline, no fault priority).
When m≥k: all wells selected with fault-priority weights.

**⚠️ User parameters:**
- **`OPT.k_max`** — Maximum number of wells
- **`OPT.m_fault_values`** — List of m values to evaluate (e.g., `[0, 3, 5, 8]`)
- **`RUN_CPSAT`** — Enable/disable CP-SAT exact solver
- **`CPSAT_K_VALUES`** — Which k values to run CP-SAT on

Results stored in `all_opt_results[m]` — one list of results per m value.


In [ ]:
# ==============================================================================
# TWO-PHASE OPTIMIZATION — m fault wells + (k-m) geobody wells
# ==============================================================================
#
# User parameters:
#   RUN_CPSAT       — Enable CP-SAT exact solver
#   CPSAT_K_VALUES  — Which k values to run CP-SAT on
#   OPT.k_max       — Maximum wells (from Cell 1)
#   OPT.m_fault_values — List of m values to test (from Cell 1)
#

RUN_CPSAT = False
CPSAT_K_VALUES = []

@dataclass
class OptimizationResultPair:
    k: int
    m: int  # number of fault-priority wells (0 = baseline)
    greedy_selected: np.ndarray
    greedy_covered: np.ndarray
    greedy_objective: float
    greedy_runtime: float
    # Breakdown
    fault_selected: Optional[np.ndarray] = None
    geo_selected: Optional[np.ndarray] = None
    # CP-SAT fields
    cpsat_selected: Optional[np.ndarray] = None
    cpsat_covered: Optional[np.ndarray] = None
    cpsat_objective: Optional[float] = None
    cpsat_runtime: Optional[float] = None
    cpsat_best_bound: Optional[float] = None
    
    @property
    def has_cpsat(self) -> bool:
        return self.cpsat_objective is not None
    
    @property
    def best_objective(self) -> float:
        return self.cpsat_objective if self.has_cpsat else self.greedy_objective
    
    @property
    def best_selected(self) -> np.ndarray:
        return self.cpsat_selected if self.has_cpsat else self.greedy_selected
    
    @property
    def best_covered(self) -> np.ndarray:
        return self.cpsat_covered if self.has_cpsat else self.greedy_covered


def two_phase_optimize(C_csr, k, m, base_weights, cone_cell_type_active):
    """
    Two-phase well selection:
      Phase 1: Pick m wells maximizing fault + overlap cells (geobody-only → weight 0)
      Phase 2: Pick k-m wells maximizing geobody-only cells (fault+overlap → weight 0),
               with cells covered in Phase 1 zeroed out.
    
    Args:
        C_csr: sparse coverage matrix (n_wells × n_active_cells)
        k: total wells to pick
        m: fault-priority wells (0 = single-phase baseline)
        base_weights: per-active-cell weights (BCF values)
        cone_cell_type_active: cell_type array indexed to active cells only
    
    Returns:
        OptimizationResultPair
    """
    t0 = perf_counter()
    
    if m == 0 or m >= k:
        # Single-phase: standard CELF
        if m >= k:
            m = k  # all fault wells
        sel, cov = celf_max_coverage(C_csr, k, base_weights)
        obj = float(base_weights[cov].sum()) if base_weights is not None else float(cov.sum())
        return OptimizationResultPair(
            k=k, m=0 if m == 0 else k,
            greedy_selected=np.array(sel, dtype=int),
            greedy_covered=np.flatnonzero(cov),
            greedy_objective=obj,
            greedy_runtime=perf_counter() - t0,
            fault_selected=None if m == 0 else np.array(sel, dtype=int),
            geo_selected=np.array([], dtype=int) if m >= k else None,
        )
    
    # --- Phase 1: Fault wells ---
    fault_weights = base_weights.copy() if base_weights is not None else np.ones(C_csr.shape[1])
    geo_only_mask = (cone_cell_type_active == 1)  # geobody without fault overlap
    fault_weights[geo_only_mask] = 0.0
    
    fault_sel, fault_cov = celf_max_coverage(C_csr, m, fault_weights)
    
    # --- Phase 2: Geobody wells ---
    geo_weights = base_weights.copy() if base_weights is not None else np.ones(C_csr.shape[1])
    # viz_category: 2=fault_dip, 3=fault_anti, 4=overlap → zero for geobody phase
    fault_overlap_mask = (cone_cell_type_active == 2) | (cone_cell_type_active == 3) | (cone_cell_type_active == 4)
    geo_weights[fault_overlap_mask] = 0.0
    
    # Zero out cells already covered by Phase 1
    geo_weights[fault_cov] = 0.0
    
    geo_sel, geo_cov = celf_max_coverage(C_csr, k - m, geo_weights)
    
    # --- Combine ---
    combined_sel = np.concatenate([fault_sel, geo_sel]).astype(int)
    combined_cov = np.zeros(C_csr.shape[1], dtype=bool)
    combined_cov[np.flatnonzero(fault_cov)] = True
    combined_cov[np.flatnonzero(geo_cov)] = True
    
    if base_weights is not None:
        obj = float(base_weights[combined_cov].sum())
    else:
        obj = float(combined_cov.sum())
    
    return OptimizationResultPair(
        k=k, m=m,
        greedy_selected=combined_sel,
        greedy_covered=np.flatnonzero(combined_cov),
        greedy_objective=obj,
        greedy_runtime=perf_counter() - t0,
        fault_selected=np.array(fault_sel, dtype=int),
        geo_selected=np.array(geo_sel, dtype=int),
    )


# ==============================================================================
# Build cell_type vector aligned to active cells (same column order as coverage matrix)
# ==============================================================================
# category array was computed in the fault processing cell
# We need it indexed to active cells only (same as coverage matrix columns)
cone_cell_type_active = category[cone_grid.active]  # 1D array, length = n_active

# Weights
base_weights = cone_grid.active_values() if VOLUME.mode == 'volume' else None

C_csr, C_csc = to_sparse_csr_csc(well_result.coverage)

# ==============================================================================
# Run optimization for all (k, m) combinations
# ==============================================================================
m_values = sorted(set(OPT.m_fault_values))
print(f"Running optimization for k = 1..{OPT.k_max}, m ∈ {m_values}...")

opt_output_dir = PATHS.output_dir / "maxcov_results"
opt_output_dir.mkdir(parents=True, exist_ok=True)

# Results: dict keyed by m, each containing list of results for k=1..k_max
all_opt_results = {}

for m in m_values:
    print(f"\n--- m = {m} (fault-priority wells) ---")
    results_m = []
    
    for k in tqdm(range(1, OPT.k_max + 1), desc=f"m={m}"):
        effective_m = min(m, k)  # can't pick more fault wells than total
        result = two_phase_optimize(C_csr, k, effective_m, base_weights, cone_cell_type_active)
        results_m.append(result)
    
    all_opt_results[m] = results_m
    total_time = sum(r.greedy_runtime for r in results_m)
    print(f"  ✓ m={m} complete in {total_time:.2f}s")

# For backward compatibility, set opt_results to baseline (m=0 or first m)
opt_results = all_opt_results[m_values[0]]

# Print summary table
for m in m_values:
    results_m = all_opt_results[m]
    print(f"\nSummary (m={m}):")
    print(f"{'k':>4} {'Obj':>10} {'Time':>8}")
    print("-" * 26)
    for r in results_m:
        print(f"{r.k:>4} {r.greedy_objective:>10.2f} {r.greedy_runtime:>7.2f}s")

# Save combined summary
summary_path = opt_output_dir / "summary_two_phase.csv"
with open(summary_path, "w", newline="") as fcsv:
    writer = csv.writer(fcsv)
    writer.writerow(["m", "k", "objective", "n_fault_wells", "n_geo_wells", "runtime_s"])
    for m_val in m_values:
        for r in all_opt_results[m_val]:
            n_f = len(r.fault_selected) if r.fault_selected is not None else 0
            n_g = len(r.geo_selected) if r.geo_selected is not None else r.k
            writer.writerow([r.m, r.k, f"{r.greedy_objective:.4f}", n_f, n_g, f"{r.greedy_runtime:.4f}"])

print(f"\n✓ Results saved to {summary_path}")


## Cell 14: Creaming Curves

Plots cumulative covered volume (BCF) vs. number of wells for each m value:
- **Combined comparison plot** — all m values overlaid
- **Individual per-m plots** — each with marginal gains bar chart
- **CSV export** — tabular data for external analysis

**Output:** `output_dir/creaming_curves/`


In [ ]:
# ==============================================================================
# CREAMING CURVES — Coverage vs. wells, one curve per m value
# ==============================================================================

creaming_dir = PATHS.output_dir / "creaming_curves"
creaming_dir.mkdir(parents=True, exist_ok=True)

m_values = sorted(all_opt_results.keys())
n_m = len(m_values)

# Determine units
if VOLUME.mode == 'volume':
    _total_potential = float(cone_grid.active_values().sum())
    y_label = 'Covered Volume (BCF)'
    y_label_marginal = 'Δ Volume (BCF)'
    unit_str = 'BCF'
else:
    _total_potential = float(cone_grid.n_active)
    y_label = 'Covered Cells'
    y_label_marginal = 'Δ Cells'
    unit_str = 'cells'

m_colors = plt.cm.tab10(np.linspace(0, 0.7, max(n_m, 1)))

# ── 1. Combined comparison plot ─────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), height_ratios=[2, 1])

y_max_all = 0

for mi, m_val in enumerate(m_values):
    results_m = all_opt_results[m_val]
    ks = np.array([r.k for r in results_m])
    objs = np.array([r.greedy_objective for r in results_m])
    marginals = np.diff(objs, prepend=0)
    
    color = m_colors[mi]
    label = f'm={m_val}' if m_val > 0 else 'm=0 (baseline)'
    ls = '-' if m_val == 0 else '--'
    lw = 2.5 if m_val == 0 else 1.5
    
    ax1.plot(ks, objs, f'o{ls}', linewidth=lw, markersize=4, color=color, label=label)
    ax2.bar(ks + mi * 0.15 - (n_m - 1) * 0.075, marginals, width=0.15,
            color=color, alpha=0.7, label=label)
    
    y_max_all = max(y_max_all, objs.max())

ax1.set_xlabel('k (total wells)')
ax1.set_ylabel(y_label)
ax1.set_xlim(0, ks.max() + 1)
ax1.set_ylim(0, y_max_all * 1.1)
ax1.legend(loc='lower right', fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_title('Creaming Curves: Fault-Priority Comparison', fontweight='bold')

ax1b = ax1.twinx()
if _total_potential > 0:
    ax1b.set_ylim(0, 100 * y_max_all * 1.1 / _total_potential)
ax1b.set_ylabel('Coverage (%)', color='gray')
ax1b.tick_params(axis='y', labelcolor='gray')

ax2.set_xlabel('k (total wells)')
ax2.set_ylabel(y_label_marginal)
ax2.set_xlim(0, ks.max() + 1)
ax2.legend(fontsize=8, ncol=min(n_m, 5))
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_title('Marginal Gains per Well')

plt.tight_layout()
fig.savefig(str(creaming_dir / "creaming_all_m_comparison.png"), dpi=300, bbox_inches='tight')
plt.show()
print(f"  Saved: creaming_all_m_comparison.png")

# ── 2. Individual plots per m ──────────────────────────────────────────────
for mi, m_val in enumerate(m_values):
    results_m = all_opt_results[m_val]
    ks = np.array([r.k for r in results_m])
    objs = np.array([r.greedy_objective for r in results_m])
    marginals = np.diff(objs, prepend=0)
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), height_ratios=[2, 1])
    
    color = m_colors[mi]
    label = f'm={m_val}' if m_val > 0 else 'm=0 (baseline)'
    
    ax1.plot(ks, objs, 'o-', linewidth=2, markersize=5, color=color, label=label)
    ax1.set_xlabel('k (total wells)')
    ax1.set_ylabel(y_label)
    ax1.set_xlim(0, ks.max() + 1)
    ax1.set_ylim(0, max(objs.max(), 1) * 1.1)
    ax1.legend(loc='lower right', fontsize=10)
    ax1.grid(True, alpha=0.3)
    
    title = f'Creaming Curve: m={m_val}'
    if m_val > 0:
        title += f' ({m_val} fault + k-{m_val} geobody wells)'
    else:
        title += ' (no fault priority)'
    ax1.set_title(title, fontweight='bold')
    
    ax1b = ax1.twinx()
    if _total_potential > 0:
        ax1b.set_ylim(0, 100 * max(objs.max(), 1) * 1.1 / _total_potential)
    ax1b.set_ylabel(f'Coverage (% of {_total_potential:.2f} {unit_str})', color='gray')
    ax1b.tick_params(axis='y', labelcolor='gray')
    
    ax2.bar(ks, marginals, width=0.6, color=color, alpha=0.7, label=f'Marginal ({label})')
    ax2.set_xlabel('k (total wells)')
    ax2.set_ylabel(y_label_marginal)
    ax2.set_xlim(0, ks.max() + 1)
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3, axis='y')
    ax2.set_title('Marginal Gains per Well')
    
    plt.tight_layout()
    fname = f"creaming_m{m_val:02d}.png"
    fig.savefig(str(creaming_dir / fname), dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"  Saved: {fname}")

# ── 3. Summary table + CSV ─────────────────────────────────────────────────
print(f"\nCreaming Curve Summary:")
print(f"  Total reachable: {_total_potential:.2f} {unit_str}")
print()
header = f"{'k':>4}"
for m_val in m_values:
    header += f"  {'m='+str(m_val):>10}"
print(header)
print("-" * (4 + 12 * n_m))

for k_idx in range(len(all_opt_results[m_values[0]])):
    row = f"{k_idx + 1:>4}"
    for m_val in m_values:
        obj = all_opt_results[m_val][k_idx].greedy_objective
        row += f"  {obj:>10.2f}"
    print(row)

# CSV export
csv_path = creaming_dir / "creaming_curves.csv"
with open(str(csv_path), 'w', newline='') as fcsv:
    writer = csv.writer(fcsv)
    writer.writerow(['k'] + [f'm={m}' for m in m_values])
    for k_idx in range(len(all_opt_results[m_values[0]])):
        row = [k_idx + 1]
        for m_val in m_values:
            row.append(f"{all_opt_results[m_val][k_idx].greedy_objective:.4f}")
        writer.writerow(row)
print(f"\n  CSV saved: {csv_path}")

# Final comparison
print(f"\nAt k={OPT.k_max}:")
for m_val in m_values:
    final = all_opt_results[m_val][-1].greedy_objective
    pct = 100 * final / _total_potential if _total_potential > 0 else 0
    print(f"  m={m_val}: {final:.2f} {unit_str} ({pct:.1f}%)")

print(f"\n✓ All creaming curves saved to {creaming_dir}")


## Cell 15: 3D Scene Export

Exports interactive 3D HTML scenes per (k, m) pair. Each scene shows:
- Gray point cloud — background geobody
- **Category-colored covered cells** — Green (geobody), Gold (fault dip), Blue (fault anti), Red (overlap)
- Red tubes — selected well paths
- Fault contour surfaces — Delaunay-triangulated from contour traces
- Cone + cylinder — reachable volume envelope

**Output:** `output_dir/3d_scenes/m_XX/k_YYY_m_XX.html`


In [ ]:
# ==============================================================================
# 3D SCENE EXPORT — Interactive HTML per (k, m)
# ==============================================================================

# Ensure build_active_voxels is available (defined in Cell 6b — 3D Grid Viz).
# If Cell 6b was skipped after kernel restart, define a minimal version here.
if 'build_active_voxels' not in dir():
    def build_active_voxels(grid, scalars=None, scalar_name="value", z_scale=1.0):
        """Fallback: build PyVista point cloud from active cells."""
        import pyvista as pv
        I, J, K = np.where(grid.active)
        n = len(I)
        if n == 0:
            return pv.PolyData()
        cx = grid.x_origin + I.astype(float) * grid.dx
        cy = grid.y_origin + J.astype(float) * grid.dy
        cz = (grid.z_origin + K.astype(float) * grid.dz) * z_scale
        pts = np.column_stack([cx, cy, cz]).astype(np.float32)
        cloud = pv.PolyData(pts)
        if scalars is None:
            scalars = grid.values[grid.active]
        cloud.point_data[scalar_name] = scalars.astype(np.float32)
        print(f"  [fallback] Built point cloud with {n:,} points")
        return cloud
    print("  NOTE: build_active_voxels not found — using fallback. Run Cell 6b for full hex mesh.")

def export_3d_scene(
    result: OptimizationResultPair,
    bg_grid: RegularGrid3D,
    cone_grid: RegularGrid3D,
    well_result: WellPathResult,
    cone: ConeConfig,
    outfile: Path,
    all_fault_contours: dict = None,
    category: np.ndarray = None,
    gray_stride: int = 100,
    max_gray_paths: int = 500,
    bg_subsample: int = 3,
):
    """
    Export 3D scene — optimized for speed.
    
    Speed improvements vs previous version:
      - Background geobody: point cloud instead of hex mesh (~10x faster)
      - Background subsampled by bg_subsample (every Nth active cell)
      - Gray paths: capped at max_gray_paths, stride increased 10->100
      - Gray paths as lines (no .tube()) — 50x fewer polygons
      - Covered cells as point cloud (not hex mesh) when >50K cells
    """
    import pyvista as pv
    
    xb, yb = well_result.platform_x, well_result.platform_y
    selected_paths = result.best_selected
    covered_idx = result.best_covered
    method_used = "CP-SAT" if result.has_cpsat else "Greedy"
    
    # ---- Background geobody (gray point cloud, subsampled) ----
    I_bg, J_bg, K_bg = np.where(bg_grid.active)
    # Subsample for speed
    if bg_subsample > 1:
        idx_sub = np.arange(0, len(I_bg), bg_subsample)
        I_bg, J_bg, K_bg = I_bg[idx_sub], J_bg[idx_sub], K_bg[idx_sub]
    
    bg_pts = np.column_stack([
        bg_grid.x_origin + I_bg.astype(np.float32) * bg_grid.dx,
        bg_grid.y_origin + J_bg.astype(np.float32) * bg_grid.dy,
        bg_grid.z_origin + K_bg.astype(np.float32) * bg_grid.dz,
    ])
    bg_mesh = pv.PolyData(bg_pts)
    del I_bg, J_bg, K_bg, bg_pts
    print(f"  Background: {bg_mesh.n_points:,} points (subsample={bg_subsample})")
    
    # ---- Covered cells ----
    active_flat = cone_grid.active_flat_indices
    covered_3d = np.zeros((cone_grid.nx, cone_grid.ny, cone_grid.nz), dtype=bool)
    if len(covered_idx) > 0 and len(active_flat) > 0:
        valid_idx = covered_idx[covered_idx < len(active_flat)]
        flat_covered = active_flat[valid_idx]
        ci, cj, ck = np.unravel_index(flat_covered, (cone_grid.nx, cone_grid.ny, cone_grid.nz))
        covered_3d[ci, cj, ck] = True
    
    I_cov, J_cov, K_cov = np.where(covered_3d)
    n_cov = len(I_cov)
    cov_vals = cone_grid.values[covered_3d]
    
    # Use point cloud for covered cells (fast) or hex mesh for small counts
    cov_pts = np.column_stack([
        cone_grid.x_origin + I_cov.astype(np.float32) * cone_grid.dx,
        cone_grid.y_origin + J_cov.astype(np.float32) * cone_grid.dy,
        cone_grid.z_origin + K_cov.astype(np.float32) * cone_grid.dz,
    ])
    covered_mesh = pv.PolyData(cov_pts)
    covered_mesh.point_data["val"] = cov_vals.astype(np.float32)
    print(f"  Covered: {n_cov:,} points")
    
    # Z range for paths
    _, _, bz = bg_grid.active_xyz()
    all_z_min = float(bz.min())
    z_for_paths = np.linspace(0.0, all_z_min, num=100) if all_z_min < 0 else np.array([0.0, -1.0])
    
    # Cone geometry
    height = float(cone.z0 - all_z_min)
    h_cone = min(cone.hd / cone.tan_theta, height)
    
    pv.global_theme.background = 'white'
    pv.global_theme.window_size = (1400, 1000)
    pl = pv.Plotter(off_screen=True)
    
    # Background: gray point cloud of all geobody cells (subsampled)
    if bg_mesh.n_points > 0:
        pl.add_mesh(bg_mesh, color="#999999", opacity=0.15,
                    point_size=1.5, render_points_as_spheres=False)
    
    # Covered cells — colored by cell category
    if n_cov > 0 and category is not None:
        cov_cat = category[covered_3d]
        
        cat_styles = [
            (1, "#4CAF50", "Covered: Geobody", 3),     # green
            (2, "#FFD700", "Covered: Fault Dip", 4),    # gold
            (3, "#4169E1", "Covered: Fault Anti", 4),   # blue
            (4, "#FF2222", "Covered: Overlap", 4),      # red
        ]
        for cat_val, color, label, pt_size in cat_styles:
            mask_c = (cov_cat == cat_val)
            if mask_c.any():
                pl.add_mesh(pv.PolyData(cov_pts[mask_c]), color=color, opacity=0.9,
                            point_size=pt_size, render_points_as_spheres=False, label=label)
        del cov_cat
    elif covered_mesh.n_points > 0:
        # Fallback: color by value if no category
        covered_mesh.point_data["val"] = cov_vals.astype(np.float32)
        pl.add_mesh(covered_mesh, scalars="val", cmap="viridis", opacity=0.9,
                    point_size=3, render_points_as_spheres=False)
    
    # ---- Fault contour surfaces ----
    if all_fault_contours:
        fault_colors_3d = ['#FF6600', '#00CC66', '#CC33FF', '#FF3399', '#33CCCC']
        for fi, (fname, contours_by_k) in enumerate(all_fault_contours.items()):
            # Collect XY contours at each z-level into a 3D point cloud
            all_pts = []
            for k_idx in sorted(contours_by_k.keys()):
                sl = contours_by_k[k_idx]
                xy = sl['xy']
                z_val = sl['z_level']
                # Subsample long contours for speed
                step = max(1, len(xy) // 50)
                xy_sub = xy[::step]
                z_col = np.full(len(xy_sub), z_val, dtype=np.float32)
                pts_3d = np.column_stack([xy_sub[:, 0], xy_sub[:, 1], z_col])
                all_pts.append(pts_3d)
            
            if not all_pts:
                continue
            
            all_pts = np.vstack(all_pts).astype(np.float32)
            
            # Build surface via Delaunay triangulation
            cloud = pv.PolyData(all_pts)
            try:
                surf = cloud.delaunay_2d(alpha=200.0)
                fc = fault_colors_3d[fi % len(fault_colors_3d)]
                pl.add_mesh(surf, color=fc, opacity=0.4, show_edges=False,
                            smooth_shading=True, label=f"Fault: {fname}")
                print(f"  Fault surface '{fname}': {surf.n_cells:,} triangles ({fc})")
            except Exception as e:
                # Fallback: render as point cloud
                fc = fault_colors_3d[fi % len(fault_colors_3d)]
                pl.add_mesh(cloud, color=fc, opacity=0.5, point_size=2,
                            render_points_as_spheres=False, label=f"Fault: {fname}")
                print(f"  Fault surface '{fname}': point cloud fallback ({e})")
            del all_pts, cloud
    
        # Wellhead marker
    tube_axis = np.array([[xb, yb, 0.0], [xb, yb, 100.0]])
    pl.add_mesh(pv.lines_from_points(tube_axis, close=False).tube(radius=50.0), color='#FFFB00')
    
    # Cone
    cone_mesh = pv.Cone(
        center=(xb, yb, cone.z0 - h_cone / 2.0),
        direction=(0.0, 0.0, 1.0), height=h_cone,
        radius=np.minimum((cone.z0 - all_z_min) * cone.tan_theta, cone.hd),
        capping=False, resolution=64
    )
    pl.add_mesh(cone_mesh, color='#1f77b4', opacity=0.1, smooth_shading=True)
    
    h_cyl = max(0.0, height - h_cone)
    if h_cyl > 0:
        cyl_mesh = pv.Cylinder(
            center=(xb, yb, cone.z0 - h_cone - h_cyl / 2.0),
            direction=(0.0, 0.0, 1.0),
            height=h_cyl, radius=cone.hd, resolution=64
        )
        pl.add_mesh(cyl_mesh, color='#1f77b4', opacity=0.1, smooth_shading=True)
    
    # Gray paths — LINES only (no .tube()), capped count
    n_total_paths = len(well_result.params)
    step = max(gray_stride, n_total_paths // max_gray_paths)
    n_gray = 0
    for i in range(0, n_total_paths, step):
        prm = well_result.params[i]
        v, alpha_deg, p1_deg, h, p2_deg, beta_deg, t_val, DLS1, DLS2 = prm[:9]
        try:
            x_line, y_line, _, _ = three_segment_path_xy(
                x0=xb, y0=yb, z=z_for_paths,
                alpha_deg=alpha_deg, v=v, p1_deg=p1_deg, h=h,
                p2_deg=p2_deg, beta_deg=beta_deg, t=t_val, DLS1_deg_per_m=DLS1, DLS2_deg_per_m=DLS2
            )
            line_pts = np.c_[x_line, y_line, z_for_paths]
            pl.add_mesh(pv.lines_from_points(line_pts, close=False),
                        color="#5C5C5C", opacity=0.1, line_width=2.0)
            n_gray += 1
        except:
            pass
    print(f"  Gray paths: {n_gray} (stride={step})")
    
    # Red paths (selected) — keep tubes for visibility
    for i in selected_paths:
        prm = well_result.params[i]
        v, alpha_deg, p1_deg, h, p2_deg, beta_deg, t_val, DLS1, DLS2 = prm[:9]
        try:
            x_line, y_line, _, _ = three_segment_path_xy(
                x0=xb, y0=yb, z=z_for_paths,
                alpha_deg=alpha_deg, v=v, p1_deg=p1_deg, h=h,
                p2_deg=p2_deg, beta_deg=beta_deg, t=t_val, DLS1_deg_per_m=DLS1, DLS2_deg_per_m=DLS2
            )
            line_pts = np.c_[x_line, y_line, z_for_paths]
            pl.add_mesh(pv.lines_from_points(line_pts, close=False).tube(radius=15.0), color='#ed3838')
        except:
            pass
    print(f"  Selected paths: {len(selected_paths)} (tubes)")
    
    # Outline box
    x_min = bg_grid.x_origin - bg_grid.dx / 2
    x_max = bg_grid.x_origin + (bg_grid.nx - 0.5) * bg_grid.dx
    y_min = bg_grid.y_origin - bg_grid.dy / 2
    y_max = bg_grid.y_origin + (bg_grid.ny - 0.5) * bg_grid.dy
    z_min_box = bg_grid.z_origin + (bg_grid.nz - 0.5) * bg_grid.dz
    outline = pv.Box(bounds=[x_min, x_max, y_min, y_max, z_min_box, 0.0])
    pl.add_mesh(outline.outline(), color='#555555', line_width=1.5)
    
    pl.add_axes(line_width=2)
    pl.show_bounds(grid=True, location='outer', all_edges=True)
    pl.camera_position = 'iso'
    pl.export_html(str(outfile))
    pl.close()
    
    del bg_mesh, covered_mesh
    gc.collect()
    
    return outfile, method_used


viz_output_dir = PATHS.output_dir / "3d_scenes"
viz_output_dir.mkdir(parents=True, exist_ok=True)

print("Exporting 3D scenes...")
export_ks = sorted(set(k for k in [1, 2, 3, 4, 5, 6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24] if 1 <= k <= len(opt_results)))

plot_m_opt_result = all_opt_results[m_values[10]] #<<<<<Specify which m to plot

for k in tqdm(export_ks, desc="Exporting HTML"):
    result = plot_m_opt_result[k - 1]
    outfile = viz_output_dir / f"k_{k:03d}.html"
    _, method = export_3d_scene(result, filtered_grid, cone_grid, well_result, CONE, outfile, all_fault_contours, category)
    print(f"  k={k}: {outfile.name} ({method})")

print(f"\n\u2713 3D scenes exported to {viz_output_dir}")


In [ ]:
# ==============================================================================
# Z-SLICE WELL VIEWER — Well paths + coverage at 6 depth slices
# ==============================================================================

def plot_well_zslices(
    result: OptimizationResultPair,
    grid: RegularGrid3D,
    category,
    all_fault_contours: dict,
    well_result: WellPathResult,
    well_cfg,
    n_slices: int = 6,
    xmin=None, xmax=None, ymin=None, ymax=None,
    figsize=(16, 10),
    dpi=300,
    outfile=None,
    show_grid_lines=False,
    grid_line_color=0.82,
):
    """
    Plot n_slices z-slices from surface to deepest active cell,
    overlaying selected well positions and coverage radii.
    
    Supports 9-parameter wells (with beta) and two-phase results (fault/geo coloring).
    """
    from matplotlib.patches import Circle
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch
    
    xb, yb = well_result.platform_x, well_result.platform_y
    selected = result.best_selected
    k = result.k
    m = result.m if hasattr(result, 'm') else 0
    method_used = "CP-SAT" if result.has_cpsat else "Greedy"
    
    # Classify each selected well as fault (Phase 1) or geobody (Phase 2)
    fault_wells_set = set()
    geo_wells_set = set()
    if hasattr(result, 'fault_selected') and result.fault_selected is not None and len(result.fault_selected) > 0:
        fault_wells_set = set(result.fault_selected.tolist())
    if hasattr(result, 'geo_selected') and result.geo_selected is not None and len(result.geo_selected) > 0:
        geo_wells_set = set(result.geo_selected.tolist())
    
    x_axis = grid.x_axis
    y_axis = grid.y_axis
    z_axis = grid.z_axis
    
    if xmin is None: xmin = x_axis[0] - grid.dx / 2
    if xmax is None: xmax = x_axis[-1] + grid.dx / 2
    if ymin is None: ymin = y_axis[0] - grid.dy / 2
    if ymax is None: ymax = y_axis[-1] + grid.dy / 2
    
    # Z levels
    active_z = z_axis[np.any(grid.active, axis=(0, 1))]
    if len(active_z) == 0:
        print("  No active cells!")
        return
    z_deep = float(active_z.min())
    z_levels = np.linspace(0.0, z_deep, n_slices)
    
    # Dense z for trajectory computation
    z_dense = np.sort(np.unique(np.r_[0.0, z_axis]))
    
    # Compute selected well trajectories
    well_paths = []
    well_types = []  # 'fault', 'geo', or 'mixed'
    
    for wi, well_idx in enumerate(selected):
        prm = well_result.params[well_idx]
        v, alpha_deg, p1_deg, h_val, p2_deg, beta_deg, t_val, DLS1, DLS2 = prm[:9]
        try:
            x_line, y_line, _, _ = three_segment_path_xy(
                x0=xb, y0=yb, z=z_dense,
                alpha_deg=alpha_deg, v=v, p1_deg=p1_deg, h=h_val,
                p2_deg=p2_deg, beta_deg=beta_deg, t=t_val,
                DLS1_deg_per_m=DLS1, DLS2_deg_per_m=DLS2
            )
            well_paths.append((z_dense, x_line, y_line))
        except:
            well_paths.append(None)
        
        # Classify well type
        if well_idx in fault_wells_set:
            well_types.append('fault')
        elif well_idx in geo_wells_set:
            well_types.append('geo')
        else:
            well_types.append('mixed')
    
    # Color scheme: fault wells = orange squares, geo wells = blue circles
    def get_well_color(wtype, wi):
        if wtype == 'fault':
            return '#FF6600'
        elif wtype == 'geo':
            return '#1E4DE9'
        else:
            return plt.cm.Set1(wi / max(len(selected), 1))
    
    # Crop indices
    i_lo = max(0, int(np.floor((xmin - x_axis[0]) / grid.dx)) - 1)
    i_hi = min(grid.nx, int(np.ceil((xmax - x_axis[0]) / grid.dx)) + 2)
    j_lo = max(0, int(np.floor((ymin - y_axis[0]) / grid.dy)) - 1)
    j_hi = min(grid.ny, int(np.ceil((ymax - y_axis[0]) / grid.dy)) + 2)
    
    x_edge_lo = grid.x_origin + i_lo * grid.dx - grid.dx / 2
    x_edge_hi = grid.x_origin + i_hi * grid.dx - grid.dx / 2
    y_edge_lo = grid.y_origin + j_lo * grid.dy - grid.dy / 2
    y_edge_hi = grid.y_origin + j_hi * grid.dy - grid.dy / 2
    
    color_map = {
        0: [1.0, 1.0, 1.0, 0.0],
        1: [0.6, 0.6, 0.6, 0.8],
        2: [1.0, 0.84, 0.0, 0.8],
        3: [0.255, 0.412, 0.882, 0.8],
        4: [1.0, 0.133, 0.133, 1.0],
    }
    
    cell_px = 3 if show_grid_lines else 1
    n_cols = 3
    n_rows = (n_slices + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    axes_flat = axes.flatten() if n_slices > 1 else [axes]
    
    fault_colors = plt.cm.tab10(np.linspace(0, 1, max(len(all_fault_contours), 1)))
    
    for si, z_target in enumerate(z_levels):
        ax = axes_flat[si]
        
        k_idx = int(np.argmin(np.abs(z_axis - z_target)))
        z_actual = z_axis[k_idx]
        
        # Build RGBA
        cat_slice = category[i_lo:i_hi, j_lo:j_hi, k_idx]
        rgba = np.zeros((i_hi - i_lo, j_hi - j_lo, 4), dtype=np.float32)
        for cat_val, color in color_map.items():
            mask = cat_slice == cat_val
            rgba[mask] = color
        
        if show_grid_lines:
            rgba = np.repeat(np.repeat(rgba, cell_px, axis=0), cell_px, axis=1)
            gc_val = grid_line_color
            rgba[::cell_px, :, :3] = gc_val
            rgba[::cell_px, :, 3] = np.maximum(rgba[::cell_px, :, 3], 0.4)
            rgba[:, ::cell_px, :3] = gc_val
            rgba[:, ::cell_px, 3] = np.maximum(rgba[:, ::cell_px, 3], 0.4)
        
        rgba_plot = np.transpose(rgba, (1, 0, 2))
        del rgba
        
        ax.imshow(rgba_plot, origin='lower',
                  extent=[x_edge_lo, x_edge_hi, y_edge_lo, y_edge_hi],
                  aspect='equal', interpolation='nearest')
        del rgba_plot
        
        # Fault contours
        for fi, (fname, contours_by_k) in enumerate(all_fault_contours.items()):
            fc = fault_colors[fi % len(fault_colors)]
            sl = None
            if k_idx in contours_by_k:
                sl = contours_by_k[k_idx]
            else:
                available_k = sorted(contours_by_k.keys())
                nearest_k = min(available_k, key=lambda kk: abs(kk - k_idx)) if available_k else None
                if nearest_k is not None and abs(nearest_k - k_idx) <= 5:
                    sl = contours_by_k[nearest_k]
            
            if sl is not None:
                xy = sl['xy']
                ax.plot(xy[:, 0], xy[:, 1], '-', color=fc, linewidth=1.5, zorder=5)
                dip_coords = extract_polygon_coords(sl['buffer_dip'])
                if dip_coords is not None:
                    ax.plot(dip_coords[:, 0], dip_coords[:, 1], '--',
                            color=fc, linewidth=0.8, alpha=0.5, zorder=4)
                anti_coords = extract_polygon_coords(sl['buffer_anti'])
                if anti_coords is not None:
                    ax.plot(anti_coords[:, 0], anti_coords[:, 1], ':',
                            color=fc, linewidth=0.8, alpha=0.5, zorder=4)
        
        # Gray background: all candidate well paths (subsampled, computed once)
        if si == 0:
            _gray_stride = max(1, len(well_result.params) // 500)
            _gray_paths = []
            for gi in range(0, len(well_result.params), _gray_stride):
                gprm = well_result.params[gi]
                gv, ga, gp1, gh, gp2, gbeta, gt, gd1, gd2 = gprm[:9]
                try:
                    gx, gy, _, _ = three_segment_path_xy(
                        x0=xb, y0=yb, z=z_dense,
                        alpha_deg=ga, v=gv, p1_deg=gp1, h=gh,
                        p2_deg=gp2, beta_deg=gbeta, t=gt,
                        DLS1_deg_per_m=gd1, DLS2_deg_per_m=gd2)
                    _gray_paths.append((gx, gy))
                except:
                    pass
        
        for gx, gy in _gray_paths:
            ax.plot(gx, gy, '-', color='#AAAAAA', linewidth=0.3, alpha=0.15, zorder=2)
        
        # Selected well paths (colored by type: fault=orange, geo=blue)
        for wi, (path_data, wtype) in enumerate(zip(well_paths, well_types)):
            if path_data is None:
                continue
            _, x_path, y_path = path_data
            path_color = '#FF6600' if wtype == 'fault' else '#1E4DE9' if wtype == 'geo' else 'red'
            ax.plot(x_path, y_path, '-', color=path_color, linewidth=1.2, alpha=0.8, zorder=6)
        
        # Well markers + coverage radius circles at this depth
        for wi, (well_idx, path_data, wtype) in enumerate(zip(selected, well_paths, well_types)):
            if path_data is None:
                continue
            z_path, x_path, y_path = path_data
            
            xw = float(np.interp(z_actual, z_path, x_path))
            yw = float(np.interp(z_actual, z_path, y_path))
            
            wc = get_well_color(wtype, wi)
            
            circle = Circle((xw, yw), well_cfg.coverage_radius,
                           fill=False, edgecolor=wc, linewidth=1.0,
                           alpha=0.6, linestyle='-', zorder=7)
            ax.add_patch(circle)
            
            marker = 's' if wtype == 'fault' else 'o'
            ax.plot(xw, yw, marker, color=wc, markersize=4, zorder=8,
                    markeredgecolor='black', markeredgewidth=0.3)
        
        ax.plot(xb, yb, '*', color='yellow', markersize=10, zorder=10,
                markeredgecolor='black', markeredgewidth=0.5)
        
        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.set_title(f'z = {z_actual:.0f} m  (layer k={k_idx})', fontsize=10)
        ax.set_aspect('equal')
        ax.tick_params(labelsize=7)
    
    for si in range(n_slices, len(axes_flat)):
        axes_flat[si].set_visible(False)
    
    title = f'Well Placement: k={k} wells'
    if m > 0:
        title += f' (m={m} fault + {k-m} geobody)'
    title += f' — {method_used}'
    fig.suptitle(title, fontsize=14, fontweight='bold')
    
    # Legend
    legend_elements = [
        Patch(facecolor='#999999', alpha=0.8, label='Geobody'),
        Patch(facecolor='#FFD700', alpha=0.8, label='Fault Dip'),
        Patch(facecolor='#4169E1', alpha=0.8, label='Fault Anti-Dip'),
        Patch(facecolor='#FF2222', alpha=1.0, label='Overlap'),
        Line2D([0], [0], marker='*', color='yellow', markersize=10,
               markeredgecolor='black', linestyle='', label='Platform'),
        Line2D([0], [0], color='#AAAAAA', linewidth=1, alpha=0.4, label='All wells (top view)'),
    ]
    
    if fault_wells_set:
        legend_elements.append(
            Line2D([0], [0], marker='s', color='#FF6600', markersize=6,
                   markeredgecolor='black', linestyle='-', linewidth=1.2,
                   label=f'Fault wells ({len(fault_wells_set)})'))
    if geo_wells_set:
        legend_elements.append(
            Line2D([0], [0], marker='o', color='#1E4DE9', markersize=6,
                   markeredgecolor='black', linestyle='-', linewidth=1.2,
                   label=f'Geobody wells ({len(geo_wells_set)})'))
    if not fault_wells_set and not geo_wells_set:
        legend_elements.append(
            Line2D([0], [0], color='red', linewidth=1.5, label='Selected wells'))
    
    fig.legend(handles=legend_elements, loc='lower center',
               ncol=min(8, len(legend_elements)),
               fontsize=8, framealpha=0.9, bbox_to_anchor=(0.5, -0.02))
    
    plt.tight_layout(rect=[0, 0.04, 1, 0.96])
    
    if outfile:
        fig.savefig(str(outfile), dpi=dpi, bbox_inches='tight')
    
    plt.close(fig)
    del fig
    gc.collect()


# ── Execute: export per m value ────────────────────────────────────────────

Z_WELL_PARAMS = dict(
    xmin=909000,
    xmax=919000,
    ymin=827000,
    ymax=844000,
)

viz_dir = PATHS.output_dir / "3d_scenes" / "well_zslices"
viz_dir.mkdir(parents=True, exist_ok=True)

export_ks = sorted(set(k for k in [1, 5, 10, 15, 20, min(24, OPT.k_max)] if 1 <= k <= OPT.k_max))

for m_val in sorted(all_opt_results.keys()):
    results_m = all_opt_results[m_val]
    m_dir = viz_dir / f"m_{m_val:02d}"
    m_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"Exporting well z-slice plots for m={m_val}...")
    valid_ks = [k for k in export_ks if k <= len(results_m)]
    
    for k in tqdm(valid_ks, desc=f"m={m_val}"):
        result = results_m[k - 1]
        outfile = m_dir / f"well_zslices_k{k:03d}_m{m_val:02d}.png"
        plot_well_zslices(
            result=result,
            grid=filtered_grid,
            category=category,
            all_fault_contours=all_fault_contours,
            well_result=well_result,
            well_cfg=WELL,
            n_slices=6,
            outfile=outfile,
            **Z_WELL_PARAMS,
        )

print(f"\nSaved to {viz_dir}")


## Cell 16: Summary

Prints final grid statistics, well search results, and optimization summary.


In [ ]:
# ==============================================================================
# FINAL SUMMARY — Grid, wells, and optimization statistics
# ==============================================================================

print("=" * 70)
print("PATHSIGHT3D v2.0 WORKFLOW COMPLETE (RegularGrid3D)")
print("=" * 70)

print(f"\n📁 Data (RegularGrid3D):")
print(f"   Grid dimensions: {raw_grid.nx} × {raw_grid.ny} × {raw_grid.nz}")
print(f"   Spacing: dx={raw_grid.dx:.2f}, dy={raw_grid.dy:.2f}, dz={raw_grid.dz:.2f}")
print(f"   Original active: {raw_grid.n_active:,}")
print(f"   Filtered active: {filtered_grid.n_active:,}")
print(f"   Cone active: {cone_grid.n_active:,}")

print(f"\n📍 Platform Analysis:")
print(f"   Candidates evaluated: {len(platform_candidates):,}")
print(f"   Best platform: ({platform_result.best_x:.1f}, {platform_result.best_y:.1f})")
print(f"   Best score: {platform_result.best_score:.0f}")

print(f"\n🛢️ Well Simulation:")
print(f"   Platform used: ({xb:.1f}, {yb:.1f})")
print(f"   Valid well paths: {len(well_result.params):,}")
print(f"   Coverage matrix: {well_result.coverage.shape}")

print(f"\n📊 Optimization:")
print(f"   k range: 1 to {OPT.k_max}")
final_result = opt_results[-1]
print(f"   GREEDY at k={final_result.k}: {final_result.greedy_objective:.0f} BCF covered")

cpsat_results = [r for r in opt_results if r.has_cpsat]
if cpsat_results:
    best_cpsat = max(cpsat_results, key=lambda r: r.cpsat_objective)
    print(f"   CP-SAT best at k={best_cpsat.k}: {best_cpsat.cpsat_objective:.0f}")
else:
    print(f"   CP-SAT: Not run")

print(f"\n📂 Output Files:")
print(f"   {opt_output_dir / 'summary.csv'}")
print(f"   {viz_output_dir}/*.html")
print("\n" + "=" * 70)
